# E21 NB2 · Duck config B (= R3K2) on vLLM 0.19 + vLLM 0.32, two halves in one notebook

- **Weights / streams:** FP8 + MTP-3 + `--mamba-ssm-cache-dtype float16`, **16 streams** (fallback: SSM dtype default = S1's engine)
- **Half A:** vLLM 0.19.0 (the S1-60K engine as the E12-V Duck server) · **Half B:** vLLM 0.32.0 (the VMS arm)
- Both halves: the same 120 evaluation tasks, fresh Duck state, ~5.5 h each (commit). Hidden rerun: half A only, 12 h.
- Duck: package 9e4b89285136, 50k + 45k attempts, before3, weak commit (R3K2 exactly); history reasoning key `reasoning` (vLLM reads only that key).
- Report: `q27x/e21_report.json` (per half: full score, attempt-1-only score, engine, tok/s, prefix cache).


In [ ]:
# ==================================================================== CELL 0 — mode + fallback submission (stdlib only)
import os, sys, json, time, random, hashlib, re, shutil, subprocess, threading, traceback, zipfile, math
from pathlib import Path
from datetime import datetime, timezone
import concurrent.futures as cf

NOTEBOOK_T0 = time.perf_counter()
# S1 (CLOCK): Kaggle's 12 h counts from kernel start; R5's cell 0 began 7 s later. Anchor the clock at the kernel
# process start (capped at 10 min in case the value is odd).
try:
    import psutil as _ps
    _since_kernel_start = max(0.0, min(600.0, time.time() - _ps.Process(os.getpid()).create_time()))
except Exception:
    _since_kernel_start = 0.0
NOTEBOOK_T0 -= _since_kernel_start
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RERUN = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true", "yes"}
ARC_ROOT = Path(os.environ.get("Q27X_ARC_ROOT", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"))
if not ARC_ROOT.exists():
    for _alt in ("/kaggle/input/arc-prize-2026-arc-agi-2",):
        if Path(_alt).exists():
            ARC_ROOT = Path(_alt)
WORKING = Path(os.environ.get("Q27X_WORKING", "/kaggle/working")); WORKING.mkdir(parents=True, exist_ok=True)
OUT = WORKING / "q27x"; OUT.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = WORKING / "submission.json"

def _load_json(p):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

test_challenges = _load_json(ARC_ROOT / "arc-agi_test_challenges.json")

def write_json_atomic(path, obj):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f)
    os.replace(tmp, path)

def identity_submission(challenges):
    return {tid: [{"attempt_1": q["input"], "attempt_2": q["input"]} for q in task["test"]]
            for tid, task in challenges.items()}

write_json_atomic(SUBMISSION_PATH, identity_submission(test_challenges))
print("MODE:", "COMPETITION RERUN (hidden test)" if RERUN else "COMMIT / INTERACTIVE (evaluation)",
      "| fallback submission.json:", len(test_challenges), "tasks | RUN_ID", RUN_ID,
      "| clock anchored", round(_since_kernel_start, 1), "s before cell 0")


In [ ]:
# ==================================================================== CELL 1 — configuration
PROFILE = "RERUN_ALL" if RERUN else os.environ.get("Q27X_PROFILE", "E21SMOKE").strip().upper()
PROFILE_TABLE = {
    # commit run: 12 public evaluation tasks, scored; also measures throughput for the 12-hour projection
    "EVAL12":    {"source": "eval", "n_tasks": 12, "time_budget_sec": 3.0 * 3600, "admit_frac": 0.80, "drain_frac": 0.90},
    # S1 quick commit: 4 eval tasks that closed fastest in R5 iteration 1 -> a valid version in about 1 h, then submit
    "S1COMMIT4": {"source": "eval", "task_list": ["1818057f", "2ba387bc", "bf45cf4b", "58490d8a"],
                  "time_budget_sec": 90 * 60, "admit_frac": 0.80, "drain_frac": 0.90},
    # R5 EVAL120: full evaluation run (120 tasks / 172 inputs) on the hidden-run clock (RERUN_ALL margins)
    "EVAL120":   {"source": "eval", "n_tasks": 120, "time_budget_sec": 12.0 * 3600 - 3 * 60,
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
    # E9/E10: PROXY48 (EDA LB-mirror) + the 8 EVAL12 tasks not in it = 56 tasks / 72 test inputs
    "EVAL14":    {"source": "eval", "task_list": ["135a2760", "136b0064", "20a9e565", "21897d95", "247ef758", "2b83f449", "3dc255db", "4c3d4a41", "4c416de3", "4c7dc4dd", "62593bfd", "64efde09", "65b59efc", "6e4f6532"],
                  "time_budget_sec": 2.0 * 3600, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
    # E12 step 3.5: GPU smoke, 1 fast task (2ba387bc: 5,355 thinking tokens in E10 ORDER2), 2 branches, 15 min solving
    "SMOKE1":    {"source": "eval", "task_list": ["2ba387bc"], "time_budget_sec": 50 * 60, "admit_frac": 0.90,
                  "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60,
                  "admit_margin_sec": 2 * 60, "solve_window_sec": 15 * 60},
    # E12 step 4: the E11-05 EVAL30 tasks (same seed sample), 4 h
    "EVAL30":    {"source": "eval", "n_tasks": 30, "time_budget_sec": 4.0 * 3600,
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
    # E12 rev 4 (user 2026-10-20): EVAL6 = 6 tasks that S49/R5/DIRECT solved and E12 EVAL30 did not (accuracy test)
    "EVAL6":     {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd"],
                  "time_budget_sec": 125 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 90 * 60},   # audits: the 100k cap is soft; 12 x ~120k = 1.44 M at 368 tok/s = 65 min + prefill
    # E12 rev 5 (user 2026-10-21): the same 6 tasks with every validated fix (A/B against rev 4 EVAL6: 3/6, 4/8)
    "EVAL6R5":   {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd"],
                  "time_budget_sec": 125 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 90 * 60},
    # E12 rev 5 (user 2026-10-21): EVAL6R5 + 6 informative EVAL30 tasks: 4 Duck rev 1 solved (247ef758 Duck-only on
    # query 1 and first-exact-wrong; bf45cf4b 9.5k; f931b4a8 38.8k; 65b59efc 67k) + 2 mid ones it missed (9385bd28 and
    # cbebaa4b, ledger solve rate 0.25). 24 branches / 6 live; 150 min solving, 185 min budget
    "EVAL12R5":  {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 185 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 150 * 60},
    # E12 rev 5: the E12 EVAL30 tasks (same seed sample) with every fix (A/B against rev 1 EVAL30 7/48 and Poetiq R5 16/48).
    # Capacity: rev 4 EVAL6 measured about 10.4 min per task at 6 live -> 30 tasks about 5.2 h of solving
    "EVAL30R5":  {"source": "eval", "n_tasks": 30, "time_budget_sec": 6.0 * 3600, "admit_frac": 0.90, "drain_frac": 0.94,
                  "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 330 * 60},
    # E12 rev 6 (user 2026-10-21): the EVAL12R5 tasks with rev 6 (A/B: rev 5 xhigh EVAL6R5 5/8, low EVAL12R5 10/19)
    "EVAL12R6":  {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 185 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 150 * 60},
    "EVAL30R6":  {"source": "eval", "n_tasks": 30, "time_budget_sec": 6.0 * 3600, "admit_frac": 0.90, "drain_frac": 0.94,
                  "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 330 * 60},
    # gate 4b (user 2026-10-21): the EVAL12R6 tasks at the hidden run's compute share. CELL 6 sets the solve window to
    # n_tasks x (hidden solve seconds / 240), the hidden solve end taken from RERUN_ALL and the start measured in this run
    "EVAL12R6G4B": {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                    "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                    "time_budget_sec": 80 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                    "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                    "hidden_equiv": True},
    # R6-X1 (user 2026-10-21): target-only vs DFlash2 parity on 4 tasks (no solving): DFlash2 twice (control), then the
    # same server without --speculative-*, the same requests at T 0, one at a time, thinking on and off
    "PARITYR6":  {"source": "eval", "task_list": ["136b0064", "bf45cf4b", "247ef758", "b5ca7ac4"],
                  "time_budget_sec": 75 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "parity": True},
    # E14-A (user 2026-10-23: one notebook; plan Notes/13): engine measurements (A1), a 25-minute Duck run with the
    # adopted levers on the G4B tasks (raw reasoning on), more engine arms (A2). Every phase has its own deadline;
    # whatever does not fit before the hard stop is skipped and reported, never squeezed.
    "E14A":      {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 200 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 25 * 60, "e14": True},
    # E14S (Notes/30, 31; user 2026-10-24: a short commit): a valid version + a GPU smoke of the submission arm (DFlash
    # width 6 + CUDA graphs 1..13) at 12 streams, no measurements. 36 min, not 30: R2 was READY at ~12.5 min (install
    # 3 + staging 2 + first launch 502 s) and the SGLang give-up is admit - 5 min (30 min -> 13 min, 36 min -> 19 min)
    "E14S":      {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 36 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 8 * 60},
    # E14S EVAL120 gate 4b (user 2026-10-24): every evaluation task (120 / 172 inputs) with the hidden run's compute per
    # task: CELL 6 sets the window to 120 x (hidden solve seconds / 240) = about 5.8 h; the coverage of the hidden run
    # (stage-1 starts, branch 1 starts, pass1 / pass2) on eval difficulty. 6.5 h budget = start + window + margins
    # E15 (Notes/35): engine arms (no E14A measurements: E14_DO_MEASURE stays False, so the first launch = the submission
    # arm W6 + graphs13 = the reference), then a 15-min Duck run on the G4B tasks with the adopted arm. 180 min budget (about 15 launches)
    "E15":       {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 180 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 15 * 60, "e15": True},
    # E15B commit (user 2026-10-25, Exp 15 Notes/08): 20 eval tasks with BOTH attempts finishing = the 16 G4B tasks + the
    # first 4 by id of the tasks solved in 1-2 of the 3 EVAL120 G4B runs (2026-10-24). 40 attempts on 16 streams = 3 waves;
    # rev 3f / 3g: waves 16 x a1, then 4 x a1 + 12 x a2, then the last 4 x a2 (4 running = faster). 45k + 30k at 22 tok/s:
    # worst ~ 83 min -> window 92, budget 115; 40k + 35k: worst ~ 74 min -> window 85, budget 110 (READY took 12-13 min;
    # the solve deadline = hard stop - 180 s still clamps it)
    "E15B":      {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b",
                                                  "2ba387bc", "1818057f", "45a5af55", "9aaea919",
                                                  "16de56c4", "1ae2feb7", "221dfab4", "269e22fb"],
                  "time_budget_sec": 128 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 105 * 60},
    # E15B EVAL120 gate 4b: every evaluation task at the hidden run's compute per task (as E14SEVAL120G4B)
    "E15BEVAL120G4B": {"source": "eval", "n_tasks": 120, "time_budget_sec": 6.5 * 3600, "admit_frac": 0.90,
                       "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                       "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "hidden_equiv": True},
    # rev 3i (user 2026-10-25): all 120 eval tasks, exactly a 6-h notebook; the solve window = the hidden run's per-task
    # share (120 x (hidden solve end - start) / 240 ~ 5.8 h), clamped by this run's hard stop - 180 s (~5.7 h)
    "E15BEVAL120R3I": {"source": "eval", "n_tasks": 120, "time_budget_sec": 6.0 * 3600, "admit_frac": 0.90,
                       "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                       "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "hidden_equiv": True},
    # E21 (user 2026-10-28): two halves (vLLM 0.19, then 0.32) on all 120 evaluation tasks in one 12-h commit; each half
    # gets a fixed solve window from its own start (E21_HALF_SOLVE_SEC); the hidden rerun = RERUN_ALL with half A only
    "E21X2EVAL120": {"source": "eval", "n_tasks": 120, "time_budget_sec": 12.0 * 3600 - 3 * 60, "admit_frac": 0.90,
                     "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                     "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "solve_window_sec": 5.45 * 3600},   # rev 2b
    # rev 2d: the GPU smoke (both halves, real loads, probes, ~2 h): 12 evaluation tasks, a 20-min window per half
    "E21SMOKE": {"source": "eval", "n_tasks": 12, "time_budget_sec": 2.5 * 3600, "admit_frac": 0.90,   # rev 2f
                 "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                 "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "solve_window_sec": 20 * 60},
    "E14SEVAL120G4B": {"source": "eval", "n_tasks": 120, "time_budget_sec": 6.5 * 3600, "admit_frac": 0.90,
                       "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                       "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "hidden_equiv": True},
    "PROXY56":   {"source": "eval", "task_list": ["0934a4d8", "20a9e565", "4c7dc4dd", "89565ca0", "136b0064", "291dc1e1", "16b78196", "271d71e2", "28a6681f", "2b83f449", "2c181942", "31f7f899", "35ab12c3", "3dc255db", "409aa875", "4c416de3", "64efde09", "6e453dd6", "7491f3cf", "7b0280bc", "7b80bb43", "7c66cb00", "1818057f", "332f06d7", "53fb4810", "7666fa5d", "da515329", "8f3a5a89", "db695cfb", "97d7923e", "de809cff", "c7f57c3e", "cb2d8a2c", "58490d8a", "dbff022c", "5961cc34", "80a900e0", "269e22fb", "8b9c3697", "e376de54", "b99e7126", "c4d067a0", "135a2760", "4a21e3da", "71e489b6", "800d221b", "dd6b8c4b", "36a08778", "21897d95", "247ef758", "4c3d4a41", "6e4f6532", "88bcf3b4", "898e7135", "8b7bacbf", "b6f77b65"], "time_budget_sec": 6.0 * 3600,
                  "admit_frac": 0.80, "drain_frac": 0.90},
    # hidden test: every task; Kaggle's limit is 12 h including install + model load
    # v1.4.3: stop 10 min before 12 h (user); hard stop 6 min before the budget end (outputs take seconds after it)
    "RERUN_ALL": {"source": "test", "n_tasks": None, "time_budget_sec": 12.0 * 3600 - 3 * 60,
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                  # v1.4.4: admit until the end (NVARC style); drain 8 min before the hard stop, admit 2 min before drain
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
}
if PROFILE not in PROFILE_TABLE:
    raise ValueError(f"unknown profile {PROFILE}; choose {list(PROFILE_TABLE)}")
P = PROFILE_TABLE[PROFILE]
SEED = 20261005
TASK_SAMPLE_SEED = 20261005

MODEL_PATH = Path(os.environ.get("Q27X_MODEL", "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"))
WHEELHOUSE = Path(os.environ.get("Q27X_WHEELHOUSE", "/kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3"))

# ---- generation budget (Kaggle discussion 745787: Qwen3.8 needs xhigh; median ~64k tokens; forced commit ~49k)
REASONING_EFFORT = "xhigh"
THINK_BUDGET = int(os.environ.get("Q27X_THINK_BUDGET", 60000))   # E11 rev 3: 60,000 = S1-60K (LB 34.58); user floor 60k   # S49: LB test of 49,152 (S1: 60,000; R5/E9: 77,824; floor 49,152)      # v1.2: NguyenThanhNhan forced the commit after ~49k thinking tokens
E11_THINK_FLOOR = 60000      # E11 rev 3 (user 2026-10-18): never below 60k (budget and admission guard)
assert THINK_BUDGET >= E11_THINK_FLOOR, (THINK_BUDGET, E11_THINK_FLOOR)
ANSWER_MAX_TOKENS = int(os.environ.get("Q27X_ANSWER_MAX", 8192))   # v1.4: grid-only answer; 49,152 + 8,192 = 57,344 = russcore's max_tokens
DRAIN_ANSWER_MAX_TOKENS = 8192                                      # v1.1 (F11): a 30x30 grid alone is ~2.7k tokens
STOP_THINKING_TEXT = "\n\nConsidering the limited time by the user, I have to give the solution based on the thinking directly now.\n</think>\n\n"
MAX_MODEL_LEN = int(os.environ.get("Q27X_MAX_MODEL_LEN", 106496))   # E9/E10: prompt (<= ~20k) + 77,824 + 8,192 fits; was 81,920   # v1.2: native context is 262k; fp8 KV makes room
CONTEXT_SAFETY = 256
MIN_THINK_BUDGET = 4096                  # v1.2: a long prompt gets a smaller thinking budget instead of being skipped
PROMPT_TOKEN_LIMIT = MAX_MODEL_LEN - ANSWER_MAX_TOKENS - CONTEXT_SAFETY - MIN_THINK_BUDGET
def phase1_cap(prompt_len):
    return max(0, min(THINK_BUDGET, MAX_MODEL_LEN - prompt_len - ANSWER_MAX_TOKENS - CONTEXT_SAFETY))
assert phase1_cap(MAX_MODEL_LEN) == 0 and phase1_cap(100) == min(THINK_BUDGET, MAX_MODEL_LEN - 100 - ANSWER_MAX_TOKENS - CONTEXT_SAFETY)
if (MAX_MODEL_LEN, THINK_BUDGET, ANSWER_MAX_TOKENS) == (106496, 77824, 8192):   # E9/E10
    assert phase1_cap(12800) == 77824 and phase1_cap(20000) == 77824 and phase1_cap(30000) == 68048
if (MAX_MODEL_LEN, THINK_BUDGET, ANSWER_MAX_TOKENS) == (81920, 49152, 8192):
    assert phase1_cap(24320) == 49152 and phase1_cap(30000) == 43472  # every real prompt (≤ ~9k tokens) keeps the full budget
SAMPLING = {"temperature": float(os.environ.get("Q27X_TEMPERATURE", 0.6)), "top_p": 0.95, "top_k": 20}   # v1.4: russcore LB-10.14 sampling

# ---- engine (PIV/V3.1 C1 settings; context, concurrency and MTP changed)
MAX_NUM_SEQS = int(os.environ.get("Q27X_MAX_NUM_SEQS", 16))   # upper bound; actual slots = KV tokens // EXPECTED_LIVE_TOKENS_PER_SEQ
MAX_NUM_BATCHED_TOKENS = 4096   # v1.3.5: = discussion max-prefill-tokens 4096; 32768 ran out of memory in run 2 (MTP drafter)
GPU_MEMORY_UTILIZATION = float(os.environ.get("Q27X_GPU_MEM", 0.90))   # v1.4.1: kept at 0.90 so prefix caching is the only change
ENABLE_PREFIX_CACHING = os.environ.get("Q27X_PREFIX_CACHE", "1") == "1"   # v1.4.1: first engine attempt (mamba 'align')
SPECULATIVE_CONFIG = {"method": "mtp", "num_speculative_tokens": int(os.environ.get("Q27X_SPEC_TOKENS", 3))}   # v1.2: vLLM Qwen3.8 recipe uses 3
KV_CACHE_DTYPE = os.environ.get("Q27X_KV_DTYPE", "fp8_e4m3")   # v1.3.4: back to the discussion recipe; run 1's crash was
# FlashInfer's JIT link step (`ld: cannot find -lcuda`), fixed in cell 5 (libcuda shim + link probe); fp8 KV itself loaded
# fine (312,000 KV tokens). Set Q27X_KV_DTYPE=auto for bf16 KV.
if os.environ.get("Q27X_NO_SPEC") == "1":
    SPECULATIVE_CONFIG = None
EXPECTED_LIVE_TOKENS_PER_SEQ = 43000      # a running task holds on average about half of its final ~60k length
# v1.3.5 correction: cell 5 divides the whole KV pool (run 2: 781 blocks × 1,600 = 1,249,600 tokens) by this. The pool is
# shared by 1 attention + 3 GDN groups; vLLM's "312,000 tokens" log line is pool // 4. A 30k-token task needs 19 attention
# + 12 GDN-state blocks (3 groups × (1 + 3 MTP)), so 16 tasks need ~496 of 781 blocks -> 16 slots (preemptions reported)

# ---- verification (fresh process per candidate: train first, then test)
VERIFY_TIMEOUT_SEC = 20.0
VERIFY_WORKERS = 4

# ---- time
TOTAL_BUDGET_SEC = float(os.environ.get("Q27X_TIME_BUDGET", P["time_budget_sec"]))
# v1.3: deadlines derived in a guaranteed order (v1.2 had DRAIN after HARD_STOP for EVAL12, so the drain never ran)
VERIFY_DEADLINE = NOTEBOOK_T0 + TOTAL_BUDGET_SEC - P.get("verify_margin_sec", 8 * 60)   # reporting only since v1.4
HARD_STOP = NOTEBOOK_T0 + TOTAL_BUDGET_SEC - P.get("hard_margin_sec", 25 * 60)        # abort everything, write outputs
DRAIN_DEADLINE = min(NOTEBOOK_T0 + TOTAL_BUDGET_SEC * P["drain_frac"], HARD_STOP - 8 * 60)    # force-close thinking
ADMIT_DEADLINE = min(NOTEBOOK_T0 + TOTAL_BUDGET_SEC * P["admit_frac"], DRAIN_DEADLINE - 7 * 60)  # no new task after this
if "drain_margin_sec" in P:   # v1.4.4 (RERUN): fixed offsets from the hard stop, so the slots stay full until the drain
    DRAIN_DEADLINE = HARD_STOP - P["drain_margin_sec"]
    ADMIT_DEADLINE = DRAIN_DEADLINE - P["admit_margin_sec"]
assert NOTEBOOK_T0 < ADMIT_DEADLINE < DRAIN_DEADLINE < HARD_STOP < VERIFY_DEADLINE < NOTEBOOK_T0 + TOTAL_BUDGET_SEC

# v1.3.5 (run 2 OOM): CUDA-graph memory is taken out of the KV budget (vLLM 0.19 envs.py 1654, v1/worker/gpu_worker.py
# 393-420); run 2 captured 0.61 GiB of graphs beyond it. Set here, before cell 5 spawns the vLLM workers (they inherit it).
os.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = "1"
# ---- E11-05: engine = SGLang 0.5.21 + RedHat INT4 (W4A16 g128) + DFlash2 drafter = the E11-04 W1 launch (2026-10-18).
# MODEL_PATH (FP8) stays: its tokenizer + chat template render every prompt exactly as in S49, and it is the vLLM fallback.
ENGINE_BACKEND = os.environ.get("E11_ENGINE", "sglang").strip().lower()       # sglang (vLLM FP8 fallback) | vllm (= S49)
E11_ALLOW_VLLM_FALLBACK = os.environ.get("E11_VLLM_FALLBACK", "1") == "1"
SGL_MODEL_PATH = Path(os.environ.get("E11_INT4", "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-int4-redhat/pytorch/w4a16-g128/1"))
SGL_DRAFT_PATH = Path(os.environ.get("E11_DFLASH", "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-dflash2/pytorch/bf16/1"))
SGL_WHEELHOUSE = Path(os.environ.get("E11_SGL_WHEELHOUSE", "/kaggle/input/datasets/foysalemonshanto/sglang-0-5-21-cu13-wheelhouse"))


def _e11_discover(path, marker, pred):
    """The given path if it holds `marker`, else the first /kaggle/input folder whose `marker` json satisfies pred."""
    if (path / marker).is_file():
        return path
    root = Path("/kaggle/input")
    for p in (sorted(root.rglob(marker)) if root.exists() else []):
        try:
            if pred(p.parent, _load_json(p)):
                return p.parent
        except Exception:
            continue
    return path


SGL_MODEL_PATH = _e11_discover(SGL_MODEL_PATH, "config.json", lambda d, c: isinstance(c, dict)
                               and (c.get("quantization_config") or {}).get("quant_method") == "compressed-tensors")
SGL_DRAFT_PATH = _e11_discover(SGL_DRAFT_PATH, "config.json", lambda d, c: isinstance(c, dict)
                               and c.get("architectures") == ["DFlash2DraftModel"])
SGL_WHEELHOUSE = _e11_discover(SGL_WHEELHOUSE, "BUILD_INFO.json", lambda d, c: isinstance(c, dict)
                               and c.get("sglang") == "0.5.21" and (d / "requirements.lock").is_file())
SGL_CONTEXT_LEN = 155648            # = the E11-04 W1 launch; every request stays inside MAX_MODEL_LEN (phase1_cap)
SGL_MAX_RUNNING = 16                # E15B (Notes/02 Rev 2c): 16 real streams; was 12 (E11-04 W1: 12 really running (mamba cache 56 / 4 state slots = 14 >= 12)
SGL_SERVER_MAX_RUNNING = 17         # E15B: client 16 + 1 (Notes/30 nodelay crash: a 13th on buffers sized 12); rev 3c: server admission limit = client 12 + 1. SGLang 0.5.21 counts the row of a request in
                                    # chunked prefill twice (scheduler.py:3971-3976) and latches batch_is_full (3846-3848) until a
                                    # running request ends -> EVAL30 ran 11 + 1 queued (2,358 of 2,509 decode lines). Upstream fix
                                    # PR #44401 (after 0.5.21); +1 is the documented workaround (PR #30295). CUDA graphs stay at 12.
SGL_LAUNCH_TIMEOUT_SEC = 30 * 60    # first launch includes JIT + CUDA graphs: 567 s in E11-04
SGL_RELAUNCH_TIMEOUT_SEC = 20 * 60  # a diagnosed relaunch (E11-04: about 75 s once the JIT cache exists)
SGL_GIVE_UP_AT = min(NOTEBOOK_T0 + (240 if RERUN else 75) * 60, ADMIT_DEADLINE - 5 * 60)   # E15B B5: RERUN 4 h   # no SGLang launch after this -> vLLM fallback
SGL_STREAM_IDLE_SEC = 1800          # a socket read silent for longer closes that request as 'error'
# rev 2 (GPT audit 2026-10-18, verified): capacity gate from the server log, fail-closed on missing evidence
SGL_MIN_RUNNING = 8                 # below this the SGLang engine is refused (-> S49 vLLM)
SGL_MIN_KV = 600_000                # KV tokens; E11-04 W1 had 881,336
SGL_KV_PER_REQUEST = 50_000         # E15B: the MEAN attempt context ~ 50k (attempt 1 at 45k ends at ~57-59k p90, max ~64k; attempt 2 less; Auditor 4); was 70k (KV // 70k = 12). concurrency <= KV // this (881,336 -> 12); a long Poetiq unit is prompt + 49k + 8k
SGL_MIN_FREE_GB = 1.0               # GPU headroom after start-up (E11-04 W1: 2.22 GB)
# rev 2: drain earlier. Aborted long prefixes were not cached in E11-04 (30k prefix: 0 cached, 24.5 s), so 12 phase-2
# continuations re-prefill up to ~0.7M tokens; 12 min instead of 8 min before the hard stop (SGLang only)
SGL_DRAIN_MARGIN_SEC = 8 * 60      # E11-06: back to 8 min (EVAL30 drain <= 83 s)
# rev 2: KV gate active on SGLang (S49's floor 12 = CONCURRENCY 12 made it inert): from 10 running, admit only below 85 %
SGL_KV_GATE_MIN_ACTIVE = 10
SGL_KV_ADMIT_MAX = 0.85

CONFIG = {k: globals()[k] for k in ["PROFILE", "RERUN", "SEED", "REASONING_EFFORT", "THINK_BUDGET", "ANSWER_MAX_TOKENS",
          "DRAIN_ANSWER_MAX_TOKENS", "MAX_MODEL_LEN", "PROMPT_TOKEN_LIMIT", "SAMPLING", "MAX_NUM_SEQS",
          "MAX_NUM_BATCHED_TOKENS", "GPU_MEMORY_UTILIZATION", "ENABLE_PREFIX_CACHING", "SPECULATIVE_CONFIG", "KV_CACHE_DTYPE", "VERIFY_TIMEOUT_SEC",
          "TOTAL_BUDGET_SEC", "MIN_THINK_BUDGET"]}
CONFIG["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = os.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"]
CONFIG["profile_row"] = P
CONFIG["e11"] = {k: (str(globals()[k]) if isinstance(globals()[k], Path) else globals()[k]) for k in (
    "ENGINE_BACKEND", "E11_ALLOW_VLLM_FALLBACK", "SGL_MODEL_PATH", "SGL_DRAFT_PATH", "SGL_WHEELHOUSE", "SGL_CONTEXT_LEN",
    "SGL_MAX_RUNNING", "SGL_SERVER_MAX_RUNNING", "SGL_LAUNCH_TIMEOUT_SEC", "SGL_RELAUNCH_TIMEOUT_SEC", "SGL_STREAM_IDLE_SEC", "SGL_MIN_RUNNING",
    "SGL_MIN_KV", "SGL_KV_PER_REQUEST", "SGL_MIN_FREE_GB", "SGL_DRAIN_MARGIN_SEC", "SGL_KV_GATE_MIN_ACTIVE", "SGL_KV_ADMIT_MAX")}
CONFIG["e11"]["give_up_after_sec"] = round(SGL_GIVE_UP_AT - NOTEBOOK_T0)
print("E11 engine:", json.dumps(CONFIG["e11"]))
print(json.dumps(CONFIG, indent=1, default=str))

os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
os.environ["HF_HUB_OFFLINE"] = "1"; os.environ["TRANSFORMERS_OFFLINE"] = "1"


# ---- E10 Poetiq donor configurations; source: poetiq-ai/poetiq-arc-agi-solver/config.py
# Original 86K context, GPU recipe, deadlines and original puzzle order above are unchanged.
POETIQ_NUM_EXPERTS = int(os.environ.get('E10_POETIQ_EXPERTS','1'))
POETIQ_MAX_ITERATIONS = int(os.environ.get('E10_POETIQ_MAX_ITERATIONS','10'))
POETIQ_MAX_SOLUTIONS = int(os.environ.get('E10_POETIQ_MAX_SOLUTIONS','2'))
POETIQ_SOLVER_TEMPERATURE = float(os.environ.get('E10_POETIQ_TEMPERATURE','1.0'))
POETIQ_SOLVER_VARIANTS = tuple(int(v) for v in os.environ.get('E10_POETIQ_PROMPT_VARIANTS','1').split(','))
POETIQ_SELECTION_PROB = float(os.environ.get('E10_POETIQ_SELECTION_PROBABILITY','1.0'))
POETIQ_SEED = int(os.environ.get('E10_POETIQ_SEED','0'))
POETIQ_SHUFFLE_EXAMPLES = os.environ.get('E10_POETIQ_SHUFFLE','0')=='1'
POETIQ_IMPROVING_ORDER = os.environ.get('E10_POETIQ_IMPROVING_ORDER','1')=='1'
POETIQ_USE_NEW_VOTING = os.environ.get('E10_POETIQ_NEW_VOTING','1')=='1'
POETIQ_COUNT_FAILED_MATCHES = os.environ.get('E10_POETIQ_COUNT_FAILED_MATCHES','1')=='1'
POETIQ_ITERS_TIEBREAK = os.environ.get('E10_POETIQ_ITERS_TIEBREAK','0')=='1'
POETIQ_LOW_TO_HIGH_ITERS = os.environ.get('E10_POETIQ_LOW_TO_HIGH_ITERS','0')=='1'
POETIQ_SANDBOX_TIMEOUT = float(os.environ.get('E10_POETIQ_SANDBOX_TIMEOUT','5'))
POETIQ_SANDBOX_WORKERS = int(os.environ.get('E10_POETIQ_SANDBOX_WORKERS','4'))
# S1: KV admission gate (no new request above this vLLM KV usage unless fewer than MIN_ACTIVE are running)
E10_KV_ADMIT_MAX = float(os.environ.get('E10_KV_ADMIT_MAX','0.95'))
E10_KV_GATE_MIN_ACTIVE = int(os.environ.get('E10_KV_GATE_MIN_ACTIVE','12'))
assert POETIQ_NUM_EXPERTS >= 1 and POETIQ_MAX_ITERATIONS >= 1
assert POETIQ_SOLVER_VARIANTS and all(v in (1,2,3) for v in POETIQ_SOLVER_VARIANTS)
assert POETIQ_MAX_SOLUTIONS >= 1 and 0 <= POETIQ_SELECTION_PROB <= 1
CONFIG['poetiq']={k:globals()[k] for k in (
    'POETIQ_NUM_EXPERTS','POETIQ_MAX_ITERATIONS','POETIQ_MAX_SOLUTIONS',
    'POETIQ_SOLVER_TEMPERATURE','POETIQ_SOLVER_VARIANTS','POETIQ_SELECTION_PROB',
    'POETIQ_SANDBOX_TIMEOUT','POETIQ_SANDBOX_WORKERS','POETIQ_USE_NEW_VOTING')}


# ---- E12: Duck REPL agent (Experiment 12, user option A 2026-10-19). Duck's own settings are in arc2_duck.DUCK_ENV
# (= dfranzen milestone-2 cell 4): 12,288 output per request, yield at 2,048 generated tokens, preserved thinking,
# 128 Ki context with drain to 58 Ki. Approved exception to the 60k-per-request floor (thinking accumulates in context).
E12_BRANCHES = int(os.environ.get("E12_BRANCHES", 2))                 # independent branches per task -> attempt_1/2
_E12_REV5 = True                                # rev 5 build: every profile (incl. the hidden rerun) runs the fixes
_E12_REV6 = True                                # rev 6 build (user 2026-10-21): code-first view + coverage scheduler
_E12_REV4 = PROFILE == "EVAL6" or _E12_REV5                            # rev 4 defaults (pre-run capacity check, 2026-10-20)
E12_STREAMS = int(os.environ.get("E12_STREAMS", 16 if _E12_REV4 else 0))   # Duck ARC3_MAX_ACTIVE_STREAMS; 0 = engine concurrency
E12_MAX_GEN_PER_BRANCH = int(os.environ.get("E12_MAX_GEN", 50000 if _E12_REV4 else 0)) or None   # E15B rev 3f: attempt 1 = 50,000 (attempt 2: E15B block)   # Duck max_generated_tokens_per_game
# rev 4: at most N branches alive (resident contexts <= KV 867k / 128k context = 6; EVAL30: 30 live -> 8.4 M re-prefill),
# branches of a task start together; and up to 60,000 output per request (EVAL30: 29/30 first requests cut at 12,288)
E12_LIVE_BRANCHES = int(os.environ.get("E12_LIVE", 12 if _E12_REV4 else 0)) or None   # 6 x 131k = 786k <= KV 867k; S12 (user): 12 x rev 5 context p50 58k = 697k
# rev 5 B5 (evidence, 2026-10-21): no productive EVAL6 reply exceeded 44,779 tokens; the 7 replies that hit 60,000 made no
# tool call; at 49,152 the input budget is 81,408 (4 of the 6 real-overflow trims would have fitted). 40k would have cut one.
E12_DUCK_MAX_OUTPUT = int(os.environ.get("E12_DUCK_MAX_OUTPUT", (12288 if _E12_REV6 else 49152 if _E12_REV5 else 60000)
                                         if _E12_REV4 else 0)) or None   # rev 6: the first-request cap; later caps are per request
# rev 5 B9 (user 2026-10-21): Qwen3.8 model-card thinking temperature 1.0 (Duck milestone 0.7)
E12_DUCK_TEMPERATURE = float(os.environ.get("E12_DUCK_TEMPERATURE", 1.0 if _E12_REV5 else 0)) or None
E12_REASONING_EFFORT = os.environ.get("E12_REASONING_EFFORT", "").strip().lower()   # "" = model default (xhigh)
if E12_REASONING_EFFORT:   # user 2026-10-21: low-effort variant of rev 5 (Qwen3.8 template: xhigh / medium / low)
    assert E12_REASONING_EFFORT in ("xhigh", "medium", "low"), E12_REASONING_EFFORT
    os.environ["E12_REASONING_EFFORT"] = E12_REASONING_EFFORT
if _E12_REV5:   # B1-B4, B6-B8 in arc2_duck (E12_ARC2_REV5); B3 counts reasoning with the served model's tokenizer
    os.environ["E12_ARC2_REV5"] = "1"
    os.environ["E12_TOKENIZER_JSON"] = str(SGL_MODEL_PATH / "tokenizer.json")
# rev 6 (user 2026-10-21; fix list "Rev 6 gap check", both plan reviews): facts view + trusted helpers + auto-check +
# state-aware resumes (arc2_duck E12_ARC2_REV6); request caps 12,288 -> 24,576 after the first successful Python call,
# 32,768 after two no-call cuts (thinking kept in history, branch allowance >= 60k, soft cap 100k); first exposure: branch 0
# of every task in ORDER2 order, paused at about 28k generated tokens, then stage 2 by evidence (resume / branch 1 with the
# grid view = the representation hedge); 10 workers, context window sized to the KV pool in CELL 6; per-request deadline.
E12_R6 = {"E12_R6_CAP_FIRST": 12288, "E12_R6_CAP_PROGRESS": 24576, "E12_R6_CAP_RESCUE": 32768,
          "E12_R6_FIRST_EXPOSURE": 0, "E12_R6_B1_VIEW": "facts", "E12_R6_MIN_ADMIT_SEC": 300,   # E15B #3, #1, #49
          "E12_CHECK_VERIFY_CAP": 24, "E12_AUTOCHECK_CAP": 24, "E12_AUTOCHECK_PER_CALL": 3, "E12_DUCK_DRAIN_TOKENS": 16384,
          # rev 6c (user 2026-10-21: 12 streams): the per-stream window = 1.0 x KV / 12 (= 73.4k at 881k). 0.9 would give
          # 66.1k (history 41.5k at the 24,576 cap); D66 ran 12 slots at context 155,648 with KV peak 90 % and 0 preemptions,
          # rev 6e G4-1 (user 2026-10-21): workers stay 12; no new task or branch while KV usage >= 0.95
          "E12_R6_FACTS_MAX_CHARS": 3600, "E12_R6_SHOW_MAX_CELLS": 900, "E12_R6_KV_FRACTION": 1.0,
          "E12_R6_P90_CUTOFF": 0,
          # rev 6b (gap check vs the problem->fix map): facts <= 800 Qwen tokens (Q-5; 120 eval tasks: full p50 1,237 ->
          # bounded p50 736 / p90 837 / max 1,396), fair-share stage 2 (R6-S3) with a 24,576-token minimum chunk
          "E12_R6_FACTS_MAX_TOKENS": 800, "E12_R6_FAIR_SHARE": 0,   # E15B #45: no chunks / pauses
          # rev 6e (user 2026-10-21: "take all", fix list G4-1..G4-18; G4B evidence): KV gate 0.95 polled every 5 s (G4-1);
          # branch-1 first cap 24,576 (G4-2; G4B b1 first calls at 12.4-18.0k); show() up to 900 cells per grid, 1,000 per
          # call (G4-4; 30x30 = 2,033 Qwen tokens of the 3,072 tool cap); provisional nudge (G4-6: 27,000 in rev 6e; 24,000 since rev 6f, H6-1; G4B
          # solved branches defined transform at 18,383 / 26,228); branch-1 indexed view <= 12,000 tokens (G4-11);
          # minimum stage-2 chunk 12,288 (G4-17: every G4B share 3.8-7.5k was below 24,576); cache diagnostic (G4-8)
          "E12_R6_KV_ADMIT": 0.95, "E12_R6_KV_POLL": 5.0, "E12_R6_CAP_FIRST_B1": 24576, "E12_R6_SHOW_CALL_CELLS": 1000,
          "E12_R6_NUDGE_TOKENS": 24000, "E12_R6_B1_GRID_MAX_TOKENS": 12000, "E12_R6_MIN_CHUNK": 12288,
          "E12_R6_CACHE_DIAG": 1}
# rev 6f (user 2026-10-22, the H6 set of the three GPT rev 6e audits): the provisional nudge at 24,000 tokens, sent in the
# next tool result (H6-1: in G4B no branch 0 started a new turn at >= 27k before the 28k pause); the cache diagnostic
# runs AFTER solving (H6-3); every commit window starts after the preflight (H6-6); one isolation retry (H6-17)
if _E12_REV6:
    os.environ["E12_ARC2_REV6"] = "1"
    for _k, _v in E12_R6.items():
        E12_R6[_k] = type(_v)(os.environ.get(_k, _v))
        os.environ[_k] = str(E12_R6[_k])
# audits (verified): non-streaming read timeout 900 s < a 60k reply at ~61 tok/s per stream (978 s) -> 1,800 s.
# The 100k per-branch cap is soft (checked after each request). max_tokens 60,000 is a ceiling, not a minimum thinking.
# The milestone context (131,072, drain 58 Ki) is kept: input budget 70,560, trim target 35,280. Only the last message is protected:
# a tool-less reply before the resume opener could be dropped (EVAL6: 5 resets; rev 5 B3 counts reasoning correctly).
E12_DUCK_REQUEST_TIMEOUT = int(os.environ.get("E12_DUCK_REQUEST_TIMEOUT", 3600 if _E12_REV4 else 0)) or None   # E15B #47
for _k, _v in (("E12_DUCK_MAX_OUTPUT", E12_DUCK_MAX_OUTPUT), ("E12_DUCK_REQUEST_TIMEOUT", E12_DUCK_REQUEST_TIMEOUT),
               ("E12_DUCK_TEMPERATURE", E12_DUCK_TEMPERATURE)):
    if _v:
        os.environ[_k] = str(_v)                                         # read by arc2_duck (CELL 3 / CELL 6)
# E15B (user 2026-10-24, Exp 15 Notes/02; rev 3f 2026-10-25): config B = 2 independent attempts per task, attempt 1 =
# 50,000 and attempt 2 = 45,000 generated tokens (attempt 1 on every
# task first, then attempt 2 with its own agent and the facts view); no pause / chunk / resume; request cap = remaining
# budget - 4,096 (a final commit turn); fixed 100,000-token Duck window (no trim inside an attempt); 16 real streams
E15B = {"E12_B_MODE": 1, "E12_B_ATTEMPT_TOKENS": E12_MAX_GEN_PER_BRANCH or 50000, "E12_B_RESERVE": 4096,
        "E12_B_WINDOW": 100000, "E12_B_ATTEMPT_TOKENS_B1": 45000,
        "E12_B_SKIP_B1_IF_ACCEPTED": 0, "E12_B_EXTRA_ATTEMPTS": 1, "E12_B_MAX_ATTEMPTS": 8, "E12_B_WEAK_COMMIT": 1, "E12_B_B1_ACC_MODE": 2}
for _k, _v in E15B.items():
    E15B[_k] = int(os.environ.get(_k, _v))
    os.environ[_k] = str(E15B[_k])
assert E15B["E12_B_ATTEMPT_TOKENS"] == E12_MAX_GEN_PER_BRANCH, "the attempt budget and the session cap must agree"
assert 0 < E15B["E12_B_ATTEMPT_TOKENS_B1"] <= E15B["E12_B_ATTEMPT_TOKENS"], "attempt 2's budget (rev 3f)"
E12_SOLVE_WINDOW_SEC = P.get("solve_window_sec")                      # SMOKE1 only
E12_STOP_BEFORE_HARD_SEC = 180   # branches stop; outputs are written after it. rev 3 (GPT re-audit): a submit at the
                                 # deadline can still verify for <= 60 s, then CELL 6/7 write; 60 s left no margin
E12_MAMBA_TRACK_64 = os.environ.get("E12_MAMBA_TRACK_64", "0") == "1"  # Duck's --mamba-track-interval/--page-size 64 (measure)
# rev 2 (GPT audits, validated 2026-10-19): Duck request snapshots (the exact messages sent; large) for the smoke only,
# and a per-sandbox-process address-space limit (Duck sets CPU / file size / open files only)
E12_REQUEST_LOGS = os.environ.get("E12_REQUEST_LOGS", "1" if PROFILE == "SMOKE1" else "0") == "1"
E12_SANDBOX_MEM_MB = int(os.environ.get("E12_SANDBOX_MEM_MB", 4096))
CONFIG["e15b"] = dict(E15B)
CONFIG["e12"] = {k: globals()[k] for k in ("E12_BRANCHES", "E12_STREAMS", "E12_MAX_GEN_PER_BRANCH", "E12_SOLVE_WINDOW_SEC",
                                           "E12_LIVE_BRANCHES", "E12_DUCK_MAX_OUTPUT", "E12_DUCK_REQUEST_TIMEOUT",
                                           "E12_DUCK_TEMPERATURE", "_E12_REV5", "E12_REASONING_EFFORT",
                                           "E12_STOP_BEFORE_HARD_SEC", "E12_MAMBA_TRACK_64", "E12_REQUEST_LOGS",
                                           "E12_SANDBOX_MEM_MB", "_E12_REV6")}
if _E12_REV6:
    CONFIG["e12"]["rev6"] = dict(E12_R6)
CONFIG["e12"]["duck_commit"] = "10882e332d883af3e61616df8c8a17d03a328a75"
print("E12:", json.dumps(CONFIG["e12"]))


# ==================================================================== E21 (Experiment 21, user 2026-10-28)
# Duck config B (= R3K2, everything above unchanged) on vLLM OpenAI servers. Half A = vLLM 0.19.0 in this kernel (the
# S1-60K engine, launched as the E12-V Duck server); half B = vLLM 0.32.0 in /tmp/vllm-venv (the VMS installer and arm).
# Both halves solve the same 120 evaluation tasks with fresh Duck state; the hidden rerun runs half A only.
E21_VARIANT = "NB2"
E21_HALF_SPECS = {"NB2": {"A": {"vllm": "0.19.0", "weights": "fp8", "seqs": 16, "ssm": "float16",
              "fallback": {"ssm": None}},
        "B": {"vllm": "0.32.0", "weights": "fp8", "seqs": 16, "ssm": "float16",
              "fallback": {"ssm": None}}}}
E21_HALVES = ["A"] if RERUN else ["A", "B"]
ENGINE_BACKEND = "vllm"                       # CELL 1b stages the variant's weights; CELL 2 installs vllm==0.19.0 here
E21_STAGE_PATHS = [MODEL_PATH]   # rev 2: NB1 stages its FP8 fallback too (VAL_engine_A2 H2)
E12_SOLVE_WINDOW_SEC = (float(os.environ.get("E21_HALF_SOLVE_SEC", P["solve_window_sec"]))
                        if P.get("solve_window_sec") else None)          # per half, from its own solve start
E21_HALF_SOLVE_SEC = E12_SOLVE_WINDOW_SEC or 0.0
E21_B_SETUP_RESERVE_SEC = float(os.environ.get("E21_B_SETUP_RESERVE_SEC", 30 * 60))   # rev 2b (C1 M3): switch + launch + preflight
E21_LAUNCH_TIMEOUT_SEC = 30 * 60
E21_LAUNCH_TIMEOUT_V032_SEC = 20 * 60      # rev 2b (C1 M3): 0.32 from staged weights was ready in ~550 s (E18)
# rev 2b (VAL_fidelity_E1 M2, E2 M2): Duck counts reasoning tokens with E12_TOKENIZER_JSON (the INT4 file, as R3K2);
# without the INT4 model attached it silently fell back to a chars/token ratio. The FP8 tokenizer has the same vocab
# and merges and encodes identically (README Finding 4, Kaggle CPU kit on the real files).
if not Path(os.environ.get("E12_TOKENIZER_JSON", "")).is_file() and (MODEL_PATH / "tokenizer.json").is_file():
    os.environ["E12_TOKENIZER_JSON"] = str(MODEL_PATH / "tokenizer.json")
E21_B_FOLLOWS_A = True                       # user option (ka): a half-A fallback config is repeated in half B
assert PROFILE in ("E21X2EVAL120", "E21SMOKE", "RERUN_ALL") or os.environ.get("E21_ANY_PROFILE") == "1", PROFILE   # rev 2 (E1 LOW-5)
assert RERUN or len(E21_HALVES) == 1 or (E12_SOLVE_WINDOW_SEC or 0) > 0, "two halves need a solve window"   # C1 L3
E21_VLLM_MAX_MODEL_LEN = 135168               # E12-V: Duck's 128 Ki context window + margin
E21_PORTS = {"A": 1234, "B": 8011}
E21_SERVED = {"fp8": "Qwen/Qwen3.8-27B-FP8", "int4": "Qwen/Qwen3.8-27B-INT4"}   # never "qwen" (Duck alias, E12-V run 1)
E18_VLLM_WHEELHOUSE = Path(os.environ.get("E18_VLLM_WHEELHOUSE", "/kaggle/input/datasets/foysalemonshanto/vllm-0-32-0-cu13-wheelhouse"))
E18_SILENCE_SEC = float(os.environ.get("E18_SILENCE_SEC", 600))
E21_FAKE = os.environ.get("E21_FAKE_SERVER") if os.environ.get("E11_HARNESS") == "1" else None   # CPU harness only
E18_FAKE = E21_FAKE
E12_R6["E12_R6_CACHE_DIAG"] = 0               # the SGLang cache diagnostic (after solving) is not run on vLLM
os.environ["E12_R6_CACHE_DIAG"] = "0"
if E12_SOLVE_WINDOW_SEC and not RERUN:
    assert 2 * E12_SOLVE_WINDOW_SEC + E21_B_SETUP_RESERVE_SEC < HARD_STOP - NOTEBOOK_T0, "two windows must fit"
CONFIG["e21"] = {"variant": E21_VARIANT, "halves": E21_HALVES, "specs": E21_HALF_SPECS[E21_VARIANT],
                 "half_solve_sec": E12_SOLVE_WINDOW_SEC, "b_setup_reserve_sec": E21_B_SETUP_RESERVE_SEC,
                 "max_model_len": E21_VLLM_MAX_MODEL_LEN, "served": E21_SERVED, "ports": E21_PORTS,
                 "stage": [str(p) for p in E21_STAGE_PATHS], "vllm032_wheelhouse": str(E18_VLLM_WHEELHOUSE),
                 "reasoning_history_key": "reasoning"}
# rev 2 (VAL_fidelity_E2 LOW-1, E1 LOW-1): CONFIG was snapshot above before this block; record what really runs
CONFIG["e11"]["ENGINE_BACKEND"] = ENGINE_BACKEND
CONFIG["e12"]["E12_SOLVE_WINDOW_SEC"] = E12_SOLVE_WINDOW_SEC
CONFIG["e12"].setdefault("rev6", {})["E12_R6_CACHE_DIAG"] = 0
CONFIG["e12"]["E12_STREAMS"] = "per half (CONFIG['e21'] / e12_run.streams)"
print("E21:", json.dumps(CONFIG["e21"], default=str), flush=True)


In [ ]:
# ==================================================================== CELL 1b — E14: first-line facts, phase record, staging
# Experiment 14 (Notes/07, 11-13). Runs before any heavy work: what the machine is (mounts / GCSFUSE, df, RAM, cgroup,
# CPU, a timed 64 MiB read), a phase record written at every change (e14_state.json) and a heartbeat, then the model
# copy to RAM/local disk with ranged parallel readers (verified, published atomically, fail open). The E14-P fix (raw
# reasoning in Duck's history, Notes/12) is switched on for every profile of this notebook.
import io as _e14_io
E14_DIR = Path(os.environ.get("E14_DIR", "/tmp/e14")); E14_DIR.mkdir(parents=True, exist_ok=True)
Q27X_PAYLOAD_SLUG = "esfersami50/q27-e15b-payload"


def q27x_payload(name, sha):
    """One file of the payload dataset (Notes/21: Kaggle commit runs of a 745 kB notebook never started, so the big
    payloads live in a dataset), found by its exact name under Q27X_PAYLOAD_DIR or /kaggle/input (at most 3 directory
    levels deep) and verified by sha256 before use."""
    roots = [Path(x) for x in os.environ.get("Q27X_PAYLOAD_DIR", "").split(os.pathsep) if x] + [Path("/kaggle/input")]
    seen = []
    for r in roots:
        for pat in (name, "*/" + name, "*/*/" + name, "*/*/*/" + name):
            for f in (sorted(r.glob(pat)) if r.is_dir() else []):
                b = f.read_bytes()
                if hashlib.sha256(b).hexdigest() == sha:
                    return b
                seen.append(str(f))
    raise FileNotFoundError(f"payload {name} (sha256 {sha[:12]}) not found or wrong under {[str(r) for r in roots]}; "
                            f"seen {seen}; attach the dataset {Q27X_PAYLOAD_SLUG}")


_E14_MOD_SHA = {"e14_loadpath.py": "7c66c179e5bb6bff7c909d0a96d4b7058c4bc0b2ff368b04367ee9f54e8fc5fc", "e14_measure.py": "ae61c9f9a03419d73dae2ba2d2aa4d5d1f4403780911832148d2dd8e8a3440d0", "e14_sglang_patches.py": "beb850abe2c072c34df0ae733642557cc44e2d808775912f291f072d29e12f11", "e14_nccl_bench.py": "355adda0d469c5ecac5abb94ff7608e2f256acc6a5dcf745e28b4d3676f03cb2", "e14_trace_summary.py": "8d89ca106e166c97204f1aeff931a50a75e203999c876f248b34e6ee24212b46"}
with zipfile.ZipFile(_e14_io.BytesIO(q27x_payload("q27x_e14_modules_55c1eb7911c6.bin", "55c1eb7911c68ba43d3c56b9c32bb00e581ccb96fe3b5b5548c5cdeb47125040"))) as _z:
    for _n, _h in _E14_MOD_SHA.items():
        (E14_DIR / _n).write_bytes(_z.read(_n))
        assert hashlib.sha256((E14_DIR / _n).read_bytes()).hexdigest() == _h, _n
sys.path.insert(0, str(E14_DIR))
import e14_loadpath as E14L, e14_measure as E14M, e14_sglang_patches as E14P, e14_trace_summary as E14T
E14_OUT = OUT / "e14"; E14_OUT.mkdir(parents=True, exist_ok=True)
E14_RESULTS = {"build": {"variant": "E15B", "arc2_duck_sha": "b9f67010237becf9a0c2631e14f48acd97f5883b4ef357c93becf4fa16543128", "modules": _E14_MOD_SHA,
                         "payload": {"dataset": Q27X_PAYLOAD_SLUG, "q27x_e14_modules_55c1eb7911c6.bin": "55c1eb7911c68ba43d3c56b9c32bb00e581ccb96fe3b5b5548c5cdeb47125040",
                                     "q27x_e15b_duck_9e4b89285136.bin": "9e4b89285136616d523028ac042dcca6afe82175081a1481d584c998ab3ee566"}},
               "phases": {}, "decisions": {}}
E14_DO_MEASURE = bool(P.get("e14")) and not RERUN
E14_STATE = E14L.E14State(WORKING / "e14_state.json")
E14_STATE.enter("PREPARE", deadline=HARD_STOP, profile=PROFILE, rerun=RERUN, measure=E14_DO_MEASURE)


def e14_save():
    try:
        write_json_atomic(E14_OUT / "e14_results.json", E14_RESULTS)
        write_json_atomic(WORKING / "e14_results.json", E14_RESULTS)
    except Exception as _e:
        print("E14 results write failed:", repr(_e)[:200], flush=True)


def _e14_heartbeat(every=60):
    while True:
        try:
            with open(WORKING / "e14_heartbeat.log", "a", encoding="utf-8") as _f:
                _f.write(f"{time.strftime('%H:%M:%S')} +{(time.perf_counter() - NOTEBOOK_T0) / 60:.1f} min "
                         f"phase {E14_STATE.rec.get('current')}\n")
        except Exception:
            pass
        time.sleep(every)


threading.Thread(target=_e14_heartbeat, daemon=True).start()
E14_RESULTS["env"] = E14L.env_report([str(SGL_MODEL_PATH), str(SGL_DRAFT_PATH), str(SGL_WHEELHOUSE), str(MODEL_PATH),
                                      "/kaggle/input"])
try:
    E14_RESULTS["read_probe"] = E14L.read_probe(E21_STAGE_PATHS[0])   # E21: the weights this variant loads
except Exception as _e:
    E14_RESULTS["read_probe"] = {"error": repr(_e)[:200]}
print("E14 env:", json.dumps(E14_RESULTS["env"])[:1500], "| probe:", json.dumps(E14_RESULTS["read_probe"]), flush=True)
E14_STAGING = None
if ENGINE_BACKEND in ("sglang", "vllm") and E14L.loadpath_enabled():       # E21: vLLM halves too
    _e14_dest = tuple(x for x in os.environ.get("E14_DEST", "/dev/shm,/tmp").split(",") if x)
    E14_STAGING = E14L.Staging(E21_STAGE_PATHS, workers=24, stall_sec=600.0,
                               deadline=max(NOTEBOOK_T0 + 10 * 60, min(NOTEBOOK_T0 + 45 * 60, SGL_GIVE_UP_AT - 20 * 60)), log=print, state=E14_STATE,
                               dest_candidates=_e14_dest)   # E15B #27
    E14_STAGING.start()
    print("E14 staging started ->", _e14_dest, flush=True)
os.environ["E12_RAW_REASONING_HISTORY"] = "1"          # E14-P (Notes/12): Duck keeps the server's raw reasoning
E14_VENV = {}


def e14_after_install(vpy, env):
    """After the SGLang venv exists: the NCCL bench (commit measurement only; the GPUs are idle while the copy runs),
    then the staged model paths (fail open: the original /kaggle/input paths)."""
    global SGL_MODEL_PATH, SGL_DRAFT_PATH
    E14_VENV.update(vpy=str(vpy), env=dict(env))
    if E14_DO_MEASURE and not E11_HARNESS_E14:
        E14_STATE.enter("NCCL_BENCH")
        _t = time.perf_counter()
        _o = E14_OUT / "nccl_bench.json"
        try:
            _r = subprocess.run([str(vpy), str(E14_DIR / "e14_nccl_bench.py"), "--out", str(_o),
                                 "--arms", "baseline,p2p_sys,shm_memcpy", "--timeout-per-arm", "240"],
                                env=env, capture_output=True, text=True, timeout=900)
            E14_RESULTS["nccl"] = json.loads(_o.read_text()) if _o.exists() else {"rc": _r.returncode,
                                                                                  "stderr": (_r.stderr or "")[-2000:]}
        except Exception as _e:
            E14_RESULTS["nccl"] = {"error": repr(_e)[:300]}
        E14_RESULTS["nccl_sec"] = round(time.perf_counter() - _t, 1)
        e14_save()
        print("E14 NCCL bench:", json.dumps((E14_RESULTS.get("nccl") or {}).get("summary"))[:800], flush=True)
    elif E14_DO_MEASURE and E11_HARNESS_E14:
        _o = E14_OUT / "nccl_bench.json"
        subprocess.run([sys.executable, str(E14_DIR / "e14_nccl_bench.py"), "--out", str(_o), "--fake"], timeout=120)
        E14_RESULTS["nccl"] = json.loads(_o.read_text()) if _o.exists() else {"error": "fake bench wrote nothing"}
    if E14_STAGING is not None:
        E14_STATE.enter("STAGING_JOIN")
        E14_STAGING.join(timeout=max(0.0, min(SGL_GIVE_UP_AT - 20 * 60, NOTEBOOK_T0 + 55 * 60) - time.perf_counter()))
        res = E14_STAGING.result
        if res is None:
            E14_RESULTS["staging"] = {"still_running": True, "note": "launch from the original paths"}
        else:
            paths, rep = res
            E14_RESULTS["staging"] = rep
            SGL_MODEL_PATH = Path(paths.get(str(SGL_MODEL_PATH), str(SGL_MODEL_PATH)))
            SGL_DRAFT_PATH = Path(paths.get(str(SGL_DRAFT_PATH), str(SGL_DRAFT_PATH)))
        E14_RESULTS["launch_paths"] = {"model": str(SGL_MODEL_PATH), "draft": str(SGL_DRAFT_PATH)}
        e14_save()
        print("E14 staging:", json.dumps(E14_RESULTS["staging"])[:1200], flush=True)
    E14_STATE.enter("LAUNCH", deadline=SGL_GIVE_UP_AT)


E11_HARNESS_E14 = os.environ.get("E11_HARNESS") == "1"
E14_TS = float(os.environ.get("E14_TIME_SCALE", "1"))    # the CPU harness shortens every window and phase cap; 1 on Kaggle
_ts = lambda s: max(1.5, float(s) * E14_TS)
CONFIG["e14"] = {"measure": E14_DO_MEASURE, "staging": E14_STAGING is not None, "raw_reasoning": True,
                 "submit_arm": "E21: vLLM halves, see CONFIG['e21']",
                 "modules": _E14_MOD_SHA, "arc2_duck_sha": "b9f67010237becf9a0c2631e14f48acd97f5883b4ef357c93becf4fa16543128"}
e14_save()


In [ ]:
# ==================================================================== CELL 2 — offline install (PIV/V3.1 parity block) + tokenizer
import importlib, importlib.metadata
def pkg_version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None

EXPECTED_RUNTIME_PACKAGES = {"vllm": "0.19.0", "transformers": "4.57.6", "tokenizers": "0.22.2", "huggingface-hub": "0.36.2"}
_loaded = {"vllm": "vllm", "transformers": "transformers", "tokenizers": "tokenizers", "huggingface-hub": "huggingface_hub"}
before_stack = {n: pkg_version(n) for n in EXPECTED_RUNTIME_PACKAGES}

def offline_pip_install(requirements):
    if not requirements: return
    if not WHEELHOUSE.exists():
        raise FileNotFoundError(f"wheelhouse not attached: {WHEELHOUSE}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--no-cache-dir",
                           "--find-links", str(WHEELHOUSE), *requirements])

# S1: read the model files into the page cache in the background while pip runs (read-only, best effort)
if os.environ.get("Q27X_SKIP_INSTALL") != "1" and E14_STAGING is None:   # E14: ranged staging replaces the cat
    try:
        import shlex as _shlex
        _wdirs = (SGL_MODEL_PATH, SGL_DRAFT_PATH) if ENGINE_BACKEND == "sglang" else (MODEL_PATH,)   # E11-05
        _wfiles = sorted(str(p) for _d in _wdirs for p in Path(_d).glob("*.safetensors"))
        if _wfiles:
            subprocess.Popen(["bash", "-c", "cat " + " ".join(_shlex.quote(p) for p in _wfiles) + " > /dev/null"],
                             stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print("weight prefetch started:", len(_wfiles), "files")
    except Exception as _e:
        print("weight prefetch not started:", repr(_e)[:200])

if os.environ.get("Q27X_SKIP_INSTALL") != "1":     # Q27X_SKIP_INSTALL is for the local harness only
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError(f"Python {sys.version.split()[0]}: the cp312 wheelhouse needs the pinned Python 3.12 Kaggle image")
    for _pkg, _mod in _loaded.items():
        if before_stack.get(_pkg) != EXPECTED_RUNTIME_PACKAGES[_pkg] and _mod in sys.modules:
            raise RuntimeError(f"{_pkg} already imported at {before_stack.get(_pkg)!r}; restart the session before Run All.")
    _tok_pkgs = [f"{p}=={EXPECTED_RUNTIME_PACKAGES[p]}" for p in ("transformers", "tokenizers", "huggingface-hub")
                 if before_stack[p] != EXPECTED_RUNTIME_PACKAGES[p]]
    if ENGINE_BACKEND == "sglang" and before_stack["vllm"] != EXPECTED_RUNTIME_PACKAGES["vllm"]:
        # E11-05: the kernel only tokenizes (S49 stack); SGLang runs in its own venv; vLLM is installed only for the fallback
        try:
            offline_pip_install(_tok_pkgs)
        except Exception as _e:
            print("tokenizer-only install failed -> the full S49 install:", repr(_e)[:300], flush=True)
            offline_pip_install(["vllm==0.19.0"])
    elif before_stack["vllm"] != EXPECTED_RUNTIME_PACKAGES["vllm"]:
        offline_pip_install(["vllm==0.19.0"])
    else:
        offline_pip_install(_tok_pkgs)
    after_stack = {n: pkg_version(n) for n in EXPECTED_RUNTIME_PACKAGES}
    for _p, _want in EXPECTED_RUNTIME_PACKAGES.items():
        if _p == "vllm" and ENGINE_BACKEND == "sglang":
            continue                                   # E11-05: checked by the fallback cell, only if it runs
        if after_stack.get(_p) != _want:
            raise RuntimeError(f"runtime parity failure: {_p}={after_stack.get(_p)!r}, expected {_want!r}")
    print("runtime parity PASS:", after_stack)

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), trust_remote_code=False)
_model_cfg = _load_json(MODEL_PATH / "config.json")
assert _model_cfg.get("model_type") == "qwen3_5", _model_cfg.get("model_type")
assert (_model_cfg.get("quantization_config") or {}).get("quant_method") == "fp8"
THINK_CLOSE_IDS = tokenizer.encode("</think>", add_special_tokens=False)
assert len(THINK_CLOSE_IDS) == 1, THINK_CLOSE_IDS
THINK_CLOSE_ID = THINK_CLOSE_IDS[0]
STOP_IDS = tokenizer.encode(STOP_THINKING_TEXT, add_special_tokens=False)
assert THINK_CLOSE_ID in STOP_IDS
_gen_cfg = _load_json(MODEL_PATH / "generation_config.json") if (MODEL_PATH / "generation_config.json").exists() else {}
_eos = _gen_cfg.get("eos_token_id", [])
EOS_IDS = set(_eos if isinstance(_eos, list) else [_eos])     # v1.1 (F8): stripped before a forced close
print("tokenizer OK | </think> id", THINK_CLOSE_ID, "| stop text", len(STOP_IDS), "tokens")

# E11-05: the SGLang server receives token ids made by this (S49) tokenizer. Check that the INT4 checkpoint has the same
# vocabulary (with added tokens) and tokenizes a rendered prompt identically; otherwise run the S49 vLLM engine.
E11_TOKENIZER_CHECK = {"ok": None, "skipped": ENGINE_BACKEND != "sglang"}
if ENGINE_BACKEND == "sglang":
    try:
        from tokenizers import Tokenizer as _HFTokenizer
        _t4 = _HFTokenizer.from_file(str(SGL_MODEL_PATH / "tokenizer.json"))
        _probe = tokenizer.apply_chat_template(
            [{"role": "user", "content": "Example 1\nInput:\n0 1 2\n3 4 5\nOutput:\n5 4 3\n```python\ndef transform(grid):\n    return grid\n```"}],
            tokenize=False, add_generation_prompt=True, enable_thinking=True, reasoning_effort=REASONING_EFFORT)
        _probe += "Let me think.\n</think>\n\n```python\nimport numpy as np\n```<|im_end|>"
        # rev 2: + a real ARC prompt fragment (every train pair of the first test task, rendered as digit rows)
        _t0 = {"train": [{"input": [[0, 1, 2], [3, 4, 5]], "output": [[5, 4, 3], [2, 1, 0]]}]}   # E15B #17: fixed
        _probe += "".join(f"\nExample {_i + 1}\nInput:\n" + "\n".join(" ".join(str(v) for v in _row) for _row in _p["input"])
                          + "\nOutput:\n" + "\n".join(" ".join(str(v) for v in _row) for _row in _p["output"])
                          for _i, _p in enumerate(_t0["train"]))
        _ids_s49 = tokenizer.encode(_probe, add_special_tokens=False)
        _ids_int4 = _t4.encode(_probe, add_special_tokens=False).ids
        _same_vocab = _t4.get_vocab(with_added_tokens=True) == tokenizer.backend_tokenizer.get_vocab(with_added_tokens=True)
        # rev 2: the BPE merges too (the two files store them in different formats: "a b" strings vs [a, b] pairs)
        _norm = lambda m: [tuple(x.split(" ", 1)) if isinstance(x, str) else tuple(x) for x in m]
        _same_merges = (_norm(_load_json(SGL_MODEL_PATH / "tokenizer.json")["model"]["merges"])
                        == _norm(_load_json(MODEL_PATH / "tokenizer.json")["model"]["merges"]))
        _int4_gen = _load_json(SGL_MODEL_PATH / "generation_config.json") if (SGL_MODEL_PATH / "generation_config.json").exists() else {}
        _e4 = _int4_gen.get("eos_token_id", [])
        _same_eos = set(_e4 if isinstance(_e4, list) else [_e4]) == EOS_IDS
        E11_TOKENIZER_CHECK = {"ok": bool(_same_vocab and _same_merges and _ids_s49 == _ids_int4 and _same_eos),
                               "same_vocab": _same_vocab, "same_merges": _same_merges, "same_probe_ids": _ids_s49 == _ids_int4,
                               "probe_tokens": len(_ids_s49), "same_eos": _same_eos}
    except Exception as _e:
        E11_TOKENIZER_CHECK = {"ok": False, "error": f"{type(_e).__name__}: {_e}"[:300]}
    if not E11_TOKENIZER_CHECK["ok"]:
        print("E11 tokenizer check FAILED -> S49 vLLM FP8 engine:", E11_TOKENIZER_CHECK, flush=True)
        ENGINE_BACKEND = "vllm"
CONFIG["e11"]["tokenizer_check"] = E11_TOKENIZER_CHECK
print("E11 tokenizer check:", E11_TOKENIZER_CHECK)


In [ ]:
# ==================================================================== CELL 3 — E12: Duck package (verbatim) + ARC-2 adapter
# Tufa Labs' Duck harness as forked by Daniel Franzen (da-fr/arc-agi-3-solution @ 10882e332d, Apache-2.0,
# LICENSE + NOTICE inside). E15B: inference/agent/tool_agent.py (yield after the batch, ARC3_YIELD_AFTER_BATCH) and
# python_tool_sandbox.py (sys shim / time / import messages, ARC2_SANDBOX_SYS_SHIM) carry two switched-off-by-default
# patches (Exp 15 src/duck_b); the other 18 files are byte-identical to that commit (sha256 manifest below);
# arc2_duck.py is the only new code (ARC-2 prompt, prelude with check/submit, host re-verification, driver, selection).
import base64 as _b64, io as _io, logging as _logging
E12_DIR = Path(os.environ.get("E12_DIR", "/tmp/e12_duck"))
_E12_ZIP_BYTES = q27x_payload("q27x_e15b_duck_9e4b89285136.bin", "9e4b89285136616d523028ac042dcca6afe82175081a1481d584c998ab3ee566")   # Notes/21: payload dataset
# E15B #17: the sandbox self-test task (identity on every pair): fixed, so an unusual first hidden task cannot fail it
E15B_SELFTEST_TASK = {"train": [{"input": [[1, 0], [0, 2]], "output": [[1, 0], [0, 2]]},
                                {"input": [[3, 3, 0]], "output": [[3, 3, 0]]}], "test": [{"input": [[4]]}]}
_E12_MANIFEST = {
"arc2_duck.py": "b9f67010237becf9a0c2631e14f48acd97f5883b4ef357c93becf4fa16543128",
"vendor/LICENSE": "3ddf9be5c28fe27dad143a5dc76eea25222ad1dd68934a047064e56ed2fa40c5",
"vendor/NOTICE.txt": "1e2b756570ed5a6d85d06e61e458021c1221a5f72ad33165ea29bb85b9053cf1",
"vendor/inference/__init__.py": "d92fed573b404870f7c6a1cef8f3e5d696195e2886e9e9183485c304fe5090f8",
"vendor/inference/agent/__init__.py": "8c561093771390b6f64fc0e87186982cd04045e8b6a2cf8d7f17e494762b4385",
"vendor/inference/agent/action_names.py": "8dd40e957046977e811eb2c2eb13fce905dbecf77f8dc7f846540f4d48af963d",
"vendor/inference/agent/noop_repeat_guard.py": "e9172d58687e9043c3445f87e11bdf74c4104d8b0bed5fb6b54745a6aa58655a",
"vendor/inference/agent/priority_pace_reference.json": "f29c16e336aef6b0dbf73ec056f554bd5192ac02f8deeda70ad152ef1214d0ab",
"vendor/inference/agent/priority_scheduler.py": "3a13659001458ea1a45ee3a812babe1b0012409eeabb23d681395796941b8e94",
"vendor/inference/agent/prompts.py": "14965a99bc55a61dad59fb79038f56c0ba1d1368350c8487b5a7b9edef0488ac",
"vendor/inference/agent/python_tool_sandbox.py": "d29d807eef107a2d9aa60354dc370782739e2b9604a8247f89279ee10ae0bf3c",
"vendor/inference/agent/runtime_state.py": "b5c55d561a63e2dcf0422ee7fd339923a9fb909c779302bbf02b8ea49ccc607f",
"vendor/inference/agent/tool_agent.py": "f7454201484f2416b9c9a7d3a365bc048901e4a384a00ce1e1a2e6ec458c7b4c",
"vendor/inference/agent/vision_context.py": "51d54a7aa9f1cd8e41b3fcc00ffd1a9cd1ada6fcde261d09faa49561fd843466",
"vendor/inference/utils/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
"vendor/inference/utils/animation.py": "6c14b613b3a7c66ce3373162d6faa1469a1ddb3ca4ab882299998c5cf14add96",
"vendor/inference/utils/frame_diff.py": "6ac3112773ffa6b1562b1e8f35a8018bd0dec9de7a78b12e7b13c529d4f46af8",
"vendor/inference/utils/grid_utils.py": "1e43809c9a4a463c65f897c48abb8e6cf4b28a32ac4ec32a07e9adcf06ab0c03",
"vendor/inference/utils/openai_compat.py": "fea0a75081372769fd854997b99e7826b334dcc5966876f31f587d19ae961165",
"vendor/inference/utils/retained_functions.py": "0d8b267cb387565c04d4977116020cc04d84ad9d9175d2bf4bc4613703cc5a01",
"vendor/inference/utils/run_artifacts.py": "789d541b395043379c23f5b0b24044e50fadab02e7f86f64cdd54714f4cab339",
"vendor/inference/utils/segmentation.py": "0d19669e5a36351e41a6b94b7c68f5b258829de78441a03a8749687c262b452c",
"vendor/inference/utils/viewer_artifacts.py": "cb5043dab8f5294608cda3675408ced904b849ae4985829b4ffd2d84b8e5d807"
}
if E12_DIR.exists():
    shutil.rmtree(E12_DIR)
with zipfile.ZipFile(_io.BytesIO(_E12_ZIP_BYTES)) as _z:
    _z.extractall(E12_DIR)
_bad = [p for p, h in _E12_MANIFEST.items() if hashlib.sha256((E12_DIR / p).read_bytes()).hexdigest() != h]
assert not _bad, ("E12 package hash mismatch", _bad)
sys.path[:0] = [str(E12_DIR), str(E12_DIR / "vendor")]
import arc2_duck as E12A
# Duck reads its settings at import: set them first. The base URL is the engine cell's (same port); the stream count is
# set again from the engine's concurrency before the gate is first used (read at call time).
os.environ.update(E12A.duck_env(f"http://127.0.0.1:{int(os.environ.get('E11_PORT', 30000))}/v1", "pending", SGL_MAX_RUNNING))
# E21: vLLM 0.19 / 0.32 read only message["reasoning"] of an assistant history message (vllm/entrypoints/chat_utils.py
# 1512 in v0.19.0; vllm/renderers/chat_utils.py 2083 in v0.32.0); "reasoning_content" (DUCK_ENV, read by SGLang) would
# be dropped from the prompt. E12-V run 2 sent "reasoning" and its prompts grew by the generated tokens (6,882 + 12,288
# -> 19,215). Duck reads the key at call time (tool_agent._reasoning_history_keys).
os.environ["ARC3_REASONING_HISTORY_KEY"] = "reasoning"
_E12_TA, _E12_DP, _e12_run_sandboxed, _ = E12A._import_duck()
# E21 rev 2 (VAL_api_D1 F1, verified in the vLLM source): vLLM 0.19 / 0.32 report finish_reason "tool_calls" whenever
# the parser found a call, also for a reply cut at max_tokens (chat_completion/serving.py 0.19 L1550-1560, 0.32
# L1099-1116); SGLang 0.5.21 kept "length", which arc2_duck's cut-call handling (rev 5 B6 / R5-2, the cut-code harvest,
# the R6 "cut" opener) keys on. Restore "length" from the token count (Duck's package files stay byte-identical).
_E21_CC_ORIG = _E12_TA.ToolAgent._chat_completion
E21_LENGTH_RESTORED = {"n": 0}
E21_CALL_WITH_CONTENT = {"n": 0}     # rev 2b (VAL_api_D1 F4): a tool call + visible text


def _e21_chat_completion(self, messages, **kw):
    res = _E21_CC_ORIG(self, messages, **kw)
    try:
        u = res.usage if isinstance(res.usage, dict) else {}
        cap = kw.get("max_tokens")
        cap = self._max_output_tokens if cap is None else cap
        ct, pt = u.get("completion_tokens"), u.get("prompt_tokens")
        _m = res.message if isinstance(res.message, dict) else {}
        if _m.get("tool_calls") and str(_m.get("content") or "").strip():
            E21_CALL_WITH_CONTENT["n"] += 1
        if str(res.finish_reason or "") == "tool_calls" and cap and isinstance(ct, int):
            lim = int(cap) if not isinstance(pt, int) else min(int(cap), E21_VLLM_MAX_MODEL_LEN - int(pt))   # 0.32 clamps
            if ct >= lim:
                res.finish_reason = "length"
                E21_LENGTH_RESTORED["n"] += 1
    except Exception:
        pass
    return res


_E12_TA.ToolAgent._chat_completion = _e21_chat_completion
E12_SANDBOX_LIMIT = E12A.install_sandbox_memory_limit(E12_SANDBOX_MEM_MB)   # rev 2: every sandbox child, before its code


class _E12LevelCountFilter(_logging.Filter):   # Duck's per-game level-count warning has no ARC-2 meaning (priority uses 10)
    def filter(self, record):
        return "total level count unavailable" not in record.getMessage()


_logging.getLogger(_E12_TA.__name__).addFilter(_E12LevelCountFilter())

# sandbox self-test with our own scripted code (the engine cell runs only if it passes)
_t = time.perf_counter()
try:
    _st_task = E15B_SELFTEST_TASK      # E15B #17 (code audit M4): a fixed task, never the first hidden task
    _st = _e12_run_sandboxed(
        code=E12A.task_prelude(_st_task) + "def _e12_ident(g):\n    return g\nres = check(_e12_ident, verbose=False)\nprint(len(res), sum(1 for r in res if 'ok' in r))",
        timeout_seconds=30, initial_state={"current_frame": None, "history": [], "valid_actions": [], "last_action_call_result": {}},
        action_handler=lambda a, stale_after=None: {"action_result": {}, "state": {}}, kept_functions=[], retain_imports=True)
    _n = len(_st_task["train"])
    E10_SANDBOX_SELFTEST = {"ok": (_st.get("stdout") or "").strip() == f"{_n} {_n}" and not _st.get("error"),
                            "stdout": (_st.get("stdout") or "")[:200], "error": (_st.get("error") or "")[:500],
                            "sec": round(time.perf_counter() - _t, 2), "python": sys.version.split()[0]}
except Exception as _e:
    E10_SANDBOX_SELFTEST = {"ok": False, "error": f"{type(_e).__name__}: {_e}"[:500]}
# rev 2: memory limit evidence (reported; the run continues without it rather than not at all). The 5 GiB request is
# made only after the read-back confirmed the limit, so it can never really allocate.
_mem = {"limit": dict(E12_SANDBOX_LIMIT)}
try:
    _st_kw = dict(timeout_seconds=30, initial_state={"current_frame": None, "history": [], "valid_actions": [],
                                                     "last_action_call_result": {}},
                  action_handler=lambda a, stale_after=None: {"action_result": {}, "state": {}}, kept_functions=[],
                  retain_imports=True)
    if E12_SANDBOX_LIMIT.get("readback_ok"):
        _big = _e12_run_sandboxed(code="x = bytearray(5 * 1024 ** 3)\nprint(len(x))", **_st_kw)
        _small = _e12_run_sandboxed(code="x = bytearray(128 * 1024 ** 2)\nprint(len(x))", **_st_kw)
        _mem.update(over_limit_rejected="MemoryError" in (_big.get("error") or ""),
                    small_ok=(_small.get("stdout") or "").strip() == str(128 * 1024 ** 2))
        _mem["ok"] = bool(_mem["over_limit_rejected"] and _mem["small_ok"])
    else:
        _mem["ok"] = False
except Exception as _e:
    _mem.update(ok=False, error=f"{type(_e).__name__}: {_e}"[:300])
_mem["limit"] = dict(E12_SANDBOX_LIMIT)
E10_SANDBOX_SELFTEST["memory_limit"] = _mem
if E12_SANDBOX_MEM_MB > 0 and not _mem.get("ok"):
    print("E12 WARNING: sandbox memory limit NOT verified:", json.dumps(_mem), flush=True)
    if os.name == "posix":   # rev 3 (GPT re-audit): on Kaggle (Linux) never run model code without the limit
        E10_SANDBOX_SELFTEST["ok"] = False
        E10_SANDBOX_SELFTEST["error"] = "memory limit not verified (fail closed): " + json.dumps(_mem)[:300]
        print("E12: solver disabled (fail closed); CELL 0 fallback submission stays", flush=True)
CONFIG["e12"]["sandbox_selftest"] = E10_SANDBOX_SELFTEST
if _E12_REV5:   # rev 5 R5-4: is B3 counting reasoning with the tokenizer (else a fallback ratio)? recorded, never fatal
    CONFIG["e12"]["rev5_tokenizer"] = E12A.tokenizer_status()
    print("E12 rev 5 reasoning tokenizer:", json.dumps(CONFIG["e12"]["rev5_tokenizer"]), flush=True)
print("E12 package OK:", len(_E12_MANIFEST), "files | sandbox self-test:", json.dumps(E10_SANDBOX_SELFTEST), flush=True)


In [ ]:
# E10: original task-order and evaluation profiles, but one Poetiq program per task/expert/iteration.
if P['source']=='eval':
    _ch=_load_json(ARC_ROOT/'arc-agi_evaluation_challenges.json')
    _sol=_load_json(ARC_ROOT/'arc-agi_evaluation_solutions.json')
    if P.get('task_list'):
        assert all(t in _ch for t in P['task_list']), 'Evaluation task ID missing'
        TASK_IDS=sorted(P['task_list'])
    else:
        TASK_IDS=sorted(random.Random(TASK_SAMPLE_SEED).sample(sorted(_ch),P['n_tasks']))
    CHALLENGES={t:_ch[t] for t in TASK_IDS}
    SOLUTIONS={t:_sol[t] for t in TASK_IDS}
else:
    CHALLENGES=dict(test_challenges)
    TASK_IDS=sorted(CHALLENGES) # no Codex ordering
    SOLUTIONS=None
N_HIDDEN_UNITS=sum(len(t['test']) for t in test_challenges.values())
print('E10 Poetiq:',len(TASK_IDS),'tasks,',sum(len(CHALLENGES[t]['test']) for t in TASK_IDS),
      'queries | experts',POETIQ_NUM_EXPERTS,'iterations per expert',POETIQ_MAX_ITERATIONS)


In [ ]:
# ==================================================================== CELL 5 — E21: vLLM engine library (two halves)
# Donor functions, verbatim (build_e21.py extracts and asserts them):
#   E12-V CELL 5 (Q27-XHIGH-E12_DUCK_REPL_VLLM_SMOKE1 e85b69c2 = the vLLM SMOKE1 run 2 Duck server): ensure_libcuda_linkable,
#     _e12v_request_json, the Prometheus reader + vLLM->SGLang name aliases (metrics_snapshot), the stats-line summary
#     (e11_decode_summary), e11_kill_server;
#   E18 R2 CELL 5 (116d2d41): e18_install (the VMS vLLM 0.32 installer), E18Server (ready = /health + its own KV line,
#     stall on 600 s of log silence; kill = SIGTERM then always SIGKILL the group), _e18_kill_gpu_pids, _wait_gpu_release
#     (= S1-60K CELL 5).
# New (E21): the per-half flag sets (vLLM 0.19 = the E12-V command; vLLM 0.32 = the VMS arm), the capacity check, one
# launch with its smoke, the per-half begin / end and the watchdog relaunch.
import urllib.request as _urlreq, signal as _signal, statistics as _statistics
import shutil as _shutil
E11_HARNESS = os.environ.get("E11_HARNESS") == "1"            # local CPU harness only: fake server, no GPU
E18_URL = None
SGL_URL = None
ENGINE_READY = False; llm = None; engine = None; CONCURRENCY = 0; E11_SERVER = None; E12_VLLM_CONCURRENCY = 0
ENGINE_INFO = {"backend": None}

# ---- E12-V CELL 5, verbatim
def ensure_libcuda_linkable(candidates=None, workdir="/tmp/q27x_libcuda_shim"):
    info = {"needed": None, "real": None, "shim": None, "probe": "FAIL"}
    try:
        work = Path(workdir); work.mkdir(parents=True, exist_ok=True)
        if candidates is None:
            existing = [Path(d, "libcuda.so") for d in ("/usr/local/cuda/lib64", "/usr/local/cuda/lib64/stubs",
                                                        "/usr/lib/x86_64-linux-gnu", "/usr/lib64")]
            existing = [p for p in existing if p.exists()]
        else:
            existing = []
        if existing:
            info.update(needed=False, real=str(existing[0]))
        else:
            cands = list(candidates) if candidates is not None else []
            if candidates is None:
                for d in ("/usr/local/nvidia/lib64", "/usr/local/nvidia/lib", "/usr/lib/x86_64-linux-gnu", "/usr/lib64",
                          "/usr/local/cuda/compat"):
                    try:
                        cands += sorted(str(p) for p in Path(d).glob("libcuda.so.*") if p.is_file())
                    except Exception:
                        pass
                try:
                    for line in subprocess.run(["ldconfig", "-p"], capture_output=True, text=True, timeout=30).stdout.splitlines():
                        if "libcuda.so.1" in line and "=>" in line:
                            cands.append(line.split("=>")[-1].strip())
                except Exception:
                    pass
            real = next((Path(c) for c in cands if Path(c).is_file()), None)
            if real is None:
                info["reason"] = "libcuda.so.1 not found"
                return info
            link = work / "libcuda.so"
            if link.exists() or link.is_symlink():
                link.unlink()
            try:
                link.symlink_to(real.resolve())
            except OSError:
                _shutil.copyfile(real, link)
            old = os.environ.get("LIBRARY_PATH", "")
            os.environ["LIBRARY_PATH"] = f"{work}:{old}" if old else str(work)
            info.update(needed=True, real=str(real), shim=str(link), library_path=os.environ["LIBRARY_PATH"])
        cxx = _shutil.which("c++") or _shutil.which("g++")
        if not cxx:
            info["reason"] = "no C++ compiler for the link probe"
            return info
        cp = subprocess.run([cxx, "-shared", "-fPIC", "-x", "c++", "-", "-lcuda", "-o", str(work / "probe.so")],
                            input='extern "C" int q27x_probe() { return 0; }\n', text=True, capture_output=True,
                            timeout=120, env=os.environ.copy())
        info["probe"] = "PASS" if cp.returncode == 0 else "FAIL"
        if cp.returncode != 0:
            info["probe_stderr"] = (cp.stderr or "")[-1500:]
    except Exception as e:
        info["reason"] = f"{type(e).__name__}: {e}"[:500]
    return info


# ---- E12-V CELL 5, verbatim
def _e12v_request_json(url, payload=None, timeout=30):   # = Duck's request_json
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    req = _urlreq.Request(url, data=data, headers={"Content-Type": "application/json"})
    with _urlreq.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8"))


# ---- E12-V CELL 5, verbatim
_E12V_PROM_RE = re.compile(r"^([A-Za-z_:][A-Za-z0-9_:]*)(\{[^}]*\})?\s+(\S+)")


# ---- E12-V CELL 5, verbatim
_E12V_ALIAS = {"vllm:num_requests_running": "num_running_reqs", "vllm:num_requests_waiting": "num_queue_reqs",
               "vllm:kv_cache_usage_perc": "token_usage"}


# ---- E12-V CELL 5, verbatim
def metrics_snapshot():
    if not ENGINE_READY:
        return []
    try:
        txt = _urlreq.urlopen(SGL_URL + "/metrics", timeout=10).read().decode("utf-8", "replace")
        rows = []
        for line in txt.splitlines():
            m = _E12V_PROM_RE.match(line) if line and not line.startswith("#") else None
            if not m or m.group(1).endswith(("_bucket", "_created")):
                continue
            name = m.group(1)
            if name in _E12V_ALIAS or any(k in name for k in ("prefix_cache", "spec_decode", "generation_tokens_total",
                                                               "prompt_tokens_total", "num_preemptions")):
                try:
                    rows.append({"name": _E12V_ALIAS.get(name, name), "labels": m.group(2), "value": float(m.group(3))})
                except ValueError:
                    pass
        return rows
    except Exception as e:
        return [{"error": repr(e)[:300]}]


# ---- E12-V CELL 5, verbatim
_E12V_STAT_RE = re.compile(r"Avg prompt throughput: ([\d.]+) tokens/s, Avg generation throughput: ([\d.]+) tokens/s, "
                           r"Running: (\d+) reqs, Waiting: (\d+) reqs, GPU KV cache usage: ([\d.]+)%, Prefix cache hit rate: ([\d.]+)%")


# ---- E12-V CELL 5, verbatim
def e11_decode_summary():
    """vLLM's periodic engine stats lines: generation throughput by running level, KV usage, prefix-cache hit rate."""
    if E11_SERVER is None:
        return None
    rows = [tuple(float(x) for x in m.groups()) for m in _E12V_STAT_RE.finditer(E11_SERVER.log_text())]
    med = lambda v: round(_statistics.median(v), 1) if v else None
    acc = re.findall(r"Mean acceptance length: ([\d.]+)", E11_SERVER.log_text())
    out = {"n_lines": len(rows),
           "running_hist": {str(int(k)): sum(1 for r in rows if r[2] == k) for k in sorted({r[2] for r in rows})},
           "gen_tok_s_median_at_concurrency": med([r[1] for r in rows if r[2] >= E12_VLLM_CONCURRENCY]),
           "gen_tok_s_median_all_busy": med([r[1] for r in rows if r[2] > 0]),
           "waiting_max": max((r[3] for r in rows), default=None), "kv_usage_pct_max": max((r[4] for r in rows), default=None),
           "prefix_cache_hit_pct_last": rows[-1][5] if rows else None,
           "spec_mean_acceptance_median": med([float(a) for a in acc]),
           "preemption_lines": sum(1 for l in E11_SERVER.log_text().splitlines() if "preempt" in l.lower())}
    try:
        write_json_atomic(OUT / "sglang_decode_summary.json", out)   # bundled under this name by CELL 7
    except Exception:
        pass
    return out


# ---- E12-V CELL 5, verbatim
def e11_kill_server(wait=True):
    if E11_SERVER is not None:
        E11_SERVER.kill(wait=wait)


# ---- E18 R2 CELL 5, verbatim
def e18_install():
    """The VMS installer: the wheelhouse's install_wheelhouse.py builds /tmp/vllm-venv (--no-deps, '+' restored)."""
    if E18_FAKE:
        return {"fake": True, "launch": [sys.executable, E18_FAKE], "env": dict(os.environ)}
    inst = next(iter(sorted(E18_VLLM_WHEELHOUSE.rglob("install_wheelhouse.py"))), None)
    if inst is None:
        raise FileNotFoundError(f"install_wheelhouse.py not under {E18_VLLM_WHEELHOUSE}: attach the vLLM 0.32.0 wheelhouse")
    t0 = time.perf_counter()
    r = subprocess.run([sys.executable, str(inst), "/tmp/vllm-venv"], capture_output=True, text=True, timeout=2400)
    if r.returncode != 0:
        raise RuntimeError("vLLM 0.32 wheelhouse install failed: " + (r.stdout + r.stderr)[-3000:])
    kv = dict(l.split("=", 1) for l in r.stdout.splitlines() if l.startswith(("VENV_PYTHON=", "CUDA_HOME=")))
    env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "CUDA_PATH")}
    env.update(CUDA_HOME=kv["CUDA_HOME"], PATH=f"/tmp/vllm-venv/bin:{kv['CUDA_HOME']}/bin:{env.get('PATH', '')}",
               HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", VLLM_NO_USAGE_STATS="1", VLLM_LOGGING_LEVEL="INFO",
               VLLM_CACHE_ROOT="/tmp/vllm-cache", VLLM_ALLOW_LONG_MAX_MODEL_LEN="1")
    return {"sec": round(time.perf_counter() - t0, 1), "venv_python": kv["VENV_PYTHON"],
            "launch": ["/tmp/vllm-venv/bin/vllm", "serve"], "env": env}


# ---- E18 R2 CELL 5, verbatim
class E18Server:
    def __init__(self, name, cmd, env):
        self.name, self.cmd, self.env, self.proc = name, list(cmd), dict(env), None
        self.log_path = OUT / f"vllm_server_{name}.log"

    def start(self):
        self.log_f = open(self.log_path, "w", encoding="utf-8")
        self.t0 = time.perf_counter()
        self.proc = subprocess.Popen(self.cmd, env=self.env, stdout=self.log_f, stderr=subprocess.STDOUT,
                                     start_new_session=True, cwd=None if E18_FAKE else "/tmp")

    def alive(self):
        return self.proc is not None and self.proc.poll() is None

    def log_text(self):
        try:
            return self.log_path.read_text(encoding="utf-8", errors="replace")
        except Exception:
            return ""

    def wait_ready(self, timeout, silence=E18_SILENCE_SEC):
        """rev 2: ready = /health answers AND this server's own "GPU KV cache size" line is in its log (B-L8); a launch
        whose log stops growing for `silence` seconds is a stall (B-M3)."""
        last, last_t = -1, time.perf_counter()
        while time.perf_counter() - self.t0 < timeout:
            if not self.alive():
                return False, f"exited rc={self.proc.returncode}"
            try:
                _urlreq.urlopen(E18_URL + "/health", timeout=10).read()
                if "GPU KV cache size" in self.log_text():
                    return True, f"ready in {time.perf_counter() - self.t0:.0f}s"
            except Exception:
                pass
            sz = self.log_path.stat().st_size if self.log_path.exists() else 0
            if sz != last:
                last, last_t = sz, time.perf_counter()
            elif time.perf_counter() - last_t > silence:
                return False, f"stall ({silence:.0f} s of log silence)"
            time.sleep(1 if E18_FAKE else 5)
        return False, "timeout"

    def kill(self, fast=False):
        """rev 2 (B-M2): SIGTERM to the process group, a grace for the API server, then SIGKILL to the group ALWAYS
        (engine core / workers stuck in CUDA or NCCL can outlive the API server)."""
        if self.proc is None:
            return
        grp = hasattr(os, "killpg")
        try:
            os.killpg(self.proc.pid, _signal.SIGTERM) if grp else self.proc.terminate()
        except Exception:
            pass
        try:
            self.proc.wait(timeout=0.1 if fast else 30)
        except Exception:
            pass
        try:
            os.killpg(self.proc.pid, getattr(_signal, "SIGKILL", _signal.SIGTERM)) if grp else self.proc.kill()
        except Exception:
            pass
        try:
            self.proc.wait(timeout=30)
        except Exception:
            pass
        try:
            self.log_f.close()
        except Exception:
            pass


# ---- E18 R2 CELL 5, verbatim
def _e18_kill_gpu_pids():
    """rev 2 (B-M2): kill every process still holding a GPU compute context (never this kernel)."""
    if E18_FAKE:
        return []
    killed = []
    try:
        out = subprocess.run(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"], capture_output=True,
                             text=True, timeout=20).stdout
        for tok in out.split():
            if tok.isdigit() and int(tok) != os.getpid():
                try:
                    os.kill(int(tok), getattr(_signal, "SIGKILL", _signal.SIGTERM)); killed.append(int(tok))
                except Exception:
                    pass
    except Exception:
        pass
    return killed


# ---- E18 R2 CELL 5 (= S1-60K CELL 5), verbatim
def _wait_gpu_release(max_sec=float(os.environ.get("Q27X_GPU_WAIT_MAX", 180))):
    """S1's cell 5 function, verbatim."""
    w0 = time.perf_counter(); used = []
    while time.perf_counter() - w0 < max_sec:
        try:
            used = [int(x) for x in subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                                                   capture_output=True, text=True, timeout=20).stdout.split()]
        except Exception:
            used = []
        if used and max(used) < 1024:
            return {"released": True, "sec": round(time.perf_counter() - w0, 1), "last_used_mb": used}
        time.sleep(5)
    return {"released": False, "sec": round(time.perf_counter() - w0, 1), "last_used_mb": used}


# ---- E21: the new code of the engine cell (the donor functions above are inserted verbatim by build_e21.py).
# rev 2 (2026-10-28): the fixes of the 10 validation reports (Experiment Review/VAL_*.md; list in the README "rev 2").

# VAL_api_D2 F2 / VAL_engine_A1 L2: vLLM 0.19 inserts ", Preemptions: N" and 0.32 also ", Deferred: N reqs" / "KV fetch"
# and drops the prefix-cache field until the first prefix query; the E12-V regex (above) missed exactly those lines.
# Replaces the E12-V _E12V_STAT_RE; groups: 1 prompt tok/s, 2 gen tok/s, 3 running, 4 waiting, 5 preemptions (or None),
# 6 KV %, 7 prefix-cache hit % (or None).
_E12V_STAT_RE = re.compile(r"Avg prompt throughput: ([\d.]+) tokens/s, Avg generation throughput: ([\d.]+) tokens/s, "
                           r"Running: (\d+) reqs, Waiting: (\d+) reqs(?:, Deferred: \d+ reqs)?(?:, KV fetch: [^%]*?completed waiting)?"
                           r"(?:, Preemptions: (\d+))?, GPU KV cache usage: ([\d.]+)%(?:, Prefix cache hit rate: ([\d.]+)%)?")


def e21_stat_rows(text):
    """vLLM's 10-s stats lines as dicts (both versions, every optional field)."""
    out = []
    for m in _E12V_STAT_RE.finditer(text):
        g = m.groups()
        out.append({"prompt_tps": float(g[0]), "gen_tps": float(g[1]), "running": int(g[2]), "waiting": int(g[3]),
                    "preempt": int(g[4]) if g[4] is not None else 0, "kv_pct": float(g[5]),
                    "prefix_pct": float(g[6]) if g[6] is not None else None})
    return out


def e11_decode_summary():
    """E12-V's stats-line summary on the rev 2 regex (preemption / deferred lines included) + the counter snapshot."""
    if E11_SERVER is None:
        return None
    txt = E11_SERVER.log_text()
    rows = e21_stat_rows(txt)
    med = lambda v: round(_statistics.median(v), 1) if v else None
    acc = re.findall(r"Mean acceptance length: ([\d.]+)", txt)
    pre = [r["prefix_pct"] for r in rows if r["prefix_pct"] is not None]
    out = {"n_lines": len(rows),
           "running_hist": {str(k): sum(1 for r in rows if r["running"] == k) for k in sorted({r["running"] for r in rows})},
           "gen_tok_s_median_at_concurrency": med([r["gen_tps"] for r in rows if r["running"] >= E12_VLLM_CONCURRENCY]),
           "gen_tok_s_median_all_busy": med([r["gen_tps"] for r in rows if r["running"] > 0]),
           "waiting_max": max((r["waiting"] for r in rows), default=None),
           "kv_usage_pct_max": max((r["kv_pct"] for r in rows), default=None),
           "prefix_cache_hit_pct_last_rolling": pre[-1] if pre else None,
           "preemptions_in_stats_lines": sum(r["preempt"] for r in rows),
           "spec_mean_acceptance_median": med([float(a) for a in acc]),
           "accept_len_median": med([float(a) for a in acc]),      # the key CELL 7 reads
           "counters": ENGINE_INFO.get("final_counters") or e21_counters()}
    try:
        write_json_atomic(OUT / "sglang_decode_summary.json", out)   # bundled under this name by CELL 7
    except Exception:
        pass
    return out


_E21_COUNTERS = ("vllm:num_preemptions_total", "vllm:prefix_cache_queries_total", "vllm:prefix_cache_hits_total",
                 "vllm:generation_tokens_total", "vllm:prompt_tokens_total", "vllm:request_success_total",
                 "vllm:spec_decode_num_accepted_tokens_total", "vllm:spec_decode_num_drafts_total")


def e21_counters():
    """VAL_engine_A2 M3 / VAL_api_D2 F6-F7: the server's own counters (preemptions, whole-run prefix hits, tokens), summed
    over label rows. Taken before every kill (a dead server loses them)."""
    try:
        rows = e21_prom()
    except Exception as e:
        return {"error": repr(e)[:200]}
    out = {}
    for r in rows:
        if r.name in _E21_COUNTERS:
            out[r.name] = out.get(r.name, 0.0) + float(r.value)
    q, h = out.get("vllm:prefix_cache_queries_total"), out.get("vllm:prefix_cache_hits_total")
    out["prefix_hit_pct_whole_run"] = round(100 * h / q, 2) if q else None
    return out


def e21_snapshot_counters():
    if E11_SERVER is not None and E11_SERVER.alive():
        c = e21_counters()
        if "error" not in c:
            ENGINE_INFO["final_counters"] = c


def e11_kill_server(wait=True):
    """E21 harness finding: the E12-V e11_kill_server calls kill(wait=...), but the server here is an E18Server whose
    kill() takes `fast` (CELL 7's final call raised TypeError, silently). rev 2: the counters are snapshot first."""
    e21_snapshot_counters()
    if E11_SERVER is not None:
        E11_SERVER.kill(fast=not wait)


def e21_duck_serving_flags(served, port):
    """E12-V CELL 5 (the vLLM SMOKE1 run 2 server): Duck's start_vllm_server flags, verbatim, plus the prompt-tokens
    details (cached_tokens in usage) and the 135,168-token context (Duck window 128 Ki + margin). rev 2: no uvicorn
    access log (VAL_api_D2 F4: our own /metrics polls kept the log growing, so the watchdog never reached /health)."""
    return ["--served-model-name", served, "--host", "127.0.0.1", "--port", str(port),
            "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder", "--generation-config", "vllm",
            "--enable-prefix-caching", "--default-chat-template-kwargs", json.dumps({"preserve_thinking": True}),
            "--reasoning-parser", "qwen3", "--max-model-len", str(E21_VLLM_MAX_MODEL_LEN), "--enable-prompt-tokens-details",
            "--disable-uvicorn-access-log"]


def e21_model_path(weights):
    return E21_STAGED.get(weights) or (SGL_MODEL_PATH if weights == "int4" else MODEL_PATH)


def e21_flags_v019(cfg, port, backend):
    """vLLM 0.19 = the E12-V CELL 5 command: Duck's flags + the S1-60K engine (TP4, fp8_e4m3 KV, MTP-3, FlashInfer,
    4096 batched tokens, 0.90 GPU memory, language-model-only, custom all-reduce off, seed). E21 changes: the model path
    (INT4 for NB1), --max-num-seqs per half, --mamba-ssm-cache-dtype for NB2 (vLLM 0.19 CacheConfig.mamba_ssm_cache_dtype)."""
    f = ["--model", str(e21_model_path(cfg["weights"]))] + e21_duck_serving_flags(E21_SERVED[cfg["weights"]], port)
    f += ["--tensor-parallel-size", "4", "--kv-cache-dtype", KV_CACHE_DTYPE, "--speculative-config", json.dumps(SPECULATIVE_CONFIG),
          "--attention-backend", backend, "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),
          "--max-num-seqs", str(cfg["seqs"]), "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
          "--language-model-only", "--disable-custom-all-reduce", "--seed", str(SEED)]
    if cfg.get("ssm"):
        f += ["--mamba-ssm-cache-dtype", cfg["ssm"]]
    return f


def e21_flags_v032(cfg, port, backend):
    """vLLM 0.32 = the VMS CELL 6 vms_flags() arm (fp8_mtp3_ssm_fp16 for NB2, int4_mtp3_s20 for NB1, the reference
    fp8_mtp3 as the fallback): TP4, fp8_e4m3 KV, max-num-seqs, 4096 batched, 0.90, custom all-reduce off, seed,
    FlashInfer, --trust-remote-code, prefix caching, MTP-3 (+ --mamba-ssm-cache-dtype); E21: Duck's serving flags and the
    135,168 context (VMS: 106,496). rev 2 (VAL_fidelity_E2 M1, VAL_engine_A1 H2): + --language-model-only, as half A and
    as the E18 R2 rung s1 that became ready on 0.32 (KV 1,110,415 with it vs 1,006,517 without, same FP8 / 20 seqs)."""
    f = [str(e21_model_path(cfg["weights"])), "--tensor-parallel-size", "4", "--kv-cache-dtype", KV_CACHE_DTYPE,
         "--max-num-seqs", str(cfg["seqs"]), "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),
         "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION), "--disable-custom-all-reduce", "--seed", str(SEED),
         "--attention-backend", backend, "--trust-remote-code",
         "--speculative-config", json.dumps({"method": "mtp", "num_speculative_tokens": 3}), "--language-model-only"]
    f += e21_duck_serving_flags(E21_SERVED[cfg["weights"]], port)
    if cfg.get("ssm"):
        f += ["--mamba-ssm-cache-dtype", cfg["ssm"]]
    return f


def e21_prom():
    """The server's /metrics as rows with .name / .labels / .value (the e11_prom() shape CELL 6 reads)."""
    from types import SimpleNamespace
    txt = _urlreq.urlopen(SGL_URL + "/metrics", timeout=10).read().decode("utf-8", "replace")
    rows = []
    for line in txt.splitlines():
        m = _E12V_PROM_RE.match(line) if line and not line.startswith("#") else None
        if not m:
            continue
        try:
            rows.append(SimpleNamespace(name=m.group(1), labels=m.group(2) or "", value=float(m.group(3))))
        except ValueError:
            pass
    return rows


E21_SIZING_TOKENS = 57000        # R3K2 per-attempt peak context p50 (p90 65k, max 79k, mean ~50k)
E21_MIN_STREAMS = 8              # = R3K2 SGL_MIN_RUNNING: fewer streams -> this config is refused (the fallback runs)


def e21_capacity(log, seqs):
    """Pre-run capacity check, fail-closed (rev 2: VAL_engine_A1 H1, VAL_engine_A2 M1, VAL_api_D2 F3, VAL_fidelity_E1 M3).
    vLLM's "Maximum concurrency for M tokens per request: X" = num_blocks / (cdiv(M, B) + F): each request holds
    cdiv(L, B) attention blocks plus F fixed Mamba blocks (align mode: 3 groups x (2 + spec tokens) = 15 with MTP-3,
    kv_cache_interface.py; checked on the E12-V log: 8.46 x (85 + 15) = 846 blocks, and 846 // 4 x 1,600 = 337,600 = its
    'GPU KV cache size' line). Streams = min(seqs, num_blocks // (cdiv(57k, B) + F)); a missing line, or < 8 streams,
    refuses the config."""
    kv = re.findall(r"GPU KV cache size: ([\d,]+) tokens", log)
    mc = re.findall(r"Maximum concurrency for ([\d,]+) tokens per request: ([\d.]+)x", log)
    bs = re.findall(r"Setting attention block size to (\d+) tokens", log)
    if not mc or not bs:
        raise RuntimeError(f"capacity: vLLM log lines missing (max concurrency {bool(mc)}, block size {bool(bs)})")
    B, M, X = int(bs[-1]), int(mc[-1][0].replace(",", "")), float(mc[-1][1])
    F = 3 * (2 + int((SPECULATIVE_CONFIG or {}).get("num_speculative_tokens", 0)))
    cd = lambda a: -(-int(a) // B)
    nb = int(round(X * (cd(M) + F)))
    need = lambda L, n: n * (cd(L) + F)
    out = {"kv_tokens_logged": int(kv[-1].replace(",", "")) if kv else None, "max_concurrency_logged": {"tokens": M, "x": X},
           "block_size": B, "fixed_blocks_per_request": F, "num_blocks": nb,
           "effective_kv_tokens": nb * B,
           "util_mean_50k": round(need(50000, seqs) / nb, 3), "util_p50_peak_57k": round(need(57000, seqs) / nb, 3),
           "util_p90_peak_65k": round(need(65000, seqs) / nb, 3), "util_max_79k": round(need(79000, seqs) / nb, 3),
           "streams_fit": {str(L): nb // (cd(L) + F) for L in (50000, 57000, 65000, 79000)}}
    out["streams"] = min(int(seqs), nb // (cd(E21_SIZING_TOKENS) + F))
    out["fits_p50_peak"] = out["streams"] >= int(seqs)
    if out["streams"] < E21_MIN_STREAMS:
        raise RuntimeError(f"capacity: only {out['streams']} streams fit ({json.dumps(out)})")
    return out


# rev 2d (GPT audit 3): Duck's own tool definition = tool_agent.ToolAgent._tools() with arc2's description
# (arc2_duck.py 1572), so the probe exercises the exact schema the Duck sends
E21_PROBE_TOOLS = [{"type": "function", "function": {"name": "python", "description": E12A.ARC2_TOOL_DESCRIPTION,
                    "parameters": {"type": "object", "properties": {"code": {"type": "string", "description": (
                        "Python code to run. The snippet is ephemeral and is not saved across tool calls.")}},
                        "required": ["code"]}}}]


def e21_contract_probe(served):
    """rev 2 (VAL_api_D1 F2, VAL_engine_A1 M1): the Duck contract on this server, before a 5.5-h half. Turn 1: tools +
    thinking (reasoning_effort low) asking for one python call. Hard failures: an HTTP error, tool-call arguments that are
    not a JSON object with a string 'code', think / tool-call markup leaking into content, a degenerate reply. If a call
    came back, turn 2 sends it as history (reasoning under 'reasoning', tool result) and the prompt must grow by about
    turn 1's prompt + completion (a dropped reasoning shows up as a shortfall, the E12-V check). No call within the
    budget (thinking ran long) is inconclusive, not a failure."""
    out = {"ok": False}
    kw = {"enable_thinking": True, "preserve_thinking": True, "reasoning_effort": "low"}
    msgs = [{"role": "user", "content": "Use the python tool exactly once to run print(6 * 7). Keep your thinking very short."}]
    r1 = _e12v_request_json(SGL_URL + "/v1/chat/completions", payload={
        "model": served, "messages": msgs, "tools": E21_PROBE_TOOLS, "tool_choice": "auto", "temperature": 0.0,
        "max_tokens": 3072, "chat_template_kwargs": kw}, timeout=600)
    m, u = r1["choices"][0]["message"], r1.get("usage") or {}
    reas, cont, calls = m.get("reasoning") or m.get("reasoning_content") or "", m.get("content") or "", m.get("tool_calls") or []
    out.update(finish=r1["choices"][0].get("finish_reason"), reasoning_chars=len(reas), content=cont[:200],
               tool_calls=len(calls), prompt=u.get("prompt_tokens"), completion=u.get("completion_tokens"))
    bad = [k for k in ("<think>", "</think>", "<tool_call>", "<function=") if k in cont]
    if bad:
        raise RuntimeError(f"contract: markup in content {bad}: {cont[:300]!r}")
    txt = (reas + cont).strip()
    if len(txt) >= 40 and len(set(txt.replace(" ", "").replace("\n", ""))) <= 2:
        raise RuntimeError(f"contract: degenerate reply {txt[:80]!r}")
    for c in calls:
        a = json.loads(c["function"]["arguments"])
        if not (isinstance(a, dict) and isinstance(a.get("code"), str) and a["code"].strip()):
            raise RuntimeError(f"contract: tool-call arguments {c['function']['arguments'][:200]!r}")
    if not calls:
        out.update(turn2="inconclusive: no tool call in turn 1")
        e21_soft(out, "cache", lambda: e21_cache_probe(served))
        out["ok"] = True
        return out
    hist = msgs + [{"role": "assistant", "reasoning": reas, "tool_calls": calls},
                   {"role": "tool", "tool_call_id": calls[0].get("id", "call_0"), "content": "42"}]
    r2 = _e12v_request_json(SGL_URL + "/v1/chat/completions", payload={
        "model": served, "messages": hist, "tools": E21_PROBE_TOOLS, "tool_choice": "auto", "temperature": 0.0,
        "max_tokens": 512, "chat_template_kwargs": kw}, timeout=600)
    p2 = (r2.get("usage") or {}).get("prompt_tokens")
    short = (int(out["prompt"]) + int(out["completion"])) - int(p2)
    out.update(turn2_prompt=p2, turn2_shortfall=short)
    if reas and short > max(128, len(reas) // 6):
        e21_soft(out, "history", lambda: (_ for _ in ()).throw(
            RuntimeError(f"contract: history prompt {p2} short by {short} tokens (reasoning dropped?)")))
    e21_soft(out, "cache", lambda: e21_cache_probe(served))
    out["ok"] = True
    return out


def e21_soft(out, key, fn):
    """rev 2f: a check that is fatal in commit / smoke (falls back) but only a recorded warning in the hidden rerun."""
    try:
        out[key] = fn()
    except Exception as e:
        if not RERUN:
            raise
        out.setdefault("warnings", []).append(f"{key}: {e}"[:600])
        print("E21 WARNING (rerun, not fatal):", key, str(e)[:300], flush=True)


def e21_cache_probe(served):
    """rev 2b (VAL_api_D2 F5): the align-mode prefix-cache path that serves 67-94 % of every later Duck prompt. A shared
    ~5,000-token prefix with a marker per request: the cold request, then the same request again (it must hit the cache,
    >= 1 block) at temperature 0, thinking off; both replies must contain the request's own marker and be equal."""
    import random as _rnd
    _r7 = _rnd.Random(7)                       # rev 2f (VAL2_config_smoke F6): one RNG (the old code repeated one digit)
    rows = " ".join(str(_r7.randrange(10)) for _ in range(6000))
    res = []
    for i in range(2):
        mark = f"QZ{_rnd.randrange(10 ** 6, 10 ** 7)}K{i}"
        msgs = [{"role": "user", "content": "Grid digits for context: " + rows + f"\n\nReply with exactly this code: {mark}"}]
        body = {"model": served, "messages": msgs, "temperature": 0.0, "max_tokens": 96,
                "chat_template_kwargs": {"enable_thinking": False}}
        r1 = _e12v_request_json(SGL_URL + "/v1/chat/completions", payload=body, timeout=300)
        r2 = _e12v_request_json(SGL_URL + "/v1/chat/completions", payload=body, timeout=300)
        g1 = r1["choices"][0]["message"].get("content") or ""
        g2 = r2["choices"][0]["message"].get("content") or ""
        u2 = r2.get("usage") or {}
        c2 = ((u2.get("prompt_tokens_details") or {}).get("cached_tokens") or 0)
        res.append({"prompt": u2.get("prompt_tokens"), "cached_repeat": c2, "own_marker": mark in g1 and mark in g2,
                    "equal": g1 == g2})
    if not all(x["own_marker"] and x["equal"] and x["cached_repeat"] >= 1024 for x in res):
        raise RuntimeError(f"cache probe failed: {json.dumps(res)}")
    return res


def e21_v032_install_bg():
    """rev 2b (VAL_timing_C1 M3, VAL_fidelity_E2 LOW-7): build the vLLM 0.32 venv on CPU while half A solves, so half B's
    setup is only its launch. The result (or the error) is what e21_launch would have produced."""
    def _run():
        try:
            E21_STATE["v032_install"] = e18_install()
        except Exception as e:
            E21_STATE["v032_install"] = {"error": f"{type(e).__name__}: {e}"[:600]}
    if E21_STATE.get("v032_install") is None and E21_STATE.get("v032_thread") is None:
        E21_STATE["v032_thread"] = threading.Thread(target=_run, daemon=True, name="e21_v032_install")
        E21_STATE["v032_thread"].start()


def e21_launch(H, cfg, tag, timeout=None):
    """One server launch for half H with config cfg; returns (server, info). Raises on failure (its own server killed).
    rev 2: the strict smoke + the contract probe + the fail-closed capacity check are part of 'ready'; a relaunch passes
    its own timeout; the failure path never touches a server or the GPUs of another half (VAL_state_B2 L1)."""
    global E18_URL, E11_SERVER
    port = E21_PORTS[H]
    E18_URL = f"http://127.0.0.1:{port}"
    backend = "FLASHINFER" if CUDA_LINK.get("probe") == "PASS" else "TRITON_ATTN"
    info = {"tag": tag, "cfg": dict(cfg), "attention_backend": backend, "port": port}
    if cfg["vllm"] == "0.19.0":
        if not E11_HARNESS:
            assert pkg_version("vllm") == "0.19.0", pkg_version("vllm")
        launch = [sys.executable, "-m", "vllm.entrypoints.openai.api_server"] if not E21_FAKE else [sys.executable, E21_FAKE]
        env = os.environ.copy()
        env.update({"USE_TF": "0", "TRANSFORMERS_NO_TF": "1", "TRANSFORMERS_NO_TORCHVISION": "1", "VLLM_NO_USAGE_STATS": "1",
                    "VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS": "1", "PYTHONUNBUFFERED": "1"})   # E12-V: Duck's vllm_env + S1
        flags = e21_flags_v019(cfg, port, backend)
    else:
        _th = E21_STATE.get("v032_thread")
        if _th is not None and _th.is_alive():
            _th.join(timeout=2400)
        if isinstance(E21_STATE.get("v032_install"), dict) and E21_STATE["v032_install"].get("error"):
            raise RuntimeError("vLLM 0.32 install failed earlier: " + E21_STATE["v032_install"]["error"], info)   # A1 M4
        if E21_STATE.get("v032_install") is None:
            try:
                E21_STATE["v032_install"] = e18_install()        # the VMS installer (E18 R2), once per notebook
            except Exception as e:
                E21_STATE["v032_install"] = {"error": f"{type(e).__name__}: {e}"[:600]}
                raise RuntimeError("vLLM 0.32 install failed: " + E21_STATE["v032_install"]["error"], info)
        inst = E21_STATE["v032_install"]
        launch, env = list(inst["launch"]), dict(inst["env"])
        env["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = "1"
        env.update(TRITON_CACHE_DIR="/tmp/e21-v032-triton", FLASHINFER_WORKSPACE_BASE="/tmp/e21-v032-flashinfer")
        flags = e21_flags_v032(cfg, port, backend)
    srv = E18Server(f"half{H}_{tag}", launch + flags, env)
    info.update(flags=flags, log=str(srv.log_path))
    if timeout is None:
        _cap = E21_LAUNCH_TIMEOUT_SEC if cfg["vllm"] == "0.19.0" else E21_LAUNCH_TIMEOUT_V032_SEC
        timeout = max(60.0, min(_cap, E21_HALF_END_PC - time.perf_counter() - 30 * 60))
    try:
        srv.start()
        E11_SERVER = srv
        print(f"E21 half {H} launch {tag}: vllm {cfg['vllm']}", " ".join(str(x) for x in launch[-2:]), " ".join(flags),
              "| ready timeout", round(timeout), flush=True)
        ok, why = srv.wait_ready(timeout)
        info.update(ready=ok, why=why, ready_sec=round(time.perf_counter() - srv.t0, 1))
        if not ok:
            raise RuntimeError(f"half {H} {tag}: not ready ({why})")
        models = _e12v_request_json(SGL_URL + "/v1/models", timeout=30)
        info["models"] = [m.get("id") for m in models.get("data", [])]
        info["capacity"] = e21_capacity(srv.log_text(), cfg["seqs"])
        # rev 2c (GPT audit 3): the engine really runs text-only (the non-default-args line of both versions)
        info["lmo_in_log"] = "'language_model_only': True" in srv.log_text()
        # the E12-V smoke, strict (rev 2: VAL_engine_A2 H1, VAL_timing_C2 M4: "!!!!" output must not pass)
        r = _e12v_request_json(SGL_URL + "/v1/chat/completions", payload={
            "model": E21_SERVED[cfg["weights"]], "messages": [{"role": "user", "content": "Answer in one short sentence: what is 2 + 2?"}],
            "temperature": 0.0, "max_tokens": 96, "chat_template_kwargs": {"enable_thinking": False}}, timeout=300)
        gen = (r["choices"][0]["message"].get("content") or "").strip()
        usage = r.get("usage") or {}
        info["smoke"] = {"generated": gen[:200], "usage": usage,
                         "cache_details_present": isinstance(usage.get("prompt_tokens_details"), dict),
                         "ok": ("4" in gen or "four" in gen.lower()) and not (len(gen) >= 8 and len(set(gen.replace(" ", ""))) <= 2)   # rev 2g (VAL2_engine F10)
                         and isinstance(usage.get("prompt_tokens"), int) and usage["prompt_tokens"] > 0
                         and isinstance(usage.get("completion_tokens"), int) and usage["completion_tokens"] > 0}
        if not info["smoke"]["ok"]:
            raise RuntimeError(f"half {H} {tag}: smoke failed {json.dumps(info['smoke'])[:400]}")
        info["contract"] = e21_contract_probe(E21_SERVED[cfg["weights"]])
        print(f"E21 half {H} {tag}: {why} | capacity", json.dumps(info["capacity"]), "| smoke", json.dumps(info["smoke"]),
              "| contract", json.dumps(info["contract"]), flush=True)
    except Exception as e:
        info["log_tail"] = srv.log_text()[-3000:]
        srv.kill()
        if E11_SERVER is srv:
            E11_SERVER = None
        if E21_HALF == H:            # a straggler relaunch of a finished half never kills the next half's server
            info["gpu_pids_killed"] = _e18_kill_gpu_pids()
            info["gpu_release"] = _wait_gpu_release()
        raise RuntimeError(f"{type(e).__name__}: {e}"[:900], info)
    return srv, info


E21_RELAUNCH_MIN_SEC = 12 * 60   # vLLM needs ~7-10 min to be ready: a relaunch with less is pointless


def e21_relaunch():
    """The CELL 6 watchdog's relaunch: the same half, the config that was ready. rev 2 (VAL_state_B1 M1, VAL_engine_A1 M6,
    VAL_fidelity_E1 LOW-3): the ready timeout follows the solve end (min(30 min, end - now - 5 min)), as the E15B donor's
    min(20 min, left - 10 min) did; none with less than 12 min."""
    H, cfg = E21_HALF, ENGINE_INFO.get("cfg")
    if len(ENGINE_INFO.get("relaunches") or []) >= 1:          # the same config already crashed once: try the other one
        _alt = [c for _t, c in e21_cfgs(ENGINE_INFO.get("spec") or E21_HALF_SPECS[E21_VARIANT][H]) if c != cfg
                and c.get("weights") == (cfg or {}).get("weights")]   # rev 2g (VAL2_engine F1): same served name
        if _alt:
            cfg = _alt[0]
            ENGINE_INFO["relaunch_cfg_switched"] = dict(cfg)
    t = min(E21_LAUNCH_TIMEOUT_SEC, globals().get("E15B_SOLVE_END_PC", E21_HALF_END_PC) - time.perf_counter() - 5 * 60)
    if t < E21_RELAUNCH_MIN_SEC:
        ENGINE_INFO.setdefault("relaunches", []).append({"skipped": f"only {round(t)} s for a relaunch"})
        return False, "too little time for a relaunch"
    e21_snapshot_counters()
    try:
        if E11_SERVER is not None:
            E11_SERVER.kill()
    except Exception:
        pass
    _e18_kill_gpu_pids()
    _wait_gpu_release()
    if E21_HALF != H or E21_STATE.get("ending") == H:      # rev 2h (GPT v3 audit 2): the half ended meanwhile
        ENGINE_INFO.setdefault("relaunches", []).append({"skipped": "the half ended during the relaunch"})
        return False, "the half ended during the relaunch"
    try:
        srv, info = e21_launch(H, cfg, f"relaunch{len(ENGINE_INFO.setdefault('relaunches', []))}", timeout=t)
        ENGINE_INFO["relaunches"].append({k: info.get(k) for k in ("tag", "ready", "why", "ready_sec", "capacity", "cfg")})
        ENGINE_INFO["cfg"] = cfg
        return True, info.get("why")
    except Exception as e:
        ENGINE_INFO.setdefault("relaunches", []).append({"error": repr(e)[:600]})
        return False, repr(e)[:300]


def e21_cfgs(spec):
    """[("primary", cfg), ("fallback", cfg)] of a half spec (the fallback overrides some keys of the primary)."""
    base = {k: v for k, v in spec.items() if k != "fallback"}
    return [("primary", dict(base)), ("fallback", dict(base, **spec["fallback"]))]


def e21_gpus_free(tag):
    """VAL_engine_A1 M3 / VAL_state_B2 L6: never launch on GPUs a previous server still holds (the E18 donor stopped)."""
    if E21_FAKE and os.environ.get("E21_FAKE_GPU_HELD") == "1":            # harness only: GPUs never released
        ENGINE_INFO.setdefault("gpu_checks", []).append({"released": False, "tag": tag, "fake_held": True})
        return False
    rel = _wait_gpu_release()
    if not rel.get("released") and not E21_FAKE:
        _e18_kill_gpu_pids()
        rel = _wait_gpu_release()
    rel["tag"] = tag
    ENGINE_INFO.setdefault("gpu_checks", []).append(rel)
    return bool(rel.get("released")) or bool(E21_FAKE)


def e21_begin_half(H):
    """Half H: its own output folder, a fresh Duck scheduler state, its vLLM server (primary config, else the fallback)."""
    global OUT, E21_HALF, E21_HALF_T0, E21_HALF_END_PC, SGL_URL, ENGINE_READY, CONCURRENCY, ENGINE_INFO, E12_STREAMS
    global E12_VLLM_CONCURRENCY, E15B_SOLVE_END_PC, engine, llm, E12_SOLVE_WINDOW_SEC, E12_RECORDS, E12_RUN
    # rev 2 (VAL_state_B2 LOW-2): the per-half state is reset first, so a later raise can never leave half A's record
    ENGINE_READY, CONCURRENCY, engine, llm = False, 0, None, None
    ENGINE_INFO = {"half": H, "variant": E21_VARIANT, "attempts": [], "backend": None, "kv_tokens": None,
                   "concurrency": None, "fatal": None, "load_sec": None, "cuda_link": CUDA_LINK}
    E12_RECORDS, E12_RUN = {}, {}               # VAL_state_B1 L3 (CELL 6 re-creates both first)
    E21_HALF, E21_HALF_T0 = H, time.perf_counter()
    OUT = BASE_OUT / f"half{H}"
    OUT.mkdir(parents=True, exist_ok=True)
    spec = json.loads(json.dumps(E21_HALF_SPECS[E21_VARIANT][H]))
    _a = (E21_STATE["halves"].get("A") or {}).get("engine") or {}
    if H != "A" and E21_B_FOLLOWS_A and _a.get("used") == "fallback":
        # user decision (2026-10-28, option ka): a half-A fallback is repeated in half B, so the two halves differ only in
        # the vLLM version
        spec = dict(spec, **spec["fallback"])
        ENGINE_INFO["spec_note"] = "half A ran its fallback config; half B uses the same config (E21_B_FOLLOWS_A)"
    ENGINE_INFO["spec"] = spec
    _stop = HARD_STOP - E12_STOP_BEFORE_HARD_SEC
    # half A of a two-half commit stops early enough for half B's setup + window; otherwise the notebook's own stop
    E21_HALF_END_PC = _stop if (H != "A" or len(E21_HALVES) == 1) else _stop - E21_HALF_SOLVE_SEC - E21_B_SETUP_RESERVE_SEC
    # rev 2 (VAL timing C1 M2 / C2 M1, state B1 M2, fidelity E1 M1 / E2, engine A1 M7, A2 M4): equal windows. Half B gets
    # half A's achieved window when that was shorter (A's setup ate into it), so 0.19 and 0.32 get the same seconds.
    if E12_SOLVE_WINDOW_SEC and H != "A":
        _wa = ((E21_STATE["halves"].get("A") or {}).get("e12_run") or {}).get("deadline_in_sec")
        if _wa and _a.get("used") and _wa < E21_HALF_SOLVE_SEC:
            E12_SOLVE_WINDOW_SEC = float(_wa)
            ENGINE_INFO["window_note"] = f"half B window = half A's achieved {round(_wa)} s"
    ENGINE_INFO.update(window_sec=E12_SOLVE_WINDOW_SEC, end_pc_in_sec=round(E21_HALF_END_PC - NOTEBOOK_T0),
                       begin_sec=round(E21_HALF_T0 - NOTEBOOK_T0))
    SGL_URL = f"http://127.0.0.1:{E21_PORTS[H]}"
    E15B_SOLVE_END_PC = float("inf")
    E12A.SERVER_DOWN.clear(); E12A.SERVER_WAITS.update(waits=0, sec=0.0); E12A.SERVER_LAST.clear(); E12A.SCHED_LAST.clear()
    # rev 2 (VAL_state_B1 H1, B2 HIGH-1, timing C2 HIGH-1, fidelity E1 HIGH-1 / E2 HIGH-1): Duck's priority gate is a
    # process-wide singleton built once with ARC3_MAX_ACTIVE_STREAMS and a min() clock; a fresh gate per half (half A's
    # stragglers keep their own reference to the old one)
    with _E12_TA._PRIORITY_GATE_LOCK:
        _E12_TA._PRIORITY_GATE = None
    _E12_TA._GATE_STATS.update(enqueued=0, blocked=0, blocked_ms=0, handover_ms=0)
    os.environ["ARC3_REASONING_HISTORY_KEY"] = "reasoning"     # vLLM reads only message["reasoning"] (see CELL 3)
    os.environ["LOCAL_ANALYZER_BASE_URL"] = os.environ["OPENAI_BASE_URL"] = SGL_URL + "/v1"
    E21_STATE.pop("probe_fail_since", None)      # rev 2f (VAL2_cells F3): the outage clock is per half
    E21_STATE["halves"][H] = {"engine": ENGINE_INFO}
    print(f"E21 half {H}: {json.dumps(spec)} | window {E12_SOLVE_WINDOW_SEC} | solve must end by {ENGINE_INFO['end_pc_in_sec']} s",
          flush=True)
    if not E10_SANDBOX_SELFTEST.get("ok", False):
        ENGINE_INFO["fatal"] = "sandbox self-test failed"
        return
    _t = time.perf_counter()
    for tag, cfg in e21_cfgs(spec):
        if not e21_gpus_free(f"before {tag}"):
            ENGINE_INFO["fatal"] = "GPU memory still held before the launch"
            break
        try:
            srv, info = e21_launch(H, cfg, tag)
            ENGINE_INFO["attempts"].append(dict(info))
            ENGINE_INFO.update(cfg=cfg, used=tag, backend=f"vllm-{cfg['vllm']}-{cfg['weights']}-mtp{(SPECULATIVE_CONFIG or {}).get('num_speculative_tokens', 0)}-s{cfg['seqs']}"
                               + (f"-ssm-{cfg['ssm']}" if cfg.get("ssm") else "") + f"-{info['attention_backend']}",
                               kv_tokens=info["capacity"].get("effective_kv_tokens"), capacity=info["capacity"],
                               max_model_len=E21_VLLM_MAX_MODEL_LEN)
            ENGINE_READY = True
            CONCURRENCY = int(info["capacity"]["streams"])
            break
        except Exception as e:
            _info = e.args[1] if len(e.args) > 1 and isinstance(e.args[1], dict) else {}
            ENGINE_INFO["attempts"].append(dict(_info, error=repr(e.args[0] if e.args else e)[:900]))
            print(f"E21 half {H} {tag} FAILED:", repr(e.args[0] if e.args else e)[:900], flush=True)
            if time.perf_counter() > E21_HALF_END_PC - 30 * 60:
                break
    ENGINE_INFO["load_sec"] = round(time.perf_counter() - _t, 1)
    ENGINE_INFO["concurrency"] = int(CONCURRENCY)
    E12_STREAMS = int(CONCURRENCY)
    E12_VLLM_CONCURRENCY = int(CONCURRENCY)      # read by e11_decode_summary (E12-V)
    if H == "A" and "B" in E21_HALVES and any(c.get("vllm") == "0.32.0" for c in E21_HALF_SPECS[E21_VARIANT].values()):
        e21_v032_install_bg()
    if not ENGINE_READY:
        ENGINE_INFO["fatal"] = ENGINE_INFO.get("fatal") or "no vLLM config became ready"
    print(f"E21 half {H} engine:", json.dumps(ENGINE_INFO, default=str)[:3000], flush=True)


def e21_scheduler_threads():
    """Duck scheduler threads still alive (solve_all_r6's worker / probe_loop closures)."""
    n = 0
    for th in threading.enumerate():
        q = getattr(getattr(th, "_target", None), "__qualname__", "") or ""
        if th.is_alive() and q.startswith("solve_all_r6.<locals>."):
            n += 1
    return n


def e21_end_half(H):
    """Stop half H's server and free the GPUs for the next half; keep its records for the E21 report. rev 2: the kill
    always runs (finally: VAL_engine_A1 M5); the last half kills fast and does not wait for the GPUs (nothing follows;
    VAL timing C1 M1 / C2 M2: the hidden rerun's tail); half A's watchdog thread is joined first (VAL_state_B2 L1)."""
    global E11_SERVER
    st = E21_STATE["halves"].setdefault(H, {})
    last = H == E21_HALVES[-1]
    E21_STATE["ending"] = H              # rev 2h: a watchdog relaunch in flight sees this and never launches
    try:
        for th in threading.enumerate():
            if th.name in ("e15b_watchdog", "e12_heartbeat") and th is not threading.current_thread():
                th.join(timeout=30)
                if th.is_alive() and th.name == "e15b_watchdog":
                    # rev 2h (GPT v3 audit 2): a relaunch is in flight; kill its server (half B has not begun, so
                    # every server / GPU process is this half's), then join again, bounded
                    st["watchdog_relaunch_cancelled"] = True
                    try:
                        if E11_SERVER is not None:
                            E11_SERVER.kill(fast=True)
                    except Exception:
                        pass
                    _e18_kill_gpu_pids()
                    th.join(timeout=180)
                st[th.name + "_alive_after_join"] = th.is_alive()
                if th.is_alive() and th.name == "e15b_watchdog" and not last:
                    E21_STATE["abort_next_half"] = f"half {H} watchdog still active after the handoff joins"
        _recs, _run = globals().get("E12_RECORDS") or {}, globals().get("E12_RUN") or {}
        if st.get("cell6_ran"):
            with (globals().get("_E12_LOCK") or threading.Lock()):
                st["records"] = {t: list(v) for t, v in _recs.items() if v}
            st["e12_run"] = dict(_run)
        _lr = int(globals().get("E21_LENGTH_RESTORED", {}).get("n", 0))
        st["length_restored"] = _lr - sum(int(v.get("length_restored") or 0) for k, v in E21_STATE["halves"].items() if k != H)
        st["gate"] = {"stats": dict(_E12_TA._GATE_STATS), "slots_env": os.environ.get("ARC3_MAX_ACTIVE_STREAMS"),
                      "free_now": getattr(_E12_TA._PRIORITY_GATE, "_free", None)}
        e21_snapshot_counters()
        st["engine"] = ENGINE_INFO
        st["decode_summary"] = e11_decode_summary() if E11_SERVER is not None else None
        _acc = (st["decode_summary"] or {}).get("accept_len_median")
        if _acc is not None and _acc < 1.5:          # an INT4 MTP-head mismatch shows up here first (VAL_engine_A1 M2)
            st["mtp_acceptance_warning"] = _acc
            print(f"E21 WARNING half {H}: MTP mean acceptance {_acc} < 1.5", flush=True)
    except Exception as e:
        st["end_error"] = repr(e)[:300]
    finally:
        try:
            if E11_SERVER is not None:
                E11_SERVER.kill(fast=last)
        except Exception:
            pass
        E11_SERVER = None
        if not last:
            st["gpu_pids_killed"] = _e18_kill_gpu_pids()
            st["gpu_release"] = _wait_gpu_release()
        if not last:     # rev 2d (GPT audit 2): a clean handoff, bounded; counted, never assumed
            _w0 = time.perf_counter()
            while e21_scheduler_threads() and time.perf_counter() - _w0 < 120:
                time.sleep(2)
            st["handoff_scheduler_threads_left"] = e21_scheduler_threads()
            st["handoff_wait_sec"] = round(time.perf_counter() - _w0, 1)
        st["end_sec"] = round(time.perf_counter() - NOTEBOOK_T0)
        print(f"E21 half {H} ended at {st['end_sec']} s | GPU release", json.dumps(st.get("gpu_release")),
              "| scheduler threads left", st.get("handoff_scheduler_threads_left"), flush=True)


BASE_OUT = OUT
E21_STATE = {"halves": {}, "v032_install": None}
E21_STAGED = {}
if E14_STAGING is not None:          # CELL 1b started the copy of this variant's weights (fail open: original paths)
    E14_STATE.enter("STAGING_JOIN")
    E14_STAGING.join(timeout=max(0.0, min(SGL_GIVE_UP_AT - 20 * 60, NOTEBOOK_T0 + 55 * 60) - time.perf_counter()))
    if E14_STAGING.result is None:
        E14_RESULTS["staging"] = {"still_running": True, "note": "launch from the original paths"}
    else:
        _paths, E14_RESULTS["staging"] = E14_STAGING.result
        for _w, _p in (("int4", SGL_MODEL_PATH), ("fp8", MODEL_PATH)):
            if _paths.get(str(_p)) and _paths[str(_p)] != str(_p):
                E21_STAGED[_w] = Path(_paths[str(_p)])
    E14_RESULTS["launch_paths"] = {k: str(v) for k, v in E21_STAGED.items()}
    e14_save()
    print("E14 staging:", json.dumps(E14_RESULTS.get("staging"))[:1200], "| staged", E14_RESULTS["launch_paths"], flush=True)
os.environ.setdefault("MAX_JOBS", "4")              # bound FlashInfer JIT build parallelism (S1-60K)
CUDA_LINK = ensure_libcuda_linkable() if not E21_FAKE else {"probe": "PASS", "fake": True}
CONFIG["cuda_link"] = CUDA_LINK
print("CUDA/FlashInfer linker preflight:", json.dumps(CUDA_LINK), flush=True)
# every launch command this notebook can make (primary + fallback of each half, both attention backends), for the record
# and for the Kaggle CPU kit, which parses each one with the real vLLM 0.19 / 0.32 argument parser
E21_FLAGS = {f"{_H}_{_tag}_{_be}": {"vllm": _cfg["vllm"], "flags": (e21_flags_v019 if _cfg["vllm"] == "0.19.0" else e21_flags_v032)(
                 _cfg, E21_PORTS[_H], _be)}
             for _H, _sp in E21_HALF_SPECS[E21_VARIANT].items() for _tag, _cfg in e21_cfgs(_sp)
             for _be in ("FLASHINFER", "TRITON_ATTN")}
CONFIG["e21"]["flags"] = E21_FLAGS
E14_STATE.enter("LAUNCH", deadline=HARD_STOP)


In [ ]:
# ==================================================================== CELL 6/7 — E21: the Duck run and its report as functions
# Both are E15B R3K2's cells (build_e21.py: the changed lines are listed in the Experiment 21 README),
# wrapped so each half runs them once (every module-level name is declared global: same behaviour).
def e21_cell6():
    # E15B CELL 6 (R3K2) with the vLLM probes / watchdog (build_e21.py), run once per half
    global _E12_INDEX, _e12_order2_score, _e12_total_cells, E12_ORDER, E12_RECORDS, E12_RUN, _E12_LOCK, E12_OUT, _tempfile, _E12_WRITE_LOCK, _e12_write_json_atomic, e12_write_submission, _e12_branch_done_r6, _e12_g4b_window, _e12_shifted_end, _e21_progress, _e15b_gen_tokens_total, _e15b_all_server_logs, _e15b_server_log_summary, _e12_capacity_probe, _e12_cache_diag, _e12_no_spec, _e12_parity, _e12_r6_preflight, _e12_branch_done, _E12_STOP, _e12_heartbeat, _e, _end, _ra, _hid_end, _start, _per, _, _kv, _first, _workers, _shift, _e15b_solve_end, E15B_SOLVE_END_PC, _e15b_tok0, _WD_POLL, _WD_STALL, _WD_FAIL, _WD_MIN_LEFT, _e15b_watchdog, _att, _ev, _bud, _g, _run, _q, _S, _full, _left
    # ==================================================================== CELL 6 — E12: Duck REPL agent, every task, E12_BRANCHES branches
    # Every branch is admitted at once; Duck's priority gate keeps ARC3_MAX_ACTIVE_STREAMS conversations active (first
    # branches in ORDER2 order before any second branch). A branch ends when the host verifies a submitted program on every
    # train pair, at the deadline, or at the optional token cap. Full thinking: Duck's transcript.txt per branch.
    _E12_INDEX = {t: i for i, t in enumerate(TASK_IDS)}


    def _e12_order2_score(tid):   # = S49 ORDER2 ([>=3 train] + [1 test]) with the cells tie-break, sorted-id order last
        _t = CHALLENGES[tid]
        return int(len(_t["train"]) >= 3) + int(len(_t["test"]) == 1)


    def _e12_total_cells(tid):
        _t = CHALLENGES[tid]
        _c = lambda g: len(g) * len(g[0]) if g and g[0] else 0
        return sum(_c(p["input"]) + _c(p["output"]) for p in _t["train"]) + sum(_c(p["input"]) for p in _t["test"])


    E12_ORDER = sorted(TASK_IDS, key=lambda t: (-_e12_order2_score(t), _e12_total_cells(t), _E12_INDEX[t]))
    E12_RECORDS = {t: [] for t in TASK_IDS}
    E12_RUN = {"started_sec": None, "model_id": None, "streams": None, "deadline_in_sec": None, "error": None,
               "branches_done": 0, "branches_accepted": 0, "finished_sec": None}
    _E12_LOCK = threading.Lock()
    E12_OUT = OUT / "duck"
    E12_OUT.mkdir(parents=True, exist_ok=True)


    import tempfile as _tempfile
    _E12_WRITE_LOCK = threading.Lock()   # rev 2 (GPT audits, reproduced): branches finish concurrently


    def _e12_write_json_atomic(path, obj):
        """A unique temp file per write + os.replace (CELL 0's helper reuses one '<name>.tmp' for every writer)."""
        fd, tmp = _tempfile.mkstemp(prefix=Path(path).name + ".", suffix=".tmp", dir=str(Path(path).parent))
        try:
            with os.fdopen(fd, "w", encoding="utf-8") as f:
                json.dump(obj, f)
            os.replace(tmp, path)
        except BaseException:
            try:
                os.unlink(tmp)
            except OSError:
                pass
            raise


    def e12_write_submission():
        with _E12_WRITE_LOCK:                          # one writer at a time; the snapshot is taken inside, so it is fresh
            sub = identity_submission(test_challenges)
            if RERUN:
                with _E12_LOCK:
                    recs = {t: list(v) for t, v in E12_RECORDS.items() if v}
                for t, rows in E12A.build_submission({t: CHALLENGES[t] for t in recs}, recs).items():
                    if t in sub and len(sub[t]) == len(rows):
                        sub[t] = rows
            _e12_write_json_atomic(SUBMISSION_PATH, sub)   # evaluation: keeps the competition-format test fallback


    def _e12_branch_done_r6(rec, _recs=E12_RECORDS, _run=E12_RUN, _lock=_E12_LOCK, _half=E21_HALF):
        # rev 6: a branch reports every pause and its finish; keep the latest per branch.
        # E21 rev 2c (GPT audits 1-3, CRITICAL, reproduced on this source): the callback is bound to ITS half's records, run
        # state and lock (default arguments, evaluated when this half's CELL 6 defines it). A half-A worker that is still
        # alive (a straggler past join grace) can never write into half B's records, and never writes half B's submission.
        with _lock:
            rows = _recs.setdefault(rec["task_id"], [])
            # rev 6g H7-1 (GPT rev 6f audits): callbacks can finish out of order; never let an older record replace a newer one
            _old = next((r for r in rows if r.get("branch") == rec.get("branch")), None)
            if _old is not None and int(rec.get("publish_seq") or 0) <= int(_old.get("publish_seq") or 0):
                _run["stale_callbacks_dropped"] = _run.get("stale_callbacks_dropped", 0) + 1
                return
            new = rec.get("branch") not in {r.get("branch") for r in rows}
            rows[:] = [r for r in rows if r.get("branch") != rec.get("branch")] + [rec]
            _run["branches_started"] = _run.get("branches_started", 0) + int(new)
            _run["branches_done"] = sum(1 for v in _recs.values() for r in v if r.get("status") == "done")
            _run["branches_accepted"] = sum(1 for v in _recs.values() for r in v if r.get("accepted"))
        if _half != E21_HALF:
            _run["late_cross_half_callbacks"] = _run.get("late_cross_half_callbacks", 0) + 1
            return
        if time.perf_counter() >= globals().get("E15B_SOLVE_END_PC", float("inf")):
            _run["late_writes_skipped"] = _run.get("late_writes_skipped", 0) + 1     # E15B B1
            return
        try:
            e12_write_submission()
        except Exception as _e:
            print("E12 submission write failed:", repr(_e)[:200], flush=True)


    def _e12_g4b_window(hidden_solve_end_sec, solve_start_sec, n_tasks, hidden_tasks=240):
        # gate 4b (user 2026-10-21): this run gets the hidden run's compute per task. H6-16: a function so the real
        # values (42,720 / 1,463 / 12 -> 2,063 s) are unit-tested, not only under the harness's 300 s cap
        per = max(0.0, float(hidden_solve_end_sec) - float(solve_start_sec)) / float(hidden_tasks)
        return per, per * int(n_tasks)


    def _e12_shifted_end(end, setup_sec, hard_end):
        # SA-1 / H6-6: a commit window starts after the preflight (and nothing else runs before solving since H6-3); the end
        # moves by the setup time, never past the hard stop
        return min(float(hard_end), float(end) + max(0.0, float(setup_sec)))


    def _e21_progress():
        # E21 rev 2h (GPT v3 HIGH-1, VAL2_dryrun F1): (requests running or waiting, prompt + generated tokens) from /metrics; None when
        # /metrics fails or lacks a field (then the stall rule never acts)
        try:
            _r = e21_prom()
        except Exception:
            return None
        _run = [x.value for x in _r if x.name in ("vllm:num_requests_running", "vllm:num_requests_waiting")]
        _tok = [x.value for x in _r if x.name in ("vllm:prompt_tokens_total", "vllm:generation_tokens_total")]
        return (sum(_run), sum(_tok)) if (_run and _tok) else None      # running + waiting on the server


    def _e15b_gen_tokens_total():
        # E15B #32 on vLLM (E21): the server's own generated-token counter (vllm:generation_tokens_total, summed over rows)
        try:
            _v = [float(r.value) for r in e21_prom() if getattr(r, "name", "") == "vllm:generation_tokens_total"
                  and getattr(r, "value", None) is not None]
            return sum(_v) if _v else None
        except Exception:
            return None


    def _e15b_all_server_logs():
        # B6 (auditor 2 F6, auditor 3 L85 / L125): every server log of the solve (the crashed ones first), not only the last
        _t = []
        for _p in ((E12_RUN.get("watchdog") or {}).get("old_logs") or []):
            try:
                _t.append(Path(_p).read_text(encoding="utf-8", errors="replace"))
            except Exception:
                pass
        try:
            _t.append(E11_SERVER.log_text())
        except Exception:
            pass
        return "\n".join(_t)


    def _e15b_server_log_summary(text):
        # E15B #42 / #53 on vLLM (E21): vLLM's 10-s engine stats lines (rev 2: e21_stat_rows, every optional field incl.
        # Preemptions / Deferred) instead of SGLang decode lines: running-level histogram (lines with work), queue > 0 share,
        # generation throughput, prefix-cache hit rate (rolling), KV usage, MTP acceptance, preemptions, the startup KV line
        import re as _re
        rows = e21_stat_rows(text)
        work = [r for r in rows if r["running"] > 0 or r["waiting"] > 0]
        _hist, _hq, _tpl = {}, {}, {}
        for r in work:
            k = r["running"]
            _hist[k] = _hist.get(k, 0) + 1
            if r["waiting"] > 0:
                _hq[k] = _hq.get(k, 0) + 1
            _tpl.setdefault(k, []).append(r["gen_tps"])
        _tp = sorted(r["gen_tps"] for r in work)
        _acc = sorted(float(a) for a in _re.findall(r"Mean acceptance length: ([\d.]+)", text))
        _kv = _re.findall(r"GPU KV cache size: ([\d,]+) tokens", text)
        _pre = [r["prefix_pct"] for r in rows if r["prefix_pct"] is not None]
        _n = len(work)
        return {"decode_lines": _n, "running_hist": dict(sorted(_hist.items())),
                "running_hist_queue_gt0": dict(sorted(_hq.items())),
                "startup": {"kv_tokens_logged": int(_kv[0].replace(",", "")) if _kv else None},
                "accept_len_p50": _acc[len(_acc) // 2] if _acc else None,
                "gen_tok_s_p50_by_running": {k: sorted(v)[len(v) // 2] for k, v in sorted(_tpl.items())},
                "running_mean_lines": round(sum(k * v for k, v in _hist.items()) / _n, 2) if _n else None,
                "queue_gt0_line_share": round(sum(_hq.values()) / _n, 3) if _n else None,
                "gen_tok_s_p50": _tp[len(_tp) // 2] if _tp else None,
                "gen_tok_s_mean": round(sum(_tp) / len(_tp), 1) if _tp else None,
                "prefix_cache_hit_pct_last_rolling": _pre[-1] if _pre else None,
                "kv_usage_pct_max": max((r["kv_pct"] for r in rows), default=None),
                "preemptions_in_stats_lines": sum(r["preempt"] for r in rows),
                "preempt_lines": sum(1 for r in rows if r["preempt"]),
                "oom_lines": sum(1 for _l in text.splitlines() if "out of memory" in _l.lower()),
                "traceback_lines": sum(1 for _l in text.splitlines() if "Traceback" in _l)}


    def _e12_capacity_probe():   # E21: vLLM 0.19 / 0.32 gauges (the max over label rows); vLLM has no separate Mamba gauge
        try:
            rows = [r for r in e21_prom() if getattr(r, "value", None) is not None]
        except Exception:
            rows = None
        if not rows:
            # E21 rev 2c (GPT audit 1 HIGH-4): Duck's KV gate admits on a missing sample (fail-open). On vLLM a failed
            # /metrics read while requests run means "unknown", so report a full pool: admissions wait; with nothing
            # running Duck's idle override still admits (no deadlock); the watchdog handles a dead server.
            E21_STATE["probe_failures"] = E21_STATE.get("probe_failures", 0) + 1
            _t0 = E21_STATE.setdefault("probe_fail_since", time.monotonic())
            if time.monotonic() - _t0 > 120:     # rev 2d: a long outage -> Duck's own fail-open (counted), not 1 stream forever
                E21_STATE["probe_fail_open"] = E21_STATE.get("probe_fail_open", 0) + 1
                return {"kv_usage": None, "running": None, "queue": None, "mamba_usage": None, "probe_failed": True}
            return {"kv_usage": 1.0, "running": None, "queue": None, "mamba_usage": None, "probe_failed": True}
        E21_STATE.pop("probe_fail_since", None)
        pick = lambda name: [float(r.value) for r in rows if r.name == name]
        out = {}
        for key, name in (("kv_usage", "vllm:kv_cache_usage_perc"), ("running", "vllm:num_requests_running"),
                          ("queue", "vllm:num_requests_waiting")):
            v = pick(name)
            out[key] = max(v) if v else None
        out["mamba_usage"] = None
        return out


    def _e12_cache_diag(budget_sec=240.0):
        # rev 6e G4-8 (GPT G4B audits): a bounded exact-prompt replay for the prefix-cache misses (G4B: 20/180 requests with
        # 0 cached tokens while KV <= 0.44; suspects SGLang #22935 split 0-hit and #36935 mamba checkpoint eviction).
        # Prompt A (about 9k tokens) warm -> exact replay -> two divergent suffixes -> 12 distinct long prompts at once
        # (churn) -> replay and a divergent suffix again. max_tokens 1, thinking off. Commit mode only.
        import random as _rnd, concurrent.futures as _cf
        t_end = time.monotonic() + float(budget_sec)

        def _blob(seed, rows=150):
            r = _rnd.Random(seed)
            return "\n".join(" ".join(str(r.randrange(10)) for _ in range(30)) for _ in range(rows))

        def _ask(msgs):
            body = {"model": E12_RUN["model_id"], "max_tokens": 1, "temperature": 0.0, "messages": msgs,
                    "chat_template_kwargs": {"enable_thinking": False}}
            t0 = time.monotonic()
            req = _urlreq.Request(SGL_URL + "/v1/chat/completions", data=json.dumps(body).encode("utf-8"),
                                  headers={"Content-Type": "application/json"})
            r = json.loads(_urlreq.urlopen(req, timeout=max(5.0, t_end - time.monotonic())).read())
            u = r.get("usage") or {}
            det = u.get("prompt_tokens_details") if isinstance(u.get("prompt_tokens_details"), dict) else {}
            return {"prompt": u.get("prompt_tokens"), "cached": int(det.get("cached_tokens") or 0),
                    "cached_reported": "cached_tokens" in det, "sec": round(time.monotonic() - t0, 2)}

        A = {"role": "system", "content": "E12-CACHE-DIAG reference A\n" + _blob(1)}
        q = lambda text: {"role": "user", "content": text}
        steps = []

        def step(name, msgs):
            if time.monotonic() > t_end - 5:
                steps.append({"step": name, "skipped": "budget"})
                return
            try:
                res = _ask(msgs)
            except Exception as _e:
                res = {"error": repr(_e)[:200]}
            res["step"] = name
            res.update({"g_" + k: v for k, v in _e12_capacity_probe().items()})
            steps.append(res)
        step("warm", [A, q("Reply with one word.")])
        step("replay_exact", [A, q("Reply with one word.")])
        step("divergent_1", [A, q("Divergent continuation one. Reply with one word.")])
        step("divergent_2", [A, q("Divergent continuation two. Reply with one word.")])
        if time.monotonic() < t_end - 30:
            churn = [[{"role": "system", "content": f"E12-CACHE-DIAG churn {i}\n" + _blob(100 + i)}, q("Reply with one word.")]
                     for i in range(max(1, int(E12_RUN.get("streams") or 12)))]

            def _safe(m):
                try:
                    return _ask(m)
                except Exception as _e:
                    return {"error": repr(_e)[:120]}
            with _cf.ThreadPoolExecutor(len(churn)) as _ex:
                cr = list(_ex.map(_safe, churn))
            steps.append({"step": "churn", "n": len(cr), "errors": sum(1 for x in cr if "error" in x),
                          "prompt_sum": sum(int(x.get("prompt") or 0) for x in cr),
                          "sec_max": max((x.get("sec") or 0) for x in cr),
                          **{"g_" + k: v for k, v in _e12_capacity_probe().items()}})
        step("replay_after_churn", [A, q("Reply with one word.")])
        step("divergent_after_churn", [A, q("Divergent continuation three. Reply with one word.")])
        by = {x["step"]: x for x in steps if "prompt" in x}
        pa = (by.get("warm") or {}).get("prompt")
        out = {"budget_sec": budget_sec, "prompt_tokens_A": pa, "steps": steps,
               "summary": {k: (by[k]["cached"], by[k]["prompt"]) for k in ("warm", "replay_exact", "divergent_1", "divergent_2",
                                                                             "replay_after_churn", "divergent_after_churn")
                           if k in by}}
        print("E12 cache diag (cached, prompt):", json.dumps(out["summary"]), flush=True)
        return out


    def _e12_no_spec(cmd):   # the same server command without --speculative-* (each takes one value)
        out, skip = [], False
        for x in cmd:
            if skip:
                skip = False
                continue
            if str(x).startswith("--speculative-"):
                skip = True
                continue
            out.append(x)
        return out


    def _e12_parity():
        # R6-X1 target-only parity (user 2026-10-21): the rev 6 system prompt + first opener of each task, with tools, T 0,
        # one request at a time, max_tokens 2,048, thinking on and off. DFlash2 twice (control), then a relaunch without
        # speculative decoding. Judged on equality and on what Duck acts on (tool-call presence and arguments, finish).
        global E11_SERVER
        TA, _, _, wrs = E12A._import_duck()
        Agent = E12A.make_agent_class()
        probes = []
        for tid in TASK_IDS:
            ag = Agent(task=CHALLENGES[tid], task_id=tid, branch=0, view="facts", model=E12_RUN["model_id"],
                       base_url=SGL_URL + "/v1", provider="vllm")
            d = E12_OUT / f"parity_{tid}"
            d.mkdir(parents=True, exist_ok=True)
            sp = d / TA.RUNTIME_STATE_FILENAME
            wrs(sp, current_frame=None, history=[])
            for think in (True, False):
                probes.append((tid, think, {"model": E12_RUN["model_id"], "max_tokens": 2048, "temperature": 0.0, "top_p": 1.0,
                                            "messages": [{"role": "system", "content": ag._system_prompt},
                                                         {"role": "user", "content": E12A.R6_TEXT["first"] + E12A.R6_COMPLETE}],
                                            "tools": ag._tools(sp), "tool_choice": "auto",
                                            "chat_template_kwargs": {"preserve_thinking": True, "enable_thinking": think}}))

        def _ask(body):
            req = _urlreq.Request(SGL_URL + "/v1/chat/completions", data=json.dumps(body).encode("utf-8"),
                                  headers={"Content-Type": "application/json"})
            r = json.loads(_urlreq.urlopen(req, timeout=900).read())
            m = r["choices"][0]["message"]
            return {"reasoning": str(m.get("reasoning_content") or ""), "content": str(m.get("content") or ""),
                    "calls": [str((tc.get("function") or {}).get("arguments", "")) for tc in (m.get("tool_calls") or [])],
                    "finish": r["choices"][0].get("finish_reason"), "tokens": (r.get("usage") or {}).get("completion_tokens")}

        def _round(tag):
            rows = []
            for tid, think, body in probes:
                try:
                    rows.append(_ask(body))
                except Exception as _e:
                    rows.append({"error": repr(_e)[:300]})
            (E12_OUT / f"parity_{tag}.json").write_text(json.dumps(rows), encoding="utf-8")
            return rows
        out = {"probes": [{"task": t, "thinking": k} for t, k, _ in probes]}
        a, b = _round("dflash_a"), _round("dflash_b")
        try:
            cmd, env = list(E11_SERVER.cmd), E11_SERVER.env
            e11_kill_server(wait=True)
            srv = E11Server(_e12_no_spec(cmd), env, "target_only")
            srv.start()
            ok, why = srv.wait_ready(min(SGL_RELAUNCH_TIMEOUT_SEC, max(60, HARD_STOP - time.perf_counter() - 600)))
            E11_SERVER = srv
            out["target_only_launch"] = {"ok": ok, "why": why, "spec_flags_removed": [x for x in cmd if str(x).startswith("--speculative-")]}
            t = _round("target_only") if ok else [{"error": "target-only launch failed"} for _ in probes]
        except Exception as _e:
            out["target_only_launch"] = {"ok": False, "why": repr(_e)[:300]}
            t = [{"error": "target-only launch failed"} for _ in probes]

        def _pre(x, y):
            n = 0
            for c1, c2 in zip(x, y):
                if c1 != c2:
                    break
                n += 1
            return n
        cmp_ = []
        for (tid, think, _), ra, rb, rt in zip(probes, a, b, t):
            row = {"task": tid, "thinking": think}
            if "error" not in ra and "error" not in rb:
                row["control_identical"] = ra == rb
            if "error" not in ra and "error" not in rt:
                ta, tt = ra["reasoning"] + ra["content"], rt["reasoning"] + rt["content"]
                row.update(target_identical=ra == rt, common_prefix_chars=_pre(ta, tt), dflash_chars=len(ta), target_chars=len(tt),
                           same_tool_call_presence=bool(ra["calls"]) == bool(rt["calls"]), same_tool_args=ra["calls"] == rt["calls"],
                           same_finish=ra["finish"] == rt["finish"], dflash_tokens=ra["tokens"], target_tokens=rt["tokens"])
            cmp_.append(row)
        out["compare"] = cmp_
        ok_rows = [r for r in cmp_ if "target_identical" in r]
        out["summary"] = {"probes": len(cmp_), "control_identical": sum(1 for r in cmp_ if r.get("control_identical")),
                          "target_identical": sum(1 for r in ok_rows if r["target_identical"]),
                          "same_tool_call_presence": sum(1 for r in ok_rows if r["same_tool_call_presence"]),
                          "same_finish": sum(1 for r in ok_rows if r["same_finish"]), "compared": len(ok_rows)}
        print("E12 parity:", json.dumps(out["summary"]), flush=True)
        return out


    def _e12_r6_preflight(n):
        # R6-X1 (bounded, inside this session): the drafter's tap-layer config (SGLang #37475) and an isolation smoke for
        # #36548 (n concurrent requests with distinct markers; a marker in another request's reply = contamination).
        out = {}
        try:
            _cfg = json.loads((SGL_DRAFT_PATH / "config.json").read_text(encoding="utf-8"))
            _flat = dict(_cfg)
            for _v in list(_cfg.values()):
                if isinstance(_v, dict):
                    _flat.update(_v)
            out["drafter_target_layer_config"] = {k: v for k, v in _flat.items() if "layer" in k.lower()
                                                  and ("target" in k.lower() or "tap" in k.lower())}
        except Exception as _e:
            out["drafter_config_error"] = repr(_e)[:200]
        try:
            import concurrent.futures as _cf, random as _rnd
            marks = ["ZQ" + str(_rnd.randrange(10 ** 6, 10 ** 7)) + "K" + str(i) for i in range(n)]

            def _one(m):
                body = {"model": E12_RUN["model_id"], "max_tokens": 24, "temperature": 0.0,
                        "messages": [{"role": "user", "content": "Reply with exactly this code and nothing else: " + m}],
                        "chat_template_kwargs": {"enable_thinking": False}}
                req = _urlreq.Request(SGL_URL + "/v1/chat/completions", data=json.dumps(body).encode("utf-8"),
                                      headers={"Content-Type": "application/json"})
                r = json.loads(_urlreq.urlopen(req, timeout=60).read())     # E21 rev 2b (C1 M3): <= ~2 min per preflight
                return str(r["choices"][0]["message"].get("content") or "")
            with _cf.ThreadPoolExecutor(n) as _ex:
                replies = list(_ex.map(_one, marks))
            out.update(E12A.isolation_verdict(marks, replies))     # C6-2: PASS / FAIL / INCONCLUSIVE
        except Exception as _e:
            out.update(isolation_error=repr(_e)[:300], isolation_state="INCONCLUSIVE", isolation_ok=None)
        try:   # R6-X1 parity CONTROL: the same request twice at T 0, one at a time; equal replies = a deterministic baseline.
            # A target-only (no DFlash) comparison needs a separate server launch and is never run beside production.
            pc = []
            for _m in ("PARITY-A-" + str(n), "PARITY-B-" + str(n)):
                _a, _b = _one(_m), _one(_m)
                pc.append({"same": _a == _b, "own_marker": _m in _a})
            out["parity_control"] = {"pairs": len(pc), "identical": sum(x["same"] for x in pc),
                                     "own_marker": sum(x["own_marker"] for x in pc),
                                     "note": "same config, batch size 1, T 0; target-only parity = separate launch (not run)"}
        except Exception as _e:
            out["parity_control"] = {"error": repr(_e)[:300]}
        print("E12 rev 6 preflight:", json.dumps(out)[:800], flush=True)
        return out


    def _e12_branch_done(rec):
        with _E12_LOCK:
            E12_RECORDS.setdefault(rec["task_id"], []).append(rec)
            E12_RUN["branches_done"] += 1
            E12_RUN["branches_accepted"] += int(bool(rec.get("accepted")))
        try:
            e12_write_submission()
        except Exception as _e:
            print("E12 submission write failed:", repr(_e)[:200], flush=True)


    _E12_STOP = threading.Event()


    def _e12_heartbeat(every=300, _stop=_E12_STOP):     # E21 rev 2c: bound to this half's stop event (GPT audit 1)
        while not _stop.wait(every):
            snap = {r.get("name"): r.get("value") for r in metrics_snapshot() if isinstance(r, dict) and "name" in r}
            keep = {k: v for k, v in snap.items() if any(s in k for s in ("num_running_reqs", "num_queue_reqs", "gen_throughput",
                                                                            "token_usage", "cache_hit_rate"))}
            print(f"E12 [{(time.perf_counter() - NOTEBOOK_T0) / 60:.0f} min] branches done {E12_RUN['branches_done']}"
                  f"/{len(TASK_IDS) * E12_BRANCHES}, accepted {E12_RUN['branches_accepted']} | {json.dumps(keep)[:400]}", flush=True)


    if ENGINE_READY and E10_SANDBOX_SELFTEST.get("ok"):
        try:
            try:
                E12_RUN["model_id"] = json.loads(_urlreq.urlopen(SGL_URL + "/v1/models", timeout=30).read())["data"][0]["id"]
            except Exception as _e:
                E12_RUN["model_id"] = E21_SERVED[(ENGINE_INFO.get("cfg") or {}).get("weights", "fp8")]   # E21: the served name
                print("E12: /v1/models failed, using the model path:", repr(_e)[:200], flush=True)
            E12_RUN["streams"] = min(int(CONCURRENCY), E12_STREAMS or int(CONCURRENCY))
            os.environ["ARC3_MAX_ACTIVE_STREAMS"] = str(E12_RUN["streams"])
            _end = min(HARD_STOP - E12_STOP_BEFORE_HARD_SEC, E21_HALF_END_PC)     # E21: this half's latest end
            if E12_SOLVE_WINDOW_SEC:
                _end = min(_end, time.perf_counter() + E12_SOLVE_WINDOW_SEC)
            if P.get("hidden_equiv"):   # gate 4b (user 2026-10-21): this run gets the hidden run's compute per task
                _ra = PROFILE_TABLE["RERUN_ALL"]
                _hid_end = _ra["time_budget_sec"] - _ra["hard_margin_sec"] - E12_STOP_BEFORE_HARD_SEC
                _start = time.perf_counter() - NOTEBOOK_T0
                _per, _ = _e12_g4b_window(_hid_end, _start, len(TASK_IDS))          # hidden test = 240 tasks
                E12_RUN["g4b"] = {"hidden_solve_end_sec": _hid_end, "solve_start_sec": round(_start), "hidden_tasks": 240,
                                  "sec_per_task": round(_per, 1), "tasks": len(TASK_IDS), "window_sec": round(_per * len(TASK_IDS))}
                _end = min(_end, time.perf_counter() + _per * len(TASK_IDS))
                print("E12 gate 4b:", json.dumps(E12_RUN["g4b"]), flush=True)
            E12_RUN["deadline_in_sec"] = round(_end - time.perf_counter())
            E12_RUN["started_sec"] = round(time.perf_counter() - NOTEBOOK_T0)
            print("E12 solving:", len(E12_ORDER), "tasks x", E12_BRANCHES, "branches | streams", E12_RUN["streams"],
                  "| model", E12_RUN["model_id"], "| stops in", E12_RUN["deadline_in_sec"], "s | order head", E12_ORDER[:5], flush=True)
            threading.Thread(target=_e12_heartbeat, daemon=True, name="e12_heartbeat").start()
            # rev 6 R6-S5 (GPT plan review 2): per-stream allowance = 90 % of the KV pool / streams, so 10 resident histories
            # plus their reply reserve fit; the reserve itself follows each request's cap (R6-D1)
            _kv = int(ENGINE_INFO.get("kv_tokens") or 866744)
            E12_RUN["rev6_kv_tokens"] = _kv
            # E15B #46 (Exp 15 Notes/02): a FIXED window, not KV / streams (16 streams -> ~54k would trim earlier than rev 6g);
            # an attempt's history stays <= ~50k, the KV admission gate (0.95) covers memory
            E12_RUN["rev6_window"] = min(131072, int(E15B["E12_B_WINDOW"]))
            os.environ["E12_DUCK_CONTEXT_WINDOW"] = str(E12_RUN["rev6_window"])
            E12_RUN["rev6_preflight"] = _e12_r6_preflight(E12_RUN["streams"])
            if E12_RUN["rev6_preflight"].get("isolation_state") == "INCONCLUSIVE":
                # H6-17 (GPT rev 6e audit 3; user 2026-10-22): a missing marker without a leak is retried once before the
                # 6-stream fail-safe; a FAIL never is
                _first = E12_RUN["rev6_preflight"]
                E12_RUN["rev6_preflight"] = _e12_r6_preflight(E12_RUN["streams"])
                E12_RUN["rev6_preflight"]["retry_of"] = {k: _first.get(k) for k in ("isolation_state", "isolation_own_marker",
                                                                                      "isolation_leaks")}
            # C6-2 (rev 6c audits): full concurrency only on an isolation PASS; FAIL / INCONCLUSIVE / error -> 6 streams
            _workers = E12A.workers_for_isolation(E12_RUN["rev6_preflight"].get("isolation_ok"), E12_RUN["streams"])
            if E12_RUN["rev6_preflight"].get("isolation_state") != "PASS":
                print("E12 rev 6 WARNING: isolation preflight", E12_RUN["rev6_preflight"].get("isolation_state"),
                      "-> workers", _workers, flush=True)
            E12_RUN["rev6_workers"] = _workers
            print("E12 rev 6: window", E12_RUN["rev6_window"], "| workers", _workers, "| first exposure",
                  E12_R6["E12_R6_FIRST_EXPOSURE"], "| branch 1 view", E12_R6["E12_R6_B1_VIEW"], flush=True)
            if not RERUN and (E12_SOLVE_WINDOW_SEC or (P.get("hidden_equiv") and E12_RUN.get("g4b"))):
                # SA-1 + H6-6 (GPT rev 6e audit 2): every commit window (gate 4b and the plain solve window) starts after the
                # preflight, bounded by the hard stop; the per-task compute of gate 4b stays the hidden run's. The hidden
                # rerun keeps its absolute end (no window). H6-3: nothing else runs before solving
                _shift = (time.perf_counter() - NOTEBOOK_T0) - E12_RUN["started_sec"]     # started_sec: set when _end was fixed
                _end = _e12_shifted_end(_end, _shift, min(HARD_STOP - E12_STOP_BEFORE_HARD_SEC, E21_HALF_END_PC))
                E12_RUN["window_shift"] = {"shift_sec": round(_shift, 1),
                                           "window_start_sec": round(time.perf_counter() - NOTEBOOK_T0)}
                if E12_RUN.get("g4b"):
                    E12_RUN["g4b"].update(window_start_sec=E12_RUN["window_shift"]["window_start_sec"], shift_sec=round(_shift, 1))
                E12_RUN["deadline_in_sec"] = round(_end - time.perf_counter())
                print("E12: the solve window starts after the preflight (shift", round(_shift, 1), "s), stops in",
                      E12_RUN["deadline_in_sec"], "s", flush=True)
            # E15B #15 (code audit H1): a server watchdog for the solve. A dead SGLang process (or /health failing for
            # 180 s) is relaunched with the same command (at most 2 times, only with >= 20 min left); while it is down the
            # branches wait (arc2_duck.SERVER_DOWN) instead of burning their consecutive-failure budget.
            E12_RUN["watchdog"] = {"relaunches": [], "health_fail_sec_max": 0, "old_logs": [], "tok_acc": 0.0, "tok_last": None,
                                   "health_probes": 0}
            _e15b_solve_end = time.perf_counter() + max(0.0, _end - time.perf_counter())
            E15B_SOLVE_END_PC = _e15b_solve_end       # B1: per-branch submission writes stop here (the final write stays)
            _e15b_tok0 = _e15b_gen_tokens_total()
            E12_RUN["watchdog"]["tok_base"] = _e15b_tok0

            # A7 (Notes/07): the limits come from env only for the CPU harness's fault test; Kaggle uses the defaults
            _WD_POLL = float(os.environ.get("E15B_WD_POLL_SEC", "10"))
            _WD_STALL = float(os.environ.get("E15B_WD_STALL_SEC", "600"))     # E21 rev 2h: no-progress limit
            _WD_FAIL = float(os.environ.get("E15B_WD_FAIL_SEC", "180"))
            _WD_MIN_LEFT = float(os.environ.get("E15B_WD_MIN_LEFT_SEC", str(20 * 60)))
            E12_RUN["watchdog"]["limits"] = {"poll_sec": _WD_POLL, "fail_sec": _WD_FAIL, "min_left_sec": _WD_MIN_LEFT}

            def _e15b_watchdog(_wd_stop=_E12_STOP, _wd_run=E12_RUN, _wd_half=E21_HALF, _e15b_solve_end=_e15b_solve_end):
                # E21 rev 2f (VAL2_cells F1): this half's objects bound at def time (a module-level binding is global
                # after wrap() and half B would rebind it)
                # E21: the E15B watchdog on vLLM: a dead server (or /health failing for _WD_FAIL s while its log is silent)
                # is relaunched once with the same half's ready config (e21_relaunch, CELL 5); while it is down the
                # branches wait (arc2_duck.SERVER_DOWN) instead of burning their consecutive-failure budget
                _bad, _sz_last = None, {}
                _wd = _wd_run["watchdog"]
                _prog = {"v": None, "t": time.monotonic()}      # rev 2h: the last progress value and when it changed
                _wd.update(stall_sec=_WD_STALL, stalls=0, no_progress_sec_max=0)
                while not _wd_stop.wait(_WD_POLL):
                    if E21_HALF != _wd_half:          # rev 2c: a half-A watchdog never acts in half B
                        return
                    _srv = E11_SERVER
                    _dead = _srv is None or not _srv.alive()
                    _stall = False
                    if not _dead:
                        _tk = _e15b_gen_tokens_total()      # B6: the last counter value of THIS server (it resets on relaunch)
                        if _tk is not None:
                            _wd["tok_last"] = _tk
                        # rev 2h (GPT v3 HIGH-1): /health 200 + a silent log while running requests make no progress = hung
                        _pr = _e21_progress()
                        if _pr is not None:
                            if not _pr[0] or _pr[1] != _prog["v"]:
                                _prog.update(v=_pr[1], t=time.monotonic())
                            else:
                                _np = time.monotonic() - _prog["t"]
                                _wd["no_progress_sec_max"] = max(_wd["no_progress_sec_max"], round(_np))
                                _stall = _np >= _WD_STALL
                    if _stall:
                        _wd["stalls"] += 1
                        print("E15B watchdog: no progress for", round(time.monotonic() - _prog["t"]), "s with",
                              _pr[0], "requests running -> treated as dead", flush=True)
                    elif not _dead:
                        try:     # B2: a growing server log = alive; /health only when it is silent
                            _sz = os.path.getsize(_srv.log_path)
                        except Exception:
                            _sz = None
                        if _sz is not None and _sz != _sz_last.get(id(_srv)):
                            _sz_last[id(_srv)] = _sz
                            _bad = None
                            continue
                        try:
                            _wd["health_probes"] += 1
                            _urlreq.urlopen(SGL_URL + "/health", timeout=10).read()
                            _bad = None
                            continue
                        except Exception:
                            _bad = _bad or time.monotonic()
                            _wd["health_fail_sec_max"] = max(_wd["health_fail_sec_max"],
                                                                             round(time.monotonic() - _bad))
                            if time.monotonic() - _bad < _WD_FAIL:
                                continue
                    _left = _e15b_solve_end - time.perf_counter()
                    _rec = {"t": round(time.perf_counter() - NOTEBOOK_T0), "dead": _dead, "left_sec": round(_left),
                            "stall": _stall}
                    if len(_wd["relaunches"]) >= 2 or _left < _WD_MIN_LEFT:
                        _rec["action"] = "none (limit or < 20 min left)"
                        _wd["relaunches"].append(_rec)
                        print("E15B watchdog: server down, no relaunch:", json.dumps(_rec), flush=True)
                        return
                    if E21_HALF != _wd_half:          # rev 2f: re-check right before acting
                        return
                    E12A.SERVER_DOWN.set()
                    print("E15B watchdog: server down -> relaunch", json.dumps(_rec), flush=True)
                    try:
                        try:
                            _wd["old_logs"].append(str(_srv.log_path))
                        except Exception:
                            pass
                        if _wd.get("tok_last") is not None and _wd.get("tok_base") is not None:   # B6
                            _wd["tok_acc"] += max(0.0, _wd["tok_last"] - _wd["tok_base"])
                        _wd["tok_base"], _wd["tok_last"] = 0.0, None
                        _ok, _why = e21_relaunch()
                        _rec.update(ok=_ok, why=_why)
                    except Exception as _e:
                        _rec.update(ok=False, why=repr(_e)[:300])
                    finally:
                        E12A.SERVER_DOWN.clear()
                        _wd["relaunches"].append(_rec)
                        _bad = None
                        _prog.update(v=None, t=time.monotonic())
                    print("E15B watchdog:", json.dumps(_rec), flush=True)
                    if not _rec.get("ok") and len(_wd["relaunches"]) >= 2:    # B3: a failed relaunch retries once
                        return
            threading.Thread(target=_e15b_watchdog, daemon=True, name="e15b_watchdog").start()
            if P.get("parity"):          # R6-X1 diagnostic profile: no solving (the fallback submission stays)
                E12_RUN["parity"] = _e12_parity()
            else:
              E12A.solve_all_r6({t: CHALLENGES[t] for t in E12_ORDER}, E12_ORDER, out_dir=E12_OUT,
                              deadline=time.monotonic() + max(0.0, _end - time.perf_counter()), base_url=SGL_URL + "/v1",
                              model_id=E12_RUN["model_id"], branches=E12_BRANCHES, max_generated_tokens=E12_MAX_GEN_PER_BRANCH,
                              save_request_logs=E12_REQUEST_LOGS, on_branch_done=_e12_branch_done_r6, workers=_workers,
                              first_exposure_tokens=E12_R6["E12_R6_FIRST_EXPOSURE"] or None, b1_view=E12_R6["E12_R6_B1_VIEW"],
                              min_admit_sec=E12_R6["E12_R6_MIN_ADMIT_SEC"], capacity_probe=_e12_capacity_probe,
                              kv_admit_limit=E12_R6["E12_R6_KV_ADMIT"], kv_poll_sec=E12_R6["E12_R6_KV_POLL"],
                              min_chunk_tokens=E12_R6["E12_R6_MIN_CHUNK"],
                              fair_share=bool(E12_R6["E12_R6_FAIR_SHARE"]),
                              stage2_p90_cutoff=bool(E12_R6["E12_R6_P90_CUTOFF"]),
                              join_grace_sec=90.0,       # E15B #54: room for the final harvest / partial test (30 s grace)
                              max_generated_tokens_b1=E15B["E12_B_ATTEMPT_TOKENS_B1"],
                              skip_b1_if_accepted=bool(E15B["E12_B_SKIP_B1_IF_ACCEPTED"]),     # rev 3i: R
                              extra_attempts=bool(E15B["E12_B_EXTRA_ATTEMPTS"]),               # rev 3i: stage 3
                              max_attempts=int(E15B["E12_B_MAX_ATTEMPTS"]),
                              # rev 3j: 1 = after the attempt 3s (notebook 1), 2 = before the attempt 3s (notebook 2)
                              b1_accepted_mode={1: "after3", 2: "before3"}[int(E15B["E12_B_B1_ACC_MODE"])],
                              weak_commit=bool(E15B["E12_B_WEAK_COMMIT"]))    # rev 3f: attempt 2's session cap
            E12_RUN["rev6_scheduler"] = dict(E12A.SCHED_LAST)
            # E21 rev 2c (GPT audits 1-2): the deadline contract as evidence (planned solve end vs when the solver returned)
            E12_RUN["solve_overrun_sec"] = round(max(0.0, time.perf_counter() - _e15b_solve_end), 1)
            E12_RUN["planned_solve_end_sec"] = round(_e15b_solve_end - NOTEBOOK_T0, 1)
            E12_RUN["solve_end_epoch"] = time.time() + (_e15b_solve_end - time.perf_counter())   # rev 2f: request t / t0 are epoch
            E12_RUN["stragglers"] = (E12_RUN["rev6_scheduler"] or {}).get("stragglers")
            if E12_RUN["solve_overrun_sec"] > 120:
                E12_RUN["deadline_violation"] = True
                print("E21 WARNING: the solver returned", E12_RUN["solve_overrun_sec"], "s after its deadline", flush=True)
            try:      # E15B #29, #30, #39: per-attempt summary + real concurrency from the 30-s server gauges
                _att = [r for v in E12_RECORDS.values() for r in v]
                _ev = [r.get("b_events") or {} for r in _att]
                # rev 3f: each attempt's own budget (attempt 1 = E12_B_ATTEMPT_TOKENS, attempt 2 = E12_B_ATTEMPT_TOKENS_B1)
                _bud = lambda r: int(E15B["E12_B_ATTEMPT_TOKENS_B1"] if r.get("branch") >= 1 else E15B["E12_B_ATTEMPT_TOKENS"])
                _g = [x for x in (E12_RUN["rev6_scheduler"].get("gauges") or []) if isinstance(x, list) and len(x) >= 5
                      and isinstance(x[3], (int, float))]
                _run = [x[3] for x in _g]
                _q = [x[4] for x in _g if isinstance(x[4], (int, float))]
                # A1 (Notes/07): the 30-s samples in which every client stream was busy (workers_running >= streams)
                _S = E12_RUN.get("streams") or 0
                _full = [x for x in _g if len(x) >= 6 and isinstance(x[5], (int, float)) and _S and x[5] >= _S]
                E12_RUN["b_summary"] = {
                    "attempts": len(_att), "attempt2_started": sum(1 for r in _att if r.get("branch") == 1),
                    "accepted": sum(1 for r in _att if r.get("accepted")),
                    "at_budget": sum(1 for r in _att if int(r.get("generated_tokens") or 0) >= _bud(r) - 64),
                    # no host-accepted candidate at all (an accepted exact check counts too), but a transform was written
                    "at_budget_with_progress": sum(1 for r in _att if int(r.get("generated_tokens") or 0)
                                                   >= _bud(r) - 64
                                                   and not any(x.get("accepted") for x in (r.get("submissions") or [])
                                                               + (r.get("checked") or []))
                                                   and ((r.get("rev6_events") or {}).get("first_transform_tokens") is not None)),
                    "commit_turns": sum(1 for e in _ev if e.get("commit_at_tokens") is not None),
                    "commit_with_tool_call": sum(1 for e in _ev if e.get("commit_tool_call")),
                    "submit_tool_routed": sum(int(e.get("submit_tool_routed") or 0) for e in _ev),
                    "reasoning_harvest_ran": sum(1 for e in _ev if (e.get("reasoning_harvest") or {}).get("ran")),
                    "reasoning_harvest_new_accepted": sum(int((e.get("reasoning_harvest") or {}).get("new_accepted") or 0) for e in _ev),
                    "trimmed_requests": sum(1 for r in _att for q in (r.get("requests") or []) if q.get("trimmed")),
                    "worker_errors": E12_RUN["rev6_scheduler"].get("worker_errors"),
                    "setup_failed": E12_RUN["rev6_scheduler"].get("setup_failed"),     # rev 3h: Auditor 4 #4
                    "cut_code_saved": sum(int(e.get("cut_code_saved") or 0) for e in _ev),     # rev 3h
                    "cut_code_harvested": sum(1 for e in _ev if (e.get("reasoning_harvest") or {}).get("cut_code_blocks")),
                    # rev 3i: R + stage 3
                    "b1_skipped": E12_RUN["rev6_scheduler"].get("b1_skipped"),
                    "extra_started": E12_RUN["rev6_scheduler"].get("extra_started"),
                    "extra_by_attempt": E12_RUN["rev6_scheduler"].get("extra_by_attempt"),
                    "extra_accepted": sum(1 for r in _att if int(r.get("branch") or 0) >= 2 and r.get("accepted")),
                    "b1_deferred": E12_RUN["rev6_scheduler"].get("b1_deferred"),                 # rev 3j
                    "b1_deferred_started": E12_RUN["rev6_scheduler"].get("b1_deferred_started"),
                    "weak_accepts": E12_RUN["rev6_scheduler"].get("weak_accepts"),               # rev 3k W1
                    "weak_only_tasks_continued": E12_RUN["rev6_scheduler"].get("weak_only_tasks_continued"),
                    "stage2_errors": E12_RUN["rev6_scheduler"].get("stage2_errors"),
                    "server_running_mean": round(sum(_run) / len(_run), 2) if _run else None,
                    "server_queue_gt0_share": round(sum(1 for v in _q if v > 0) / len(_q), 3) if _q else None,
                    "gauge_samples": len(_g), "streams": E12_RUN.get("streams"),
                    "client_full_samples": len(_full),
                    "server_running_mean_client_full": round(sum(x[3] for x in _full) / len(_full), 2) if _full else None,
                    "server_at_streams_share_client_full": (round(sum(1 for x in _full if x[3] >= _S) / len(_full), 3)
                                                            if _full else None),
                    "kv_stale_checks": E12_RUN["rev6_scheduler"].get("kv_stale_checks"),      # A4
                    # #34: accepted submissions vs harvested exact checks; #31 roll-up; #35 the effort actually sent
                    "accepted_submit": sum(int((r.get("b_candidate") or {}).get("submitted_accepted") or 0) > 0 for r in _att),
                    "harvested_only": sum(1 for r in _att if int((r.get("b_candidate") or {}).get("harvested") or 0) > 0
                                          and not int((r.get("b_candidate") or {}).get("submitted_accepted") or 0)),
                    "never_wrote_candidate": sum(1 for r in _att if not (r.get("b_candidate") or {}).get("candidate_source_written")),
                    "wrote_never_checked": sum(1 for r in _att if (r.get("b_candidate") or {}).get("candidate_source_written")
                                               and not (r.get("b_candidate") or {}).get("check_executed")),
                    "partial_test_runs": sum(1 for e in _ev if (e.get("partial_test") or {}).get("name")),
                    "effective_effort": E12_REASONING_EFFORT or "xhigh (template default)",
                    # #32: server-side generated tokens in the solve vs the attempts' count
                    # B6: per server (the counter resets on a relaunch): the finished servers' deltas + the current one
                    "server_gen_tokens": (lambda a, b: round((E12_RUN["watchdog"].get("tok_acc") or 0.0) + a - b)
                                          if a is not None and b is not None else None)(
                        _e15b_gen_tokens_total(), E12_RUN["watchdog"].get("tok_base")),
                    "counted_gen_tokens": sum(int(r.get("generated_tokens") or 0) for r in _att),
                    "server_down_waits": dict(E12A.SERVER_WAITS), "watchdog": E12_RUN.get("watchdog")}
                try:
                    E12_RUN["b_summary"]["server_log"] = _e15b_server_log_summary(_e15b_all_server_logs())   # #42 / #53; B6
                except Exception as _e:
                    E12_RUN["b_summary"]["server_log"] = {"error": repr(_e)[:200]}
                print("E15B summary:", json.dumps(E12_RUN["b_summary"]), flush=True)
                if _run and E12_RUN["b_summary"]["server_running_mean"] < (E12_RUN.get("streams") or 0) - 0.5:
                    print("E15B WARNING: mean server running", E12_RUN["b_summary"]["server_running_mean"], "< streams - 0.5",
                          flush=True)
            except Exception as _e:
                E12_RUN["b_summary"] = {"error": repr(_e)[:300]}
            # H6-3 (GPT rev 6e audits): the cache diagnostic runs AFTER solving, so the solver starts from the same server
            # state as the hidden rerun (which never runs it). Its time comes from what is left before the output margin;
            # the solve deadline never moves for it. Skipped when under 90 s are left
            if not RERUN and not P.get("parity") and E12_R6.get("E12_R6_CACHE_DIAG"):
                _left = HARD_STOP - E12_STOP_BEFORE_HARD_SEC - time.perf_counter() - 30
                if (E12_RUN.get("rev6_scheduler") or {}).get("stragglers"):      # rev 6g H7-4
                    E12_RUN["cache_diag"] = {"skipped": "a solver straggler may still be sending requests"}
                elif _left >= 90:
                    try:
                        E12_RUN["cache_diag"] = _e12_cache_diag(budget_sec=min(240.0, _left))
                        E12_RUN["cache_diag"]["ran"] = "after solving"
                    except Exception as _e:
                        E12_RUN["cache_diag"] = {"error": repr(_e)[:300]}
                else:
                    E12_RUN["cache_diag"] = {"skipped": "less than 90 s before the output margin", "left_sec": round(_left)}
        except BaseException:
            E12_RUN["error"] = traceback.format_exc()[-3000:]
            print("E12 solver error (submission.json keeps what was written):", E12_RUN["error"], flush=True)
    else:
        E12_RUN["error"] = f"not run: ENGINE_READY={ENGINE_READY}, sandbox self-test={E10_SANDBOX_SELFTEST.get('ok')}"
        print("E12:", E12_RUN["error"], "-> CELL 0 fallback submission", flush=True)
    _E12_STOP.set()
    E12_RUN["finished_sec"] = round(time.perf_counter() - NOTEBOOK_T0)
    try:
        e12_write_submission()
    except Exception:
        print("E12 final submission write failed:", traceback.format_exc()[-1000:], flush=True)


def e21_cell7():
    # E15B CELL 7 (R3K2) with a per-half bundle name, run once per half
    global _recs, _gen, _usage_summary, branches, t, r, subs, _all_u, report, _r6_metrics, _r6_metrics_more, _r6e_metrics, _zero_classes, _r6e_concurrency, hits, pass1, pass2, total, task_norm, recs_eval, th, qi, att, gold, a, b, acc, harv, _e15b_has, _B_TOK, _pa, _rows, _rs, _golds, _f, a0, a1, n0, n1, hit, r0, bp, k, v, _x, _ok, _hi, _e, _e11e, _gh, _solved, _yt, _sc, _pack, z, fp
    # ==================================================================== CELL 7 — E12: report, score (evaluation) and bundle
    try:
        with _E12_LOCK:
            _recs = {t: list(v) for t, v in E12_RECORDS.items()}
        _gen = E12A.build_submission({t: CHALLENGES[t] for t in TASK_IDS}, _recs)
        write_json_atomic(OUT / "e12_generated_attempts.json", _gen)

        def _usage_summary(u):
            p = sum(int(x.get("prompt") or 0) for x in u); c = sum(int(x.get("cached") or 0) for x in u)
            return {"requests": len(u), "prompt_tokens": p, "cached_tokens": c, "cached_pct": round(100 * c / p, 1) if p else None,
                    "completion_tokens": sum(int(x.get("completion") or 0) for x in u),
                    "cached_pct_by_request": [round(100 * int(x.get("cached") or 0) / int(x["prompt"]), 1) if x.get("prompt") else None
                                              for x in u][:400]}
        branches = []
        for t in TASK_IDS:
            for r in _recs.get(t, []):
                subs = r.get("submissions") or []
                branches.append({"task": t, "branch": r.get("branch"), "accepted": bool(r.get("accepted")), "turns": r.get("turns"),
                                 "failures": r.get("failures"), "note": r.get("note"), "generated_tokens": r.get("generated_tokens"),
                                 "total_tokens": r.get("total_tokens"), "sec": round((r.get("t1") or 0) - (r.get("t0") or 0), 1),
                                 "submissions": len(subs), "train_exact_per_submission": [sum(map(bool, s.get("train_exact") or []))
                                                                                           for s in subs],
                                 "usage": _usage_summary(r.get("usage") or []),
                                 # rev 5 metrics (absent before rev 5): harvested check() candidates and event counters
                                 "checked_verified": len(r.get("checked") or []),
                                 "checked_kept": sum(1 for s in (r.get("checked") or []) if s.get("accepted")),
                                 "rev5_events": r.get("rev5_events"),
                                 # rev 6 (absent before rev 6)
                                 "view": r.get("view"), "status": r.get("status"), "runs": r.get("runs"),
                                 "progress_best": (r.get("progress") or {}).get("best"), "rev6_events": r.get("rev6_events"),
                                 "stage2_class": r.get("stage2_class"),
                                 "overfit_flags": sorted({f for v in (r.get("checked") or []) + (r.get("submissions") or [])
                                                          if v.get("accepted") for f in (v.get("overfit_flags") or [])})})
        _all_u = [x for t in TASK_IDS for r in _recs.get(t, []) for x in (r.get("usage") or [])]
        report = {"run_id": RUN_ID, "profile": PROFILE, "rerun": RERUN, "configuration": CONFIG, "engine": ENGINE_INFO,
                  "e12_run": E12_RUN, "order": E12_ORDER, "branches": branches, "usage_all": {k: v for k, v in _usage_summary(_all_u).items()
                                                                                            if k != "cached_pct_by_request"},
                  "tasks_planned": len(TASK_IDS), "queries_planned": sum(len(CHALLENGES[t]["test"]) for t in TASK_IDS)}
        def _r6_metrics(rows):   # R6-M1: one report contract (censored first-check = the branch total)
            import statistics as _st
            med = lambda v: _st.median(v) if v else None
            ev = [r.get("rev6_events") or {} for r in rows]
            tot = lambda k: sum(int(e.get(k) or 0) for e in ev)
            ft = [e.get("first_tool_tokens") for e in ev if e.get("first_tool_tokens") is not None]
            fc = [(e.get("first_check_tokens") if e.get("first_check_tokens") is not None else r.get("generated_tokens") or 0)
                  for e, r in zip(ev, rows)]
            hist = lambda k: {kk: sum(int((e.get(k) or {}).get(kk, 0)) for e in ev)
                              for kk in sorted({x for e in ev for x in (e.get(k) or {})})}
            vs = sorted(x for e in ev for x in (e.get("verify_sec") or []))
            b0 = [r for r in rows if r.get("branch") == 0]
            return {"branches": len(rows),
                    "tasks_exposed": len({r.get("task") for r in b0 if (r.get("generated_tokens") or 0) > 0}),
                    "tasks_meaningful": len({r.get("task") for r in rows
                                             if (r.get("rev6_events") or {}).get("first_tool_tokens") is not None}),
                    "first_tool_tokens_median": med(ft), "first_check_tokens_median_censored": med(fc),
                    # G4-13 (GPT G4B audit 1 M1-B): the observed median is over checked branches only (selection bias)
                    "first_check_tokens_median_observed": med([e["first_check_tokens"] for e in ev
                                                               if e.get("first_check_tokens") is not None]),
                    "first_check_tokens_median_observed_b0": med([e["first_check_tokens"] for e, r in zip(ev, rows)
                                                                  if e.get("first_check_tokens") is not None and r.get("branch") == 0]),
                    "checked_branches": sum(1 for e in ev if e.get("first_check_tokens") is not None),
                    "never_checked": sum(1 for e in ev if e.get("first_check_tokens") is None),
                    "autocheck_calls": tot("autocheck_calls"), "autocheck_functions": tot("autocheck_functions"),
                    "grid_rows_suppressed": tot("grid_rows_suppressed"), "length_cut_no_call": tot("length_cut_no_call"),
                    "nudges_replaced": tot("nudges_replaced"), "timeouts_clamped": tot("timeouts_clamped"),
                    "verify_skipped_deadline": tot("verify_skipped_deadline"),
                    "accepted_duplicate_outputs": tot("accepted_duplicate_outputs"),
                    "resumes": hist("resumes"), "caps": hist("caps"),
                    "verify_sec_p50": vs[len(vs) // 2] if vs else None, "verify_sec_p90": vs[int(len(vs) * 0.9)] if vs else None,
                    "status": {s: sum(1 for r in rows if r.get("status") == s) for s in ("paused", "done", "new")},
                    **_r6_metrics_more([dict(r, task=t) for t in TASK_IDS for r in _recs.get(t, [])])}

        def _r6_metrics_more(rows):   # rev 6b additions (fix-list R6-M1; raw branch records; label-free, no truth)
            ev = [r.get("rev6_events") or {} for r in rows]
            q = lambda v, f: sorted(v)[min(len(v) - 1, int(f * len(v)))] if v else None
            tot = lambda k: sum(int(e.get(k) or 0) for e in ev)
            rc = tot("reasoning_chars")
            ft = [e.get("facts_tokens") for e in ev if e.get("facts_tokens") is not None]
            acc = [(r, v) for r in rows for v in (r.get("checked") or []) + (r.get("submissions") or []) if v.get("accepted")]
            # G4-7 / G4-13 (GPT G4B audits): a check() and a submit() of one program are 2 events, 1 program
            uniq, preds, progs = {}, {}, {}
            for r, v in acc:
                uniq.setdefault((r.get("task"), r.get("branch"), v.get("name"), json.dumps(v.get("test_outputs"))), v)
                preds.setdefault(r.get("task"), set()).add(json.dumps(v.get("test_outputs")))
                # H6-9 (GPT rev 6e audits 2/3): a program = its source + retained dependencies (src_key), across branches
                progs.setdefault(r.get("task"), set()).add(v.get("src_key") or (r.get("branch"), v.get("name"),
                                                                                 json.dumps(v.get("test_outputs"))))
            flags = {}
            for v in uniq.values():
                for f in v.get("overfit_flags") or []:
                    flags[f] = flags.get(f, 0) + 1
            by_task = {}
            for r, v in acc:
                by_task.setdefault(r.get("task"), {}).setdefault(r.get("branch"), set()).add(json.dumps(v.get("test_outputs")))
            both = [t for t, d in by_task.items() if len(d) >= 2]
            agree = sum(1 for t in both if set.intersection(*by_task[t].values()))
            b1 = {}
            for r in rows:
                if r.get("branch") == 1 and r.get("stage2_class") is not None:
                    own = by_task.get(r.get("task"), {})
                    new = len(own.get(1, set()) - own.get(0, set()))
                    c = b1.setdefault(str(r["stage2_class"]), {"branches": 0, "new_distinct_verified": 0, "tokens": 0})
                    c["branches"] += 1
                    c["new_distinct_verified"] += new
                    c["tokens"] += int(r.get("generated_tokens") or 0)
            runs = [x for r in rows for x in (r.get("runs") or [])]
            osh = [x["overshoot"] for x in runs if "overshoot" in x]
            rp = [x["first_uncached"] for x in runs if x.get("stage") == 2 and "first_uncached" in x]
            pe = [a / b for r in rows for a, b in ((r.get("rev5_events") or {}).get("prompt_estimates") or []) if b]
            us = [u for r in rows for u in (r.get("usage") or [])]
            try:
                dec = e11_decode_summary() or {}
            except Exception:
                dec = {}
            rh = dec.get("running_hist") or {}
            return {"facts_tokens_p50": q(ft, .5), "facts_tokens_p90": q(ft, .9), "facts_tokens_max": max(ft) if ft else None,
                    "hand_reading_grid_row_share": round(tot("reasoning_grid_row_chars") / rc, 4) if rc else None,
                    "reasoning_digit_share": round(tot("reasoning_digit_chars") / rc, 4) if rc else None,
                    "reasoning_repeat_share": round(tot("reasoning_repeat_chars") / rc, 4) if rc else None,
                    "tool_calls": tot("tool_calls"), "redefinition_share": round(tot("redefinitions") / max(1, tot("tool_calls")), 4),
                    "infra_errors": tot("infra_errors"), "trim_reasoning_only_blocks": tot("trim_reasoning_only_blocks"),
                    "facts_over_target": sum(1 for e in ev if e.get("facts_over_target")),
                    "accepted_events": len(acc), "accepted_candidates": len(uniq), "unique_verified_programs": len(uniq),
                    # H7-5: the field above is keyed (task, branch, name, outputs), i.e. name / output candidates; the program
                    # count by source + dependencies is verified_programs_by_source
                    "unique_name_output_candidates": len(uniq),
                    "distinct_test_predictions": sum(len(v) for v in preds.values()),
                    # H6-9: the old field counted distinct predictions; both are reported now
                    "verified_programs_by_source": sum(len(v) for v in progs.values()),
                    "verified_programs_per_task": {t: len(v) for t, v in sorted(progs.items())},
                    "distinct_predictions_per_task": {t: len(v) for t, v in sorted(preds.items())},
                    "overfit_flags": flags, "overfit_flags_counted_on": "unique programs",
                    "tasks_with_two_branch_candidates": len(both), "tasks_where_branches_agree": agree,
                    "branch1_yield_by_stage2_class": b1,
                    "overshoot_p50": q(osh, .5), "overshoot_max": max(osh) if osh else None,
                    "stage2_resume_uncached_p50": q(rp, .5), "stage2_resume_uncached_p90": q(rp, .9),
                    "prompt_estimate_over_actual_p50": round(q(pe, .5), 3) if pe else None,
                    "uncached_prompt_tokens": sum(int(u.get("prompt") or 0) - int(u.get("cached") or 0) for u in us),
                    "generated_tokens": sum(int(r.get("generated_tokens") or 0) for r in rows),
                    "real_running_max": max((int(k) for k, v in rh.items() if v), default=None) if rh else None,
                    "accept_len_median": dec.get("accept_len_median"),
                    "fair_share_last": ((E12_RUN.get("rev6_scheduler") or {}).get("shares") or [None])[-1],
                    **_r6e_metrics(rows, ev, b1, us)}

        def _r6e_metrics(rows, ev, b1, us):   # rev 6e: G4-14 / G4-15 / G4-18 (label-free, no truth)
            import statistics as _st
            med = lambda v: round(_st.median(v), 1) if v else None
            n = len(rows)
            ftt = [e.get("first_transform_tokens") for e in ev]
            obs = [x for x in ftt if x is not None]
            cens = [x if x is not None else int(r.get("generated_tokens") or 0) for x, r in zip(ftt, rows)]
            best = {}
            for r in rows:
                b = ((r.get("progress") or {}).get("best") or {})
                k = f"{b.get('k')}/{b.get('n')}" if b else "none"
                best[k] = best.get(k, 0) + 1
            b1_tok = sum(c["tokens"] for c in b1.values())
            b1_new = sum(c["new_distinct_verified"] for c in b1.values())
            reqs = [q for r in rows for q in (r.get("requests") or [])]
            later = [q for q in reqs if q.get("i", 0) > 0 and q.get("prompt")]
            zero = [q for q in later if not q.get("cached")]
            runs2 = [x for r in rows for x in (r.get("runs") or []) if x.get("stage") == 2]

            def _yield(sel):   # H6-14 / H7-3: new verified predictions per 100k generated tokens (all / b0 / b1, by stage)
                rr = [(x, r) for r in rows for x in (r.get("runs") or []) if sel(x, r)]
                tok = sum(int(x.get("tokens") or 0) for x, _ in rr)
                nb = sum(int(x.get("new_in_branch") or 0) for x, _ in rr)
                nt = sum(int(x.get("new_for_task") or 0) for x, _ in rr)
                return {"runs": len(rr), "tokens": tok, "new_in_branch": nb, "new_for_task": nt,
                        "per_100k_for_task": round(1e5 * nt / tok, 3) if tok else None,
                        "per_100k_in_branch": round(1e5 * nb / tok, 3) if tok else None}
            nud = [(e, r) for e, r in zip(ev, rows) if e.get("provisional_nudge_tokens") is not None]
            gap = lambda k: [e[k] - e["provisional_nudge_tokens"] for e, _ in nud if e.get(k) is not None]
            return {
                # H6-14 (GPT rev 6e audits): the allocation question is decided on these
                "verified_yield": {"all": _yield(lambda x, r: True), "b0": _yield(lambda x, r: r.get("branch") == 0),
                                   "b1": _yield(lambda x, r: r.get("branch") == 1),
                                   "stage1": _yield(lambda x, r: x.get("stage") == 1),
                                   "stage2": _yield(lambda x, r: x.get("stage") == 2)},
                # H6-18: nudge efficacy (H6-1: sent in a tool result at 24k, else in an opener)
                "nudge": {"branches": len(nud), "via": {k: sum(1 for e, _ in nud if e.get("provisional_nudge_via") == k)
                                                         for k in ("tool", "opener")},
                          # H7-10: emitted (above) vs seen by the model in a later request
                          "seen": sum(1 for e, _ in nud if e.get("provisional_nudge_seen_request") is not None),
                          "emitted_not_seen": sum(1 for e, _ in nud if e.get("provisional_nudge_seen_request") is None),
                          "tokens_p50": med([e["provisional_nudge_tokens"] for e, _ in nud]),
                          "transform_after": len(gap("first_transform_after_nudge")),
                          "tokens_to_transform_p50": med(gap("first_transform_after_nudge")),
                          "check_after": len(gap("first_check_after_nudge")),
                          "tokens_to_check_p50": med(gap("first_check_after_nudge")),
                          "accepted": sum(1 for _, r in nud if r.get("accepted")
                                          or any(v.get("accepted") for v in (r.get("checked") or [])))},
                "progress_same_name_new_source": sum(int(e.get("progress_same_name_new_source") or 0) for e in ev),
                "indexed_view_over_target": sum(1 for e in ev if e.get("view_over_target")),             # H7-8
                "tasks_with_response": len({r.get("task") for r in rows if r.get("branch") == 0
                                            and (r.get("generated_tokens") or 0) > 0}),                  # H7-7
                # G4-14
                "first_transform_tokens_median_observed": med(obs), "first_transform_tokens_median_censored": med(cens),
                "branches_with_transform": len(obs), "branches_with_transform_within_20k": sum(1 for x in obs if x <= 20000),
                "share_with_transform_within_20k": round(sum(1 for x in obs if x <= 20000) / n, 3) if n else None,
                "best_single_function_pairs": dict(sorted(best.items())),
                "b1_new_distinct_verified_per_100k_tokens": round(1e5 * b1_new / b1_tok, 3) if b1_tok else None,
                "provisional_nudges": sum(1 for e in ev if e.get("provisional_nudge_tokens") is not None),
                # G4-18
                "progress_reports": sum(int(e.get("progress_reports") or 0) for e in ev),
                "progress_same_improved": sum(int(e.get("progress_same_improved") or 0) for e in ev),
                "progress_new_hypothesis": sum(int(e.get("progress_new_hypothesis") or 0) for e in ev),
                "stage2_runs": len(runs2), "stage2_runs_improved": sum(1 for x in runs2 if x.get("improved")),
                # G4-15 (null prompt_tokens_details = 0 cached: SGLang 0.5.21 omits it when nothing was cached)
                "requests_instrumented": len(reqs), "later_requests": len(later), "later_zero_hit": len(zero),
                "later_zero_hit_prefix_intact": sum(1 for q in zero if q.get("prefix_ok")),
                "later_zero_hit_trimmed": sum(1 for q in zero if q.get("trimmed")),
                "later_zero_hit_idle_sec_p50": med([q["idle_sec"] for q in zero if q.get("idle_sec") is not None]),
                "later_idle_sec_p50": med([q["idle_sec"] for q in later if q.get("idle_sec") is not None]),
                "later_zero_hit_mamba_p50": med([q["mamba"] for q in zero if q.get("mamba") is not None]),
                "later_prefix_intact_share": round(sum(1 for q in later if q.get("prefix_ok")) / len(later), 3) if later else None,
                # H6-10: character-prefix share and tools / template changes for the zero-hit later requests (still a proxy)
                "later_zero_hit_char_prefix_full": sum(1 for q in zero if q.get("serialized_char_prefix") is not None
                                                       and q.get("prev_chars") and q["serialized_char_prefix"] >= q["prev_chars"] - 2),
                # H7-13 (GPT rev 6f audit 3): unexplained = the serialized prompt kept its prefix, tools / template unchanged,
                # not trimmed, and still 0 cached; otherwise the prompt changed (reason). A proxy, never proof of eviction
                "later_zero_hit_class": _zero_classes(zero),
                "later_zero_hit_tools_or_template_changed": sum(1 for q in zero if q.get("tools_same") is False
                                                                or q.get("template_same") is False),
                "cached_not_reported_requests": sum(1 for q in reqs if q.get("cached_reported") is False),
                "cache_null_detail_requests": sum(1 for u in us if u.get("cached") is None),
                "cache_note": "aggregate over all requests; a missing prompt_tokens_details means 0 cached (vLLM 0.19 / 0.32, as SGLang 0.5.21)",
                "concurrency": _r6e_concurrency()}

        def _zero_classes(zero):   # H7-13
            out = {}
            for q in zero:
                full = (q.get("serialized_char_prefix") is not None and q.get("prev_chars")
                        and q["serialized_char_prefix"] >= q["prev_chars"] - 2)
                if q.get("trimmed"):
                    k = "prompt_changed:trimmed"
                elif q.get("tools_same") is False:
                    k = "prompt_changed:tools"
                elif q.get("template_same") is False:
                    k = "prompt_changed:template"
                elif not full:
                    k = "prompt_changed:prefix"
                else:
                    k = "unexplained_cache_miss"
                out[k] = out.get(k, 0) + 1
            return out

        def _r6e_concurrency():   # G4-15 / Q5: decode rate by running count, split by "a prefill batch ran just before"
            import re as _re, statistics as _st
            try:
                text = E11_SERVER.log_text() if E11_SERVER is not None else ""
            except Exception:
                return None
            g = lambda pat, ln: (lambda m: float(m.group(1)) if m else None)(_re.search(pat, ln))
            by, after = {}, False
            for ln in text.splitlines():
                if "TP0]" not in ln:
                    continue
                if "Prefill batch" in ln:
                    after = True
                    continue
                if "Decode batch" not in ln:
                    continue
                n = g(r"#running-req: (\d+)", ln)
                row = by.setdefault(str(int(n)) if n is not None else "?", {"lines": 0, "tok": [], "tok_after_prefill": [],
                                                                             "accept": [], "mamba": []})
                row["lines"] += 1
                t = g(r"gen throughput \(token/s\): ([\d.]+)", ln)
                if t is not None:
                    (row["tok_after_prefill"] if after else row["tok"]).append(t)
                for k, pat in (("accept", r"accept len: ([\d.]+)"), ("mamba", r"mamba usage: ([\d.]+)")):
                    v = g(pat, ln)
                    if v is not None:
                        row[k].append(v)
                after = False
            m = lambda v: round(_st.median(v), 2) if v else None
            return {k: {"lines": v["lines"], "tok_s_p50_clean": m(v["tok"]), "tok_s_p50_after_prefill": m(v["tok_after_prefill"]),
                        "share_after_prefill": round(len(v["tok_after_prefill"]) / max(1, len(v["tok"]) + len(v["tok_after_prefill"])), 3),
                        "accept_len_p50": m(v["accept"]), "mamba_usage_p50": m(v["mamba"])}
                    for k, v in sorted(by.items(), key=lambda kv: (len(kv[0]), kv[0]))}
        if _E12_REV6:
            report["rev6"] = {"metrics": _r6_metrics(branches), "scheduler": E12_RUN.get("rev6_scheduler"),
                              "window": E12_RUN.get("rev6_window"), "workers": E12_RUN.get("rev6_workers"),
                              "preflight": E12_RUN.get("rev6_preflight"), "g4b": E12_RUN.get("g4b"),
                              "parity": E12_RUN.get("parity"), "cache_diag": E12_RUN.get("cache_diag")}
        if SOLUTIONS is not None:
            hits = pass1 = pass2 = total = 0; task_norm = 0.0; recs_eval = []
            for t in TASK_IDS:
                th = 0
                for qi, att in enumerate(_gen[t]):
                    gold = SOLUTIONS[t][qi]; a = att["attempt_1"] == gold; b = att["attempt_2"] == gold
                    hits += a or b; pass1 += a; pass2 += b; total += 1; th += a or b
                    acc = [s for r in _recs.get(t, []) for s in (r.get("submissions") or []) if s.get("accepted")]
                    harv = [s for r in _recs.get(t, []) for s in (r.get("checked") or []) if s.get("accepted")]
                    recs_eval.append({"task": t, "query": qi, "hit": bool(a or b), "pass1": bool(a), "pass2": bool(b),
                                      "accepted_programs": len(acc),
                                      "accepted_programs_correct": sum(1 for s in acc if qi < len(s.get("test_outputs") or [])
                                                                       and s["test_outputs"][qi] == gold),
                                      "harvested_programs": len(harv),
                                      "harvested_programs_correct": sum(1 for s in harv if qi < len(s.get("test_outputs") or [])
                                                                        and s["test_outputs"][qi] == gold)})
                task_norm += th / max(1, len(_gen[t]))
            report.update(eval_query_hits=hits, eval_query_total=total, eval_pass1_hits=pass1, eval_pass2_hits=pass2,
                          eval_task_norm_pct=round(100 * task_norm / max(1, len(TASK_IDS)), 2), eval_unit_records=recs_eval)
            try:    # E15B A2 (Exp 15 Notes/07): what each attempt found. A task counts for an attempt when that attempt has a
                # right output for EVERY test input: "acc" = among its host-accepted candidates, "any" = also partial ones
                def _e15b_has(r, qi, gold, acc_only):
                    return any(qi < len(s.get("test_outputs") or []) and s["test_outputs"][qi] == gold
                               for s in (r.get("submissions") or []) + (r.get("checked") or []) if s.get("accepted") or not acc_only)
                _B_TOK = int(E15B["E12_B_ATTEMPT_TOKENS"]) if "E15B" in globals() else 35000
                _pa = dict.fromkeys(("tasks", "b1_started", "hit", "b0_acc", "b1_acc", "both_acc", "b0_only_acc", "b1_only_acc",
                                     "b0_any", "b1_any", "b1_only_any", "hit_without_b0_any", "b0_budget_partial",
                                     "b0_budget_partial_b1_acc"), 0)
                _rows = []
                for t in TASK_IDS:
                    _rs = {r.get("branch"): r for r in _recs.get(t, [])}
                    _golds = SOLUTIONS[t]
                    _f = lambda b, acc: b in _rs and all(_e15b_has(_rs[b], qi, g, acc) for qi, g in enumerate(_golds))
                    a0, a1, n0, n1 = _f(0, True), _f(1, True), _f(0, False), _f(1, False)
                    hit = all(att["attempt_1"] == g or att["attempt_2"] == g for att, g in zip(_gen[t], _golds))
                    r0 = _rs.get(0) or {}
                    bp = (int(r0.get("generated_tokens") or 0) >= _B_TOK - 64
                          and not any(x.get("accepted") for x in (r0.get("submissions") or []) + (r0.get("checked") or []))
                          and (r0.get("rev6_events") or {}).get("first_transform_tokens") is not None)
                    for k, v in (("tasks", True), ("b1_started", 1 in _rs), ("hit", hit), ("b0_acc", a0), ("b1_acc", a1),
                                 ("both_acc", a0 and a1), ("b0_only_acc", a0 and not a1), ("b1_only_acc", a1 and not a0),
                                 ("b0_any", n0), ("b1_any", n1), ("b1_only_any", n1 and not n0),
                                 ("hit_without_b0_any", hit and not n0), ("b0_budget_partial", bp),
                                 ("b0_budget_partial_b1_acc", bp and a1)):
                        _pa[k] += int(bool(v))
                    _rows.append({"task": t, "hit": hit, "b0_acc": a0, "b1_acc": a1, "b0_any": n0, "b1_any": n1,
                                  "b0_budget_partial": bp, "b0_tokens": r0.get("generated_tokens"),
                                  "b1_tokens": (_rs.get(1) or {}).get("generated_tokens")})
                report["eval_b_attempts"] = dict(_pa, rows=_rows)
                # rev 3i: what stage 3 added (a task hit whose only right accepted answer came from attempt 3+)
                _x = {"tasks_with_attempt3plus": 0, "attempt3plus_acc_tasks": 0, "hit_only_from_attempt3plus": 0}
                for t in TASK_IDS:
                    _rs = {r.get("branch"): r for r in _recs.get(t, [])}
                    _golds = SOLUTIONS[t]
                    _ok = lambda b: b in _rs and all(_e15b_has(_rs[b], qi, g, True) for qi, g in enumerate(_golds))
                    _hi = [b for b in _rs if (b or 0) >= 2]
                    if _hi:
                        _x["tasks_with_attempt3plus"] += 1
                        if any(_ok(b) for b in _hi):
                            _x["attempt3plus_acc_tasks"] += 1
                            if not _ok(0) and not _ok(1):
                                _x["hit_only_from_attempt3plus"] += 1
                report["eval_b_attempts"].update(_x)
                print("E15B stage 3 (eval):", json.dumps(_x), flush=True)
                print("E15B attempts (eval):", json.dumps(_pa), flush=True)
            except Exception as _e:
                report["eval_b_attempts"] = {"error": repr(_e)[:300]}
        try:
            report["e11"] = {"decode_summary": e11_decode_summary(),
                             "shim": engine.stats if isinstance(getattr(engine, "stats", None), dict) else None,
                             "server_log": str(E11_SERVER.log_path) if E11_SERVER is not None else None}
        except Exception as _e11e:
            report["e11"] = {"error": repr(_e11e)[:300]}
        if _E12_REV6 and isinstance(report.get("rev6"), dict):      # rev 6g H7-12 (GPT rev 6f audit 2): per GPU-hour
            try:
                _gh = 4 * max(1, (E12_RUN.get("finished_sec") or 0) - (E12_RUN.get("started_sec") or 0)) / 3600.0
                _solved = len({u["task"] for u in (report.get("eval_unit_records") or [])}
                              - {u["task"] for u in (report.get("eval_unit_records") or []) if not u.get("hit")})
                _yt = (((report["rev6"].get("metrics") or {}).get("verified_yield") or {}).get("all") or {}).get("new_for_task")
                report["rev6"]["per_gpu_hour"] = {
                    "gpu_hours": round(_gh, 3),
                    "correct_inputs": round(report["eval_query_hits"] / _gh, 3) if report.get("eval_query_hits") is not None else None,
                    "tasks_solved": round(_solved / _gh, 3) if report.get("eval_unit_records") else None,
                    "new_for_task_predictions": round(_yt / _gh, 3) if _yt is not None else None}
            except Exception as _e:
                report["rev6"]["per_gpu_hour"] = {"error": repr(_e)[:200]}
        write_json_atomic(OUT / "e12_run_report.json", report)
        if _E12_REV6:
            print("E12 rev 6 metrics:", json.dumps(report["rev6"]["metrics"])[:1500], flush=True)
            _sc = report["rev6"].get("scheduler") or {}
            # A3-F4 (rev 6c audits): full coverage is claimed only when stage1_left == 0
            print("E12 rev 6 coverage: tasks initialized", _sc.get("tasks_initialized"), "| request sent",
                  _sc.get("tasks_with_request_sent"), "| response", (report["rev6"].get("metrics") or {}).get("tasks_with_response"),
                  "/", len(TASK_IDS), "| stage-1 starts left",
                  _sc.get("stage1_left"), "| workers", report["rev6"].get("workers"), "(fixed)", "| KV holds", _sc.get("kv_holds"),
                  "for", _sc.get("kv_hold_sec"), "s | idle overrides", _sc.get("kv_idle_overrides"),
                  "| KV samples", _sc.get("kv_samples"), "missing", _sc.get("kv_missing"),
                  "| stragglers", _sc.get("stragglers"), flush=True)
        print("E12 RUN REPORT:", OUT / "e12_run_report.json", "| eval", report.get("eval_query_hits"), "/", report.get("eval_query_total"),
              "| task-norm", report.get("eval_task_norm_pct"), "| accepted branches", E12_RUN["branches_accepted"],
              "| cached", report["usage_all"].get("cached_pct"), "%", flush=True)
        e12_write_submission()
        _pack = WORKING / f"Q27X_E21_{E21_VARIANT}_HALF{E21_HALF}_DUCK_BUNDLE.zip"
        with zipfile.ZipFile(_pack, "w", zipfile.ZIP_DEFLATED) as z:
            for fp in (SUBMISSION_PATH, OUT / "e12_run_report.json", OUT / "e12_generated_attempts.json",
                       OUT / "sglang_decode_summary.json", OUT / "sglang_requests.jsonl"):
                if fp.exists():
                    z.write(fp, arcname=fp.name)
            for fp in sorted(E12_OUT.rglob("*")):     # transcripts (full thinking) + branch results
                if fp.is_file() and (fp.name in ("transcript.txt", "branch_result.json") or fp.name.endswith("requests.jsonl")):
                    z.write(fp, arcname="duck/" + fp.relative_to(E12_OUT).as_posix())
            if E11_SERVER is not None and E11_SERVER.log_path.exists():
                z.write(E11_SERVER.log_path, arcname=E11_SERVER.log_path.name)
        print("E12 bundle:", _pack, round(_pack.stat().st_size / 1e6, 1), "MB", flush=True)
    except BaseException:
        print("E12 reporting error (submission.json still exists):", traceback.format_exc()[-3000:], flush=True)
    try:
        e11_kill_server(wait=False)
    except Exception:
        pass


In [ ]:
# ==================================================================== CELL 8 — E21: run the halves
# Commit: half A (vLLM 0.19) then half B (vLLM 0.32) on the same 120 evaluation tasks, each with fresh Duck state and its
# own folder q27x/half<H>/; the hidden rerun: half A only. Each step is isolated: a failed half never stops the next.
def _e21_cell6_of(H):
    E21_STATE["halves"].setdefault(H, {})["cell6_ran"] = True     # CELL 6 re-creates E12_RECORDS / E12_RUN first
    e21_cell6()


for _H in E21_HALVES:
    if _H != E21_HALVES[0] and E21_STATE.get("abort_next_half"):        # rev 2h (GPT v3 audit 2): fail closed
        E21_STATE["halves"].setdefault(_H, {}).setdefault("errors", []).append(
            {"step": "handoff", "error": "PREVIOUS_WATCHDOG_STILL_ACTIVE: " + str(E21_STATE["abort_next_half"])})
        print(f"E21 half {_H} NOT started:", E21_STATE["abort_next_half"], flush=True)
        continue
    for _step, _fn in (("begin", lambda: e21_begin_half(_H)), ("cell6", lambda: _e21_cell6_of(_H)), ("cell7", e21_cell7),
                       ("end", lambda: e21_end_half(_H))):
        try:
            _fn()
        except BaseException:
            E21_STATE["halves"].setdefault(_H, {}).setdefault("errors", []).append({"step": _step,
                                                                                   "traceback": traceback.format_exc()[-3000:]})
            print(f"E21 half {_H} {_step} error:", traceback.format_exc()[-2000:], flush=True)
            if _step == "begin":
                ENGINE_READY = False


In [ ]:
# ==================================================================== CELL 9 — E21: the two-half report and bundle
# Per half: the full score (attempts 1 + 2, from its CELL 7 report), the score of attempt 1 alone (branch 0 only; the
# same for both halves whatever attempt 2 reached), engine config actually used, capacity, throughput, prefix cache,
# preemptions, timeline. Written to q27x/e21_report.json and printed as one table.
def _e21_score(recs):
    if SOLUTIONS is None:
        return None
    gen = E12A.build_submission({t: CHALLENGES[t] for t in TASK_IDS}, {t: list(recs.get(t, [])) for t in TASK_IDS})
    hits = total = 0; tn = 0.0
    for t in TASK_IDS:
        th = 0
        for qi, att in enumerate(gen[t]):
            g = SOLUTIONS[t][qi]; h = att["attempt_1"] == g or att["attempt_2"] == g
            hits += h; total += 1; th += h
        tn += th / max(1, len(gen[t]))
    return {"query_hits": hits, "queries": total, "task_norm_pct": round(100 * tn / max(1, len(TASK_IDS)), 2)}


def _e21_stage1(recs, run):
    """rev 2b (VAL_timing_C2 LOW-4): the attempt-1-only score compares halves only if stage 1 (attempt 1 of every task)
    finished in both: when the last attempt 1 ended, how many attempts 1 never started or were cut by the deadline."""
    b0 = [r for v in recs.values() for r in v if r.get("branch") == 0]
    ends = [max((q.get("t", 0) + (q.get("sec") or 0)) for q in (r.get("requests") or [])) for r in b0 if r.get("requests")]
    t0 = min((r.get("t0") or 0) for r in b0) if b0 else None
    last = max(ends) if ends else None
    dl = (run or {}).get("solve_end_epoch")          # rev 2f (VAL2_cells F2): the solve deadline, not the last end
    return {"attempt1_records": len(b0), "tasks": len(TASK_IDS),
            "attempt1_missing": len(TASK_IDS) - len({r.get("task_id") for r in b0}),
            "attempt1_cut_by_deadline": (sum(1 for x in ends if dl - x < 90) if (ends and dl) else None),
            "attempt1_all_done_h": round((last - t0) / 3600, 2) if (ends and t0) else None}


def _e21_pass1(recs):
    if SOLUTIONS is None:
        return None
    gen = E12A.build_submission({t: CHALLENGES[t] for t in TASK_IDS},
                                {t: [r for r in recs.get(t, []) if r.get("branch") == 0] for t in TASK_IDS})
    h = sum(int(a["attempt_1"] == SOLUTIONS[t][qi]) for t in TASK_IDS for qi, a in enumerate(gen[t]))
    return {"pass1_query_hits": h, "queries": sum(len(gen[t]) for t in TASK_IDS)}


E21_REPORT = {"run_id": RUN_ID, "profile": PROFILE, "rerun": RERUN, "variant": E21_VARIANT, "config": CONFIG.get("e21"),
              "halves": {}}
try:
    for _H in E21_HALVES:
        _st = E21_STATE["halves"].get(_H, {})
        _rec = _st.get("records") or {}
        _rep = {}
        try:
            _rep = json.loads((BASE_OUT / f"half{_H}" / "e12_run_report.json").read_text(encoding="utf-8"))
        except Exception as _e:
            _rep = {"error": repr(_e)[:200]}
        _eng = _st.get("engine") or {}
        _run = _st.get("e12_run") or {}
        _bs = _run.get("b_summary") or {}
        _sl = _bs.get("server_log") or {}
        E21_REPORT["halves"][_H] = {
            "engine_used": _eng.get("backend"), "config_used": _eng.get("used"), "spec": _eng.get("spec"),
            "attempts": [{k: a.get(k) for k in ("tag", "ready", "why", "ready_sec", "error")} for a in _eng.get("attempts", [])],
            "capacity": _eng.get("capacity"), "streams": _run.get("streams"), "workers": _run.get("rev6_workers"),
            "begin_sec": _eng.get("begin_sec"), "solve_start_sec": _run.get("started_sec"),
            "solve_finished_sec": _run.get("finished_sec"), "deadline_in_sec": _run.get("deadline_in_sec"),
            "end_sec": _st.get("end_sec"), "relaunches": _eng.get("relaunches"),
            "score_full": {k: _rep.get(k) for k in ("eval_query_hits", "eval_query_total", "eval_pass1_hits",
                                                    "eval_pass2_hits", "eval_task_norm_pct")},
            "score_attempt1_only": _e21_score({t: [r for r in v if r.get("branch") == 0] for t, v in _rec.items()}),
            "attempts_started": _bs.get("attempts"), "attempt2_started": _bs.get("attempt2_started"),
            "accepted": _bs.get("accepted"), "weak_accepts": _bs.get("weak_accepts"),
            "server_running_mean": _bs.get("server_running_mean"), "gen_tok_s_p50": _sl.get("gen_tok_s_p50"),
            "gen_tok_s_mean": _sl.get("gen_tok_s_mean"),
            # rev 2e (Kaggle CPU kit H3): the rev 2 log summary names it "..._rolling" (vLLM's last-1,000-request rate)
            "prefix_cache_hit_pct_last": _sl.get("prefix_cache_hit_pct_last_rolling"),
            "prefix_hit_pct_whole_run": (_eng.get("final_counters") or {}).get("prefix_hit_pct_whole_run"),
            "kv_usage_pct_max": _sl.get("kv_usage_pct_max"), "preempt_lines": _sl.get("preempt_lines"),
            "cached_pct_requests": (_rep.get("usage_all") or {}).get("cached_pct"),
            "server_gen_tokens": _bs.get("server_gen_tokens"), "counted_gen_tokens": _bs.get("counted_gen_tokens"),
            # rev 2: the solve window actually used, the prefill the server recomputed (vLLM align mode never reuses a
            # previous reply: VAL_api_D2 F1), Duck's gate, the server counters, the cut-call restorations, the probe
            "window_sec": _eng.get("window_sec"), "window_note": _eng.get("window_note"), "spec_note": _eng.get("spec_note"),
            "recomputed_prompt_tokens": sum(int(u.get("prompt") or 0) - int(u.get("cached") or 0)
                                            for v in _rec.values() for r in v for u in (r.get("usage") or [])),
            "prompt_tokens": sum(int(u.get("prompt") or 0) for v in _rec.values() for r in v for u in (r.get("usage") or [])),
            "gate": _st.get("gate"), "counters": _eng.get("final_counters"),
            "length_restored": _st.get("length_restored"),
            "stage1": _e21_stage1(_rec, _run),
            "score_attempt1_strict_pass1": _e21_pass1(_rec),
            "contract": next((a.get("contract") for a in reversed(_eng.get("attempts") or []) if a.get("contract")), None),
            "gpu_checks": _eng.get("gpu_checks"),
            "planned_end_sec": _eng.get("end_pc_in_sec"), "planned_solve_end_sec": _run.get("planned_solve_end_sec"),
            "solve_overrun_sec": _run.get("solve_overrun_sec"), "deadline_violation": _run.get("deadline_violation"),
            "stragglers": _run.get("stragglers"), "late_cross_half_callbacks": _run.get("late_cross_half_callbacks"),
            "lmo_in_log": next((a.get("lmo_in_log") for a in reversed(_eng.get("attempts") or []) if "lmo_in_log" in a), None),
            "errors": _st.get("errors"),
            # rev 2h (GPT v3 HIGH-2): solver error, complete scoring, the watchdog's stall evidence
            "solver_error": (_run.get("error") or None) and str(_run.get("error"))[-600:],
            "scoring_complete": (isinstance(_rep.get("eval_query_total"), int) and _rep["eval_query_total"]
                                 == sum(len((CHALLENGES.get(t) or {}).get("test") or []) for t in TASK_IDS) > 0),
            "watchdog_stalls": ((_run.get("watchdog") or {}).get("stalls")),
            "no_progress_sec_max": ((_run.get("watchdog") or {}).get("no_progress_sec_max"))}
    E21_REPORT["tool_calls_with_content"] = globals().get("E21_CALL_WITH_CONTENT")
    _ws = [h.get("deadline_in_sec") for h in E21_REPORT["halves"].values() if h.get("deadline_in_sec")]
    E21_REPORT["windows_sec"] = _ws
    # rev 2c (GPT audits 1/3), rev 2e (CPU kit H3: compare the configs that ran, not the tags; a relaunch that switched
    # config mid-half invalidates the comparison): valid only with equal windows (1 %), the same weights / seqs / SSM in
    # both halves (only the vLLM version differs) and no deadline violation
    _ran = {}
    for H in E21_REPORT["halves"]:
        _e = (E21_STATE["halves"].get(H, {}).get("engine") or {})
        _c = _e.get("cfg")
        _ran[H] = {"cfg": None if not _c else {k: _c.get(k) for k in ("weights", "seqs", "ssm")}, "tag": _e.get("used"),
                   "switched": bool(_e.get("relaunch_cfg_switched"))}
    _prim = {H: {k: v for k, v in E21_HALF_SPECS[E21_VARIANT][H].items() if k in ("weights", "seqs", "ssm")} for H in _ran}
    E21_REPORT["configs_ran"] = _ran
    if any(v["cfg"] is None for v in _ran.values()):
        E21_REPORT["config_status"] = "NOT_RUN"
    elif any(v["switched"] for v in _ran.values()):
        E21_REPORT["config_status"] = "SWITCHED_MIDHALF"
    elif len({json.dumps(v["cfg"], sort_keys=True) for v in _ran.values()}) > 1:
        E21_REPORT["config_status"] = "CONFOUNDED_FALLBACK"
    elif all(v["cfg"] == _prim[H] for H, v in _ran.items()):
        E21_REPORT["config_status"] = "PRIMARY_BOTH"
    else:
        E21_REPORT["config_status"] = "FALLBACK_BOTH"
    E21_REPORT["comparison_valid"] = (len(_ws) == 2 and max(_ws) - min(_ws) <= 0.01 * max(_ws)
                                      and E21_REPORT["config_status"] in ("PRIMARY_BOTH", "FALLBACK_BOTH")
                                      # rev 2e (CPU kit gateAB): the same client streams in both halves
                                      and len({h.get("streams") for h in E21_REPORT["halves"].values()}) == 1
                                      and not any(h.get("deadline_violation") for h in E21_REPORT["halves"].values())
                                      # rev 2h (GPT v3 HIGH-2): both halves ran cleanly and were fully scored
                                      and all(not h.get("errors") and not h.get("solver_error")
                                              and h.get("scoring_complete") is True and not h.get("watchdog_stalls")
                                              for h in E21_REPORT["halves"].values()))
    E21_REPORT["windows_equal"] = (max(_ws) - min(_ws) <= 300) if len(_ws) > 1 else None
    if E21_REPORT["windows_equal"] is False:
        print("E21 WARNING: the halves' solve windows differ by more than 5 min:", _ws, flush=True)
    write_json_atomic(BASE_OUT / "e21_report.json", E21_REPORT)
    print("E21 REPORT", E21_VARIANT, "| profile", PROFILE, "| rerun", RERUN, flush=True)
    for _H, _h in E21_REPORT["halves"].items():
        print(f"  half {_H}: {_h['engine_used']} ({_h['config_used']}) | streams {_h['streams']} | full",
              json.dumps(_h["score_full"]), "| attempt 1 only", json.dumps(_h["score_attempt1_only"]),
              "| attempt 2 started", _h["attempt2_started"], "| tok/s p50", _h["gen_tok_s_p50"],
              "| prefix hit", _h["prefix_cache_hit_pct_last"], "% | cached", _h["cached_pct_requests"],
              "% | preempt lines", _h["preempt_lines"], "| window", _h["window_sec"], "| recomputed",
              _h["recomputed_prompt_tokens"], "/", _h["prompt_tokens"], "| length restored", _h["length_restored"],
              "| errors", len(_h["errors"] or []), flush=True)
except BaseException:
    print("E21 report error:", traceback.format_exc()[-3000:], flush=True)
try:
    _pack = WORKING / f"Q27X_E21_{E21_VARIANT}_BUNDLE.zip"
    with zipfile.ZipFile(_pack, "w", zipfile.ZIP_DEFLATED) as z:
        for fp in [BASE_OUT / "e21_report.json", SUBMISSION_PATH] + sorted(BASE_OUT.glob("half*/e12_run_report.json")) \
                + sorted(BASE_OUT.glob("half*/vllm_server_*.log")) + sorted(BASE_OUT.glob("half*/sglang_decode_summary.json")) \
                + sorted((BASE_OUT / "e14").glob("*")) + [WORKING / "e14_state.json"]:
            if fp.exists() and fp.is_file():
                z.write(fp, arcname=fp.relative_to(WORKING).as_posix() if WORKING in fp.parents else fp.name)
    print("E21 bundle:", _pack, round(_pack.stat().st_size / 1e6, 1), "MB", flush=True)
except BaseException:
    print("E21 bundle error:", traceback.format_exc()[-2000:], flush=True)
